# Probe: Language Prior (one-stage E2E)

Standalone frozen-LLM EEG→text study for Flan-T5-large and BART-large. The notebook is self-contained: it does not import project-local modules.

1. Corpus UIDs are losslessly canonicalized to signed-int64 Python integers before mapping/split checks; null text is rejected before normalization. SemKey Q-Merger maps EEG `(B,1280,128)` to 96 tokens, then projects to the frozen LLM width.
2. EEG training and checkpoint selection both use `0.5 CLIP + 0.5 teacher-forced AR + 0.7 masked commitment`. Full-target greedy-rollout CE is display-only.
3. Canonical decoder histories are BART `decoder_start → BOS → optional instruction → lexical target → EOS` and T5 `decoder_start → optional instruction → lexical target → EOS`; native BART `gt_text` start-only generation retains HF forced-BOS compatibility.
4. `gt_text` runs deterministically once per LLM (`n=1`); EEG, `noise50` (`alpha=0.5`, not 50% SNR), and `noise100` run per configured seed.
5. Prefill settings come only from `CONFIG['prefill_ns']`. CSVs retain raw stripped decodes and continuations decoded directly from the generated token suffix; primary metrics use these token-boundary-exact continuations, with whole-output diagnostics shown separately.
6. Checkpoint loads compare complete model/tokenizer/effective-config/corpus metadata, including deterministic model config, tokenizer vocabulary/behavior/revision, corpus, and conditional local-weight fingerprints.
7. Before/after EEG t-SNE uses the same sampled rows and one shared fit, so panel coordinates are comparable.

Outputs remain under `outputs/probe/seed_{seed}/{llm_key}/{data_key}/`: prediction CSVs, EEG checkpoints/curves, and t-SNE figures. Metrics are printed/plotted only.


In [ ]:
# =============================================================
# Cell 1: Imports + HF mirror + CONFIG
# =============================================================
from __future__ import annotations

import os
import time
import gc
import hashlib
import json
import math
import pickle
import random
import shutil
import unicodedata
from pathlib import Path
from pickle import UnpicklingError
from typing import Any, Dict, Iterator, List, Literal, Optional, Sequence, Tuple, Union

if not os.environ.get('HF_ENDPOINT'):
    os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
print(f'HF_ENDPOINT: {os.environ.get("HF_ENDPOINT")}')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.data import DataLoader, Dataset, Sampler
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from torchmetrics.functional.text import bleu_score
from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    BartForConditionalGeneration,
    BartTokenizer,
    T5ForConditionalGeneration,
)
from transformers.modeling_outputs import BaseModelOutput


def _allow_torch_load_for_trusted_hf() -> None:
    """Disable the transformers torch<2.6 load gate for trusted Hub checkpoints.

    Some transformers versions refuse ``torch.load`` unless torch>=2.6. This
    notebook only loads public Flan-T5 / BART weights, so the check is skipped.
    """
    try:
        from transformers.utils import import_utils as _iu
        if hasattr(_iu, 'check_torch_load_is_safe'):
            _iu.check_torch_load_is_safe = lambda *a, **k: None  # type: ignore[assignment]
    except Exception:
        pass


_allow_torch_load_for_trusted_hf()


def _first_existing(*paths: str) -> str:
    """Return the first path that exists on disk, else the first candidate.

    Args:
        *paths: Candidate filesystem paths, tried in order.

    Returns:
        The first existing path, or ``paths[0]`` when none exist (so CONFIG still
        has a documented default even on a fresh machine).
    """
    for p in paths:
        if os.path.exists(p):
            return p
    return paths[0]


def _configure_hf_cache() -> Optional[str]:
    """Create and select a writable Hugging Face cache directory.

    Tries AutoDL tmp paths first, then ``./models``. Sets ``HF_HOME`` and the
    Hub cache env vars on the first directory that can be created.

    Returns:
        Absolute cache path, or None if every candidate failed.
    """
    candidates = [
        './autodl-tmp/models',
        '/root/autodl-tmp/models',
        './models',
    ]
    for c in candidates:
        try:
            resolved = Path(c).resolve()
            resolved.mkdir(parents=True, exist_ok=True)
            os.environ.setdefault('HF_HOME', str(resolved))
            os.environ.setdefault('HUGGINGFACE_HUB_CACHE', str(resolved))
            os.environ.setdefault('HF_HUB_CACHE', str(resolved))
            return str(resolved)
        except Exception:
            continue
    return None


HF_CACHE = _configure_hf_cache()

CONFIG: Dict[str, Any] = {
    'data_path': _first_existing(
        './autodl-tmp/preprocessed_data/zuco_merged_whiten_norm.df',
        '/root/autodl-tmp/preprocessed_data/zuco_merged_whiten_norm.df',
        './preprocessed_data/zuco_merged_whiten_norm.df',
    ),
    'output_dir': './outputs/probe',
    'model_cache_dir': HF_CACHE,
    'llm_load_max_retries': 5,
    'llm_load_retry_backoff_s': 30,
    'in_len': 1280,
    'in_dim': 128,
    'use_channel_weights': True,
    'out_len': 96,
    'hidden_dim': 128,
    'prompt_dim': 128,
    'n_in_blocks': 6,
    'n_out_blocks': 6,
    'num_heads': 8,
    'mlp_ratio': 4,
    'dropout': 0.1,
    'prompt_drop_probs': (0.0, 0.0, 0.0),
    'use_ei': True,
    'max_target_tokens': 64,
    'input_text_len': 96,
    'w_clip': 0.5,
    'commitment_weight': 0.7,
    'w_ar': 0.5,
    'epochs': 100,
    'lr': 1e-4,
    'lr_restart_epochs': 15,
    'min_lr': 1e-9,
    'weight_decay': 0.05,
    'grad_clip': 1.0,
    'patience': 5,
    'batch_size': 72,
    'num_workers': 0,
    'seed': 2026,
    'seeds': [2026, 42, 36],
    'tsne_max_batches': 16,
    'tsne_perplexity': 30,
    'tsne_max_points': 2000,
    'gen_max_batches': None,  # full test loader
    'run_smoke_only': False,
    'data_keys': ['gt_text', 'eeg', 'noise50', 'noise100'],
    'prefill_ns': [0, 1, 3, 5],
    'decoder_prompt': 'Based on the following signals, translate the sentence',
    'llm_keys_to_run': ['bart_large', 'flan_t5_large'],
}

LLM_CONFIGS: List[Dict[str, Any]] = [
    {
        'key': 'bart_large',
        'repo_id': 'facebook/bart-large',
        'modelscope_id': 'AI-ModelScope/bart-large',
        'family': 'encdec_bart',
        'source': 'modelscope',
        'batch_size': 72,
    },
    {
        'key': 'flan_t5_large',
        'repo_id': 'google/flan-t5-large',
        'family': 'encdec_t5',
        'source': 'hf_mirror',
        'batch_size': 72,
    },
]

DATA_KEY_ALPHA: Dict[str, Optional[float]] = {
    'gt_text': None,
    'eeg': 0.0,
    'noise50': 0.5,
    'noise100': 1.0,
}


def get_prefill_settings(config: Dict[str, Any]) -> List[Tuple[str, str, str, int]]:
    """Build prediction-column names from ``config['prefill_ns']``.

    Args:
        config: Run config; must contain a unique list of non-negative integers
            under ``prefill_ns``. ``0`` is labeled ``bos``.

    Returns:
        Tuples ``(pred_col, raw_pred_col, setting_label, n_words)`` in the
        configured order.
    """
    values = [int(n) for n in config.get('prefill_ns', [0])]
    assert values and all(n >= 0 for n in values), values
    assert len(values) == len(set(values)), f'duplicate prefill settings: {values}'
    settings: List[Tuple[str, str, str, int]] = []
    for n_words in values:
        label = 'bos' if n_words == 0 else f'prefill{n_words}'
        settings.append((f'pred_{label}', f'raw_pred_{label}', label, n_words))
    return settings


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)
assert abs(float(CONFIG['prompt_dim']) - float(CONFIG['in_dim'])) < 1e-8
assert int(CONFIG['input_text_len']) == int(CONFIG['out_len'])
print(f'Device: {DEVICE}')
print(
    f'seed={CONFIG["seed"]}; seeds={CONFIG["seeds"]}; out={CONFIG["output_dir"]}; model_cache={HF_CACHE}; '
    f'w_clip={CONFIG["w_clip"]} w_ar={CONFIG["w_ar"]} commit={CONFIG["commitment_weight"]} '
    f'epochs={CONFIG["epochs"]} patience={CONFIG["patience"]} lr={CONFIG["lr"]} '
    f'data_keys={CONFIG["data_keys"]} prefills={CONFIG["prefill_ns"]} '
    f'decoder_prompt={CONFIG["decoder_prompt"]!r}'
)


In [ ]:
# =============================================================
# ZuCo data pipeline: pickle load, split checks, GLIM sampler
# =============================================================

ALL_SUBJECTS = [
    'ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW',
    'ZMG', 'ZPH', 'YAC', 'YAG', 'YAK', 'YDG', 'YDR', 'YFR', 'YFS', 'YHS',
    'YIS', 'YLS', 'YMD', 'YMS', 'YRH', 'YRK', 'YRP', 'YSD', 'YSL', 'YTL',
]

PROMPT_KEYS: Dict[str, List[str]] = {
    'task': ['<UNK>'] + ['<NR>', '<TSR>'],
    'dataset': ['<UNK>'] + ['ZuCo1', 'ZuCo2'],
    'subject': ['<UNK>'] + ALL_SUBJECTS,
}


def task_to_prompt_token(task_key: str) -> str:
    """Map a ZuCo task id to the encoder prompt token.

    Args:
        task_key: Raw task string from the corpus (e.g. ``task1``, ``task3``).

    Returns:
        ``<TSR>`` for sentiment (task3), otherwise ``<NR>``.
    """
    return '<TSR>' if task_key == 'task3' else '<NR>'


def unpack_prompt_batch(
    prompt_batch: Union[List[Tuple[str, str, str]], Tuple[str, str, str], List, Tuple],
) -> Tuple[List[str], List[str], List[str]]:
    """Split a batch of ``(task, dataset, subject)`` rows into three lists.

    Args:
        prompt_batch: Either a list of 3-tuples (DataLoader collate) or a single
            3-tuple of strings.

    Returns:
        ``(tasks, datasets, subjects)`` with one string per batch row.
    """
    if isinstance(prompt_batch, list) and all(
        isinstance(row, tuple) and len(row) == 3 for row in prompt_batch
    ):
        rows = prompt_batch
        return (
            [str(row[0]) for row in rows],
            [str(row[1]) for row in rows],
            [str(row[2]) for row in rows],
        )
    if isinstance(prompt_batch, tuple) and len(prompt_batch) == 3 and all(
        isinstance(value, str) for value in prompt_batch
    ):
        first, second, third = prompt_batch
        return [str(first)], [str(second)], [str(third)]
    rows = list(prompt_batch)
    assert all(isinstance(row, (list, tuple)) and len(row) == 3 for row in rows)
    return (
        [str(row[0]) for row in rows],
        [str(row[1]) for row in rows],
        [str(row[2]) for row in rows],
    )


# --- pandas StringDtype pickle compat ---
import pandas._libs.arrays as _pd_libarrays
import pandas._libs.internals as _pd_libinternals
import pandas.core.indexes.base as _pd_indexes_base


class _CompatStringDtype:
    """Stand-in for pandas ``StringDtype`` when unpickling older ZuCo frames."""

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        """Record storage / NA metadata expected by the pickle payload.

        Args:
            *args: Optional ``(storage, na_value)`` positional fields.
            **kwargs: Named ``storage`` / ``na_value`` overrides.
        """
        self.storage = kwargs.get('storage', args[0] if args else 'python')
        self.na_value = kwargs.get('na_value', args[1] if len(args) > 1 else np.nan)
        self.kind = 'O'
        self.name = 'string'
        self.type = str


class _CompatStringArray(_pd_libarrays.NDArrayBacked):
    """Object-array stand-in for pandas ``StringArray`` during unpickle."""

    def __setstate__(self, state: Any) -> None:
        """Restore the backing ndarray from a pickle state dict or tuple.

        Args:
            state: Pickled StringArray payload (dict with ``_ndarray``/``data``,
                or a tuple whose second element is the array).
        """
        if isinstance(state, dict):
            data = np.asarray(state.get('_ndarray', state.get('data')), dtype=object)
        elif isinstance(state, tuple) and len(state) >= 2:
            data = np.asarray(state[1], dtype=object)
        else:
            raise TypeError(f'Unsupported StringArray state: {type(state)}')
        _pd_libarrays.NDArrayBacked.__init__(self, data, np.dtype('O'))

    def __array__(self, dtype: Any = None, copy: Any = None) -> np.ndarray:
        """Return the object ndarray, optionally recast.

        Args:
            dtype: Optional NumPy dtype for the returned array.
            copy: Unused; accepted for NumPy protocol compatibility.

        Returns:
            Object (or recast) ndarray of string values.
        """
        arr = np.asarray(self._ndarray, dtype=object)
        return np.asarray(arr, dtype=dtype) if dtype is not None else arr

    def __iter__(self) -> Iterator[Any]:
        """Iterate string values in the backing array."""
        return iter(self._ndarray)

    def __len__(self) -> int:
        """Number of stored string values."""
        return len(self._ndarray)


# Save the true pandas functions on their modules so rerunning this cell cannot
# capture a previous compatibility wrapper and recurse indefinitely.
if not hasattr(_pd_indexes_base, '_probe_orig_new_index'):
    current_new_index = _pd_indexes_base._new_Index
    previous_new_index = globals().get('_ORIG_NEW_INDEX', current_new_index)
    _pd_indexes_base._probe_orig_new_index = (
        previous_new_index if getattr(current_new_index, '__name__', '') == '_new_index_compat'
        else current_new_index
    )
if not hasattr(_pd_libinternals, '_probe_orig_unpickle_block'):
    current_unpickle_block = _pd_libinternals._unpickle_block
    previous_unpickle_block = globals().get('_ORIG_UNPICKLE_BLOCK', current_unpickle_block)
    _pd_libinternals._probe_orig_unpickle_block = (
        previous_unpickle_block
        if getattr(current_unpickle_block, '__name__', '') == '_unpickle_block_compat'
        else current_unpickle_block
    )
_ORIG_NEW_INDEX = _pd_indexes_base._probe_orig_new_index
_ORIG_UNPICKLE_BLOCK = _pd_libinternals._probe_orig_unpickle_block


def _new_index_compat(cls: Any, d: Dict[str, Any]) -> Any:
    """Rebuild a pandas Index, converting compat string arrays to object arrays.

    Args:
        cls: Index class being reconstructed.
        d: Pickle state dict; ``data`` may be a ``_CompatStringArray``.

    Returns:
        Index produced by the original pandas ``_new_Index``.
    """
    if isinstance(d, dict) and 'data' in d and isinstance(d['data'], _CompatStringArray):
        d = dict(d)
        d['data'] = np.asarray(d['data']._ndarray, dtype=object)
    return _ORIG_NEW_INDEX(cls, d)


def _unpickle_block_compat(values: Any, placement: Any, ndim: int) -> Any:
    """Unpickle an internals Block, flattening compat string arrays first.

    Args:
        values: Block values; may be a ``_CompatStringArray``.
        placement: Block column placement.
        ndim: Expected block ndim (1-d values are reshaped when ``ndim==2``).

    Returns:
        Block from the original pandas ``_unpickle_block``.
    """
    if isinstance(values, _CompatStringArray) or type(values).__name__ == '_CompatStringArray':
        values = np.asarray(values._ndarray, dtype=object)
    if isinstance(values, np.ndarray) and values.ndim == 1 and int(ndim) == 2:
        values = values.reshape(1, -1)
    return _ORIG_UNPICKLE_BLOCK(values, placement, ndim)


_pd_indexes_base._new_Index = _new_index_compat  # type: ignore[assignment]
_pd_libinternals._unpickle_block = _unpickle_block_compat  # type: ignore[assignment]


class _CompatUnpickler(pickle.Unpickler):
    """Unpickler that remaps pandas StringArray/StringDtype to compat classes."""

    def find_class(self, module: str, name: str) -> Any:
        """Resolve a pickle class, substituting string-dtype compat types.

        Args:
            module: Declared module of the pickled class.
            name: Declared class / function name.

        Returns:
            The local compat class or the standard pickle lookup result.
        """
        if name == 'StringArray':
            return _CompatStringArray
        if name == 'StringDtype':
            return _CompatStringDtype
        if name == '_new_Index':
            return _new_index_compat
        if name == '_unpickle_block':
            return _unpickle_block_compat
        return super().find_class(module, name)


def read_dataframe_pickle(path: Union[str, Path]) -> pd.DataFrame:
    """Load a DataFrame pickle through the string-dtype compatibility unpickler.

    Args:
        path: Path to a pandas DataFrame pickle.

    Returns:
        Loaded frame with string columns cast to object dtype.

    Raises:
        TypeError: If the pickle does not contain a DataFrame.
    """
    path = Path(path)
    with open(path, 'rb') as handle:
        obj = _CompatUnpickler(handle).load()
    if not isinstance(obj, pd.DataFrame):
        raise TypeError(f'Expected DataFrame, got {type(obj)}')
    for col in list(obj.columns):
        if pd.api.types.is_string_dtype(obj[col]) or str(obj[col].dtype).startswith('string'):
            obj[col] = obj[col].astype(object)
        else:
            try:
                if isinstance(obj[col].array, _CompatStringArray):
                    obj[col] = np.asarray(obj[col].array._ndarray, dtype=object)
            except Exception:
                pass
    return obj


_PICKLE_MAGIC = b'\x80'
_DATA_PATH_CANDIDATES: Tuple[str, ...] = (
    './autodl-tmp/preprocessed_data/zuco_merged_whiten_norm.df',
    '/root/autodl-tmp/preprocessed_data/zuco_merged_whiten_norm.df',
    './preprocessed_data/zuco_merged_whiten_norm.df',
    './autodl-tmp/zuco_merged_whiten_norm.df',
)


def _collect_data_candidates(requested: str) -> List[Path]:
    """Deduplicate the requested corpus path plus known fallback locations.

    Args:
        requested: Path from ``CONFIG['data_path']``.

    Returns:
        Unique candidate paths, requested first.
    """
    ordered: List[Path] = []
    seen: set[str] = set()
    for raw in (requested, *_DATA_PATH_CANDIDATES):
        p = Path(raw)
        key = str(p.resolve()) if p.exists() else str(p)
        if key in seen:
            continue
        seen.add(key)
        ordered.append(p)
    return ordered


def _stream_file_sha256(
    path: Union[str, Path],
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Compute SHA-256 in one fixed-size streaming pass.

    Args:
        path: File to hash.
        chunk_size: Read size in bytes (must be >= 1).

    Returns:
        Hex digest of the file contents.
    """
    assert int(chunk_size) >= 1
    digest = hashlib.sha256()
    with open(Path(path), 'rb') as handle:
        while True:
            chunk = handle.read(int(chunk_size))
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def _load_pickle_dataframe_with_fallback(requested: str) -> Tuple[pd.DataFrame, Path]:
    """Try the requested path and fallbacks until a pickle DataFrame loads.

    Args:
        requested: Preferred corpus path.

    Returns:
        ``(dataframe, resolved_path)`` for the first readable pickle.

    Raises:
        UnpicklingError: If no candidate exists or all fail to load.
    """
    candidates = _collect_data_candidates(requested)
    errors: List[str] = []
    for path in candidates:
        if not path.exists():
            continue
        size = path.stat().st_size
        with open(path, 'rb') as f:
            header = f.read(1)
        if header != _PICKLE_MAGIC:
            continue
        print(f'Loading {path} ({size:,} bytes) ...')
        try:
            return read_dataframe_pickle(path), path
        except Exception as exc:
            errors.append(f'{path}: {exc}')
            print(f'  [WARN] Failed: {exc}')
    raise UnpicklingError(
        'No loadable ZuCo corpus pickle found.\n' + '\n'.join(errors)
    )


_IEEE754_SAFE_INTEGER_MAX = (1 << 53) - 1
_INT64_MIN = -(1 << 63)
_INT64_MAX = (1 << 63) - 1


def _require_signed_int64_uid(value: int) -> int:
    """Return a Python int only when it fits signed int64 exactly.

    Args:
        value: Integer-like UID after earlier type conversion.

    Returns:
        Canonical signed-int64 Python ``int``.
    """
    canonical = int(value)
    assert _INT64_MIN <= canonical <= _INT64_MAX, (
        f'text uid outside signed int64 range: {canonical!r}'
    )
    return canonical


def canonicalize_text_uid(value: Any) -> int:
    """Losslessly canonicalize a supported UID to a signed-int64 Python ``int``.

    Args:
        value: int, finite integral float within the IEEE-754 exact range, or a
            digit string (optional ``+``/``-``). Nulls and bools are rejected.

    Returns:
        Canonical signed-int64 Python ``int``.
    """
    if value is None or isinstance(value, (bool, np.bool_)):
        raise AssertionError(f'invalid text uid: {value!r}')
    is_null = pd.isna(value)
    if isinstance(is_null, (bool, np.bool_)) and bool(is_null):
        raise AssertionError(f'null text uid: {value!r}')
    if isinstance(value, (int, np.integer)):
        return _require_signed_int64_uid(int(value))
    if isinstance(value, (float, np.floating)):
        numeric = float(value)
        assert math.isfinite(numeric) and numeric.is_integer(), (
            f'nonintegral text uid: {value!r}'
        )
        assert abs(numeric) <= _IEEE754_SAFE_INTEGER_MAX, (
            f'float text uid exceeds exact IEEE-754 integer range: {value!r}'
        )
        return _require_signed_int64_uid(int(numeric))
    if isinstance(value, str):
        text = unicodedata.normalize('NFKC', value).strip()
        signless = text[1:] if text[:1] in {'+', '-'} else text
        assert signless and all('0' <= char <= '9' for char in signless), (
            f'malformed text uid: {value!r}'
        )
        return _require_signed_int64_uid(int(text))
    raise AssertionError(f'unsupported text uid type: {type(value).__name__}')


def _normalize_corpus_text(value: Any) -> str:
    """Reject nulls, then NFKC/case-fold and collapse whitespace.

    Args:
        value: Raw ``input text`` cell.

    Returns:
        Normalized comparison string used for split-leakage checks.
    """
    if value is None:
        raise AssertionError('null input text')
    is_null = pd.isna(value)
    if isinstance(is_null, (bool, np.bool_)) and bool(is_null):
        raise AssertionError('null input text')
    normalized = unicodedata.normalize('NFKC', str(value)).casefold()
    return ' '.join(normalized.split())


def canonicalize_corpus_identity_columns(merged: pd.DataFrame) -> None:
    """Canonicalize UIDs in-place and reject any null input texts.

    Args:
        merged: Corpus frame; must include ``text uid`` and ``input text``.
    """
    assert 'text uid' in merged.columns and 'input text' in merged.columns
    canonical_uids = [canonicalize_text_uid(value) for value in merged['text uid']]
    merged['text uid'] = pd.Series(canonical_uids, index=merged.index, dtype=object)
    assert all(type(value) is int for value in merged['text uid'])
    for value in merged['input text']:
        _ = _normalize_corpus_text(value)


def validate_split_integrity(merged: pd.DataFrame) -> None:
    """Reject UID/text conflicts and cross-phase leakage.

    Args:
        merged: Corpus with ``phase``, ``text uid``, and ``input text``.
    """
    canonicalize_corpus_identity_columns(merged)
    phases = merged['phase'].astype(str)
    phase_names = set(phases.unique().tolist())
    expected = {'train', 'val', 'test'}
    assert phase_names == expected, f'phases must be exactly {sorted(expected)}, got {sorted(phase_names)}'

    normalized = merged['input text'].map(_normalize_corpus_text)
    assert bool(normalized.str.len().gt(0).all()), 'empty normalized input text'
    uid_text_counts = pd.DataFrame({
        'uid': merged['text uid'].tolist(), 'normalized_text': normalized.tolist(),
    }).groupby('uid', dropna=False)['normalized_text'].nunique(dropna=False)
    conflicts = uid_text_counts[uid_text_counts > 1]
    assert conflicts.empty, f'text uid maps to multiple normalized texts: {conflicts.index[:5].tolist()}'

    uid_sets: Dict[str, set[int]] = {}
    text_sets: Dict[str, set[str]] = {}
    for phase in ('train', 'val', 'test'):
        phase_mask = phases.eq(phase)
        assert bool(phase_mask.any()), f'empty phase={phase}'
        uid_sets[phase] = set(merged.loc[phase_mask, 'text uid'].tolist())
        text_sets[phase] = set(normalized.loc[phase_mask].tolist())
    for i, left in enumerate(('train', 'val', 'test')):
        for right in ('train', 'val', 'test')[i + 1:]:
            uid_overlap = uid_sets[left] & uid_sets[right]
            text_overlap = text_sets[left] & text_sets[right]
            assert not uid_overlap, f'UID leakage {left}/{right}: {list(uid_overlap)[:5]}'
            assert not text_overlap, f'normalized-text leakage {left}/{right}: {list(text_overlap)[:3]}'


def validate_corpus_samples(merged: pd.DataFrame) -> None:
    """Validate every raw EEG/mask pair without stacking the whole corpus.

    Args:
        merged: Corpus with object columns ``eeg`` ``(1280, 128)`` and
            ``mask`` ``(1280,)`` of {0,1} with at least one valid sample.
    """
    for row_pos, (eeg_value, mask_value) in enumerate(zip(merged['eeg'], merged['mask'])):
        eeg = np.asarray(eeg_value)
        mask = np.asarray(mask_value)
        assert eeg.shape == (1280, 128), f'row {row_pos}: EEG shape {eeg.shape}'
        assert mask.shape == (1280,), f'row {row_pos}: mask shape {mask.shape}'
        assert bool(np.isfinite(eeg).all()), f'row {row_pos}: non-finite EEG'
        assert bool(np.isin(mask, (0, 1)).all()), f'row {row_pos}: mask outside {{0,1}}'
        assert bool(mask.astype(bool).any()), f'row {row_pos}: no valid EEG point'


def corpus_provenance(merged: pd.DataFrame) -> Dict[str, Any]:
    """Copy the serializable provenance record attached at load time.

    Args:
        merged: Frame previously returned by ``load_merged_corpus``.

    Returns:
        Provenance dict, or empty if attrs were not set.
    """
    stored = merged.attrs.get('data_provenance', {})
    return dict(stored) if isinstance(stored, dict) else {}


def load_merged_corpus(data_path: str) -> pd.DataFrame:
    """Load and validate the standalone ZuCo whitened corpus.

    Args:
        data_path: Preferred pickle path; fallbacks are tried on failure.

    Returns:
        Reset-index frame with provenance in ``attrs['data_provenance']``.
    """
    df, df_path = _load_pickle_dataframe_with_fallback(data_path)
    if Path(data_path) != df_path:
        print(f'  Resolved data_path: {data_path} -> {df_path}')
    merged = df.reset_index(drop=True)
    if 'row_index' not in merged.columns:
        merged['row_index'] = np.arange(len(merged), dtype=np.int64)
    assert 'phase' in merged.columns
    required = {
        'eeg', 'mask', 'input text', 'text uid', 'phase', 'dataset', 'task', 'subject',
    }
    missing = required - set(merged.columns)
    assert not missing, f'Missing columns: {missing}'
    validate_split_integrity(merged)
    validate_corpus_samples(merged)
    counts = merged['phase'].astype(str).value_counts()
    print('  Computing streaming corpus SHA-256 ...')
    corpus_sha256 = _stream_file_sha256(df_path)
    merged.attrs['data_provenance'] = {
        'requested_path': str(data_path),
        'resolved_path': str(df_path.resolve()),
        'file_size_bytes': int(df_path.stat().st_size),
        'sha256': corpus_sha256,
        'rows': int(len(merged)),
        'phase_counts': {phase: int(counts.get(phase, 0)) for phase in ('train', 'val', 'test')},
        'integrity_checks': 'canonical-int-uids+non-null-text+exact-phases+uid-text-unique+split-disjoint+sample-shape-finite-mask',
    }
    print(
        f'  Corpus phases: train={int(counts.get("train", 0))}, '
        f'val={int(counts.get("val", 0))}, test={int(counts.get("test", 0))}; '
        f'validated {len(merged):,} EEG/mask samples'
    )
    return merged


def get_phase_df(merged: pd.DataFrame, phase: str) -> pd.DataFrame:
    """Return the rows of one split, reset-indexed.

    Args:
        merged: Full corpus.
        phase: ``train``, ``val``, or ``test``.

    Returns:
        Non-empty phase frame.
    """
    assert phase in ('train', 'val', 'test')
    frame = merged.loc[merged['phase'].astype(str) == phase].reset_index(drop=True)
    assert len(frame) > 0, f'Empty phase={phase}'
    return frame


class ProbeDataset(Dataset):
    """Per-row EEG/text dataset for CLIP + generation (one row per corpus sample)."""

    def __init__(self, df: pd.DataFrame) -> None:
        """Cache columns needed by the collate / encoder path.

        Args:
            df: Phase frame from ``get_phase_df``.
        """
        self.eeg = df['eeg'].tolist()
        self.mask = df['mask'].tolist()
        self.input_text = df['input text'].tolist()
        for value in self.input_text:
            _ = _normalize_corpus_text(value)
        self.text_uid = [canonicalize_text_uid(value) for value in df['text uid']]
        self.dataset = df['dataset'].tolist()
        self.task = df['task'].tolist()
        self.subject = df['subject'].tolist()
        self.row_index = (
            [int(x) for x in df['row_index'].tolist()]
            if 'row_index' in df.columns else list(range(len(df)))
        )

    def __len__(self) -> int:
        """Number of EEG/text rows."""
        return len(self.eeg)

    def __getitem__(self, idx: int) -> dict:
        """Return one sample dict (EEG tensor, mask, texts, prompt tokens).

        Args:
            idx: Dataset index.

        Returns:
            Collate-ready sample with ``prompt`` as ``(task, dataset, subject)``.
        """
        ds_tok = self.dataset[idx] if self.dataset[idx] in PROMPT_KEYS['dataset'] else '<UNK>'
        sub_tok = self.subject[idx] if self.subject[idx] in PROMPT_KEYS['subject'] else '<UNK>'
        text = str(self.input_text[idx])
        return {
            'eeg': torch.from_numpy(np.array(self.eeg[idx], dtype=np.float32)),
            'mask': torch.from_numpy(np.array(self.mask[idx], dtype=np.int32)),
            'input text': text,
            'target text': text,
            'text uid': canonicalize_text_uid(self.text_uid[idx]),
            'prompt': (task_to_prompt_token(str(self.task[idx])), ds_tok, sub_tok),
            'row_index': int(self.row_index[idx]),
            'dataset_raw': str(self.dataset[idx]),
            'task_raw': str(self.task[idx]),
            'subject_raw': str(self.subject[idx]),
        }


def probe_collate_fn(batch: List[dict]) -> dict:
    """Stack EEG/mask/UID tensors; leave text and prompt fields as lists.

    Args:
        batch: Samples from ``ProbeDataset.__getitem__``.

    Returns:
        Batched dict for ``ProbeSystem``.
    """
    out: Dict[str, Any] = {}
    for k in batch[0].keys():
        vals = [b[k] for b in batch]
        if k == 'eeg':
            out[k] = torch.stack(vals, dim=0)
        elif k == 'mask':
            out[k] = torch.stack(vals, dim=0)
        elif k in ('text uid', 'row_index'):
            out[k] = torch.tensor(vals, dtype=torch.long)
        else:
            out[k] = vals
    return out


def _dataset_text_uids(dataset: Dataset) -> List[int]:
    """Return the per-index text UID list used by ``GLIMSampler``.

    Args:
        dataset: Must expose ``text_uid``.

    Returns:
        Canonical signed-int64 UID for every dataset index.
    """
    n = len(dataset)
    if not hasattr(dataset, 'text_uid'):
        raise AssertionError('dataset missing text_uid')
    uids = dataset.text_uid  # type: ignore[attr-defined]
    out = [canonicalize_text_uid(uids[i]) for i in range(n)]
    assert len(out) == n
    return out


class GLIMSampler(Sampler[List[int]]):
    """Exhaustive batches with at most one row per text UID in each batch."""

    def __init__(
        self,
        dataset: Dataset,
        identifiers: Sequence[int],
        phase: str,
        batch_size: int,
        seed: int,
        shuffle: bool,
    ) -> None:
        """Build a UID-unique batch plan for one phase.

        Args:
            dataset: Phase dataset (length must match ``identifiers``).
            identifiers: Per-index text UIDs.
            phase: ``train``, ``val``, or ``test`` (informational).
            batch_size: Maximum rows per batch.
            seed: Epoch-0 RNG seed; later epochs add ``epoch``.
            shuffle: If True, permute rows before packing.
        """
        assert phase in ('train', 'val', 'test')
        assert batch_size >= 1
        self.dataset = dataset
        self.identifiers = torch.tensor(list(identifiers), dtype=torch.long)
        assert int(self.identifiers.numel()) == len(dataset) and len(dataset) >= 1
        self.phase = phase
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.shuffle = bool(shuffle)
        self.epoch = 0
        self.n_batches = self._estimate_len()

    def set_epoch(self, epoch: int) -> None:
        """Set the epoch used as an RNG offset.

        Args:
            epoch: Non-negative epoch index.
        """
        self.epoch = int(epoch)

    def _estimate_len(self) -> int:
        """Minimum batches so every UID group and every row can be packed.

        Returns:
            ``max(max_uid_count, ceil(n / batch_size))``.
        """
        _, counts = torch.unique(self.identifiers, return_counts=True)
        max_uid_count = int(counts.max().item())
        capacity_batches = math.ceil(len(self.dataset) / self.batch_size)
        return max(max_uid_count, capacity_batches)

    def __len__(self) -> int:
        """Number of batches produced each epoch."""
        return self.n_batches

    def __iter__(self) -> Iterator[List[int]]:
        """Yield every row exactly once in UID-unique batches, then increment epoch."""
        generator = torch.Generator().manual_seed(self.seed + self.epoch)
        order = (
            torch.randperm(len(self.dataset), generator=generator)
            if self.shuffle else torch.arange(len(self.dataset))
        )
        batches, unused = self.sample_batches(
            order, self.identifiers[order], self.batch_size,
        )
        assert not unused
        assert len(batches) == self.n_batches
        flat = [idx for batch in batches for idx in batch]
        assert len(flat) == len(self.dataset)
        assert len(set(flat)) == len(self.dataset), 'sampler repeated or omitted a row'
        self.epoch += 1
        for batch in batches:
            uid_t = self.identifiers[torch.tensor(batch, dtype=torch.long)]
            assert 1 <= len(batch) <= self.batch_size
            assert int(torch.unique(uid_t).numel()) == len(batch)
            yield batch

    @classmethod
    def sample_batches(
        cls,
        indices: torch.Tensor,
        identifiers: torch.Tensor,
        batch_size: int,
    ) -> Tuple[List[List[int]], List[List[int]]]:
        """Assign every row once to the minimum feasible number of UID-unique batches.

        Largest UID groups are placed first into the current least-full slots.

        Args:
            indices: Row indices in visit order.
            identifiers: Text UID for each ``indices`` entry.
            batch_size: Maximum rows per batch.

        Returns:
            ``(batches, [])`` — the second list is always empty (kept so the
            caller can assert that no rows were left unused).
        """
        assert indices.ndim == identifiers.ndim == 1
        assert int(indices.numel()) == int(identifiers.numel())
        assert int(batch_size) >= 1 and int(indices.numel()) >= 1
        groups: Dict[int, List[int]] = {}
        for index, uid in zip(indices.tolist(), identifiers.tolist()):
            groups.setdefault(int(uid), []).append(int(index))
        n_rows = int(indices.numel())
        n_batches = max(
            math.ceil(n_rows / int(batch_size)),
            max(len(rows) for rows in groups.values()),
        )
        batches: List[List[int]] = [[] for _ in range(n_batches)]
        ordered_groups = sorted(groups.values(), key=len, reverse=True)
        for rows in ordered_groups:
            slots = sorted(range(n_batches), key=lambda slot: (len(batches[slot]), slot))[:len(rows)]
            assert len(slots) == len(rows)
            for row_index, slot in zip(rows, slots):
                assert len(batches[slot]) < int(batch_size)
                batches[slot].append(int(row_index))
        batches.sort(key=len, reverse=True)
        flat = [idx for batch in batches for idx in batch]
        assert len(flat) == n_rows and len(set(flat)) == n_rows
        assert all(1 <= len(batch) <= int(batch_size) for batch in batches)
        return batches, []


class PhaseDataModule:
    """Train / val / test DataLoaders over a pooled ZuCo corpus."""

    def __init__(
        self,
        merged_df: pd.DataFrame,
        batch_size: int = 72,
        num_workers: int = 0,
    ) -> None:
        """Store the corpus; call ``setup`` before requesting loaders.

        Args:
            merged_df: Output of ``load_merged_corpus``.
            batch_size: GLIM / sequential batch size.
            num_workers: DataLoader workers (0 is safest with object EEG cells).
        """
        self.merged_df = merged_df
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.data_provenance = corpus_provenance(merged_df)
        self.train_set: Optional[ProbeDataset] = None
        self.val_set: Optional[ProbeDataset] = None
        self.test_set: Optional[ProbeDataset] = None

    def setup(self) -> None:
        """Build ``ProbeDataset`` objects for train, val, and test."""
        train_df = get_phase_df(self.merged_df, 'train')
        val_df = get_phase_df(self.merged_df, 'val')
        test_df = get_phase_df(self.merged_df, 'test')
        print(
            f'  [pooled] Train: {len(train_df):,}  |  Val: {len(val_df):,}  '
            f'|  Test: {len(test_df):,}'
        )
        self.train_set = ProbeDataset(train_df)
        self.val_set = ProbeDataset(val_df)
        self.test_set = ProbeDataset(test_df)

    def train_dataloader(self, seed: int = 2026) -> DataLoader:
        """Shuffled GLIM loader (unique text UID per batch).

        Args:
            seed: Sampler seed for this run.

        Returns:
            Training DataLoader.
        """
        assert self.train_set is not None
        sampler = GLIMSampler(
            self.train_set,
            _dataset_text_uids(self.train_set),
            phase='train',
            batch_size=self.batch_size,
            seed=seed,
            shuffle=True,
        )
        return DataLoader(
            self.train_set, batch_sampler=sampler,
            num_workers=self.num_workers, pin_memory=True, collate_fn=probe_collate_fn,
        )

    def val_dataloader(self, seed: int = 2026, sequential: bool = False) -> DataLoader:
        """Validation loader: GLIM by default, or sequential for t-SNE leftovers.

        Args:
            seed: Sampler seed when not sequential.
            sequential: If True, iterate rows in corpus order (no UID packing).

        Returns:
            Validation DataLoader.
        """
        assert self.val_set is not None
        if sequential:
            return DataLoader(
                self.val_set, batch_size=self.batch_size, shuffle=False,
                num_workers=self.num_workers, pin_memory=True, collate_fn=probe_collate_fn,
            )
        sampler = GLIMSampler(
            self.val_set,
            _dataset_text_uids(self.val_set),
            phase='val',
            batch_size=self.batch_size,
            seed=seed,
            shuffle=False,
        )
        return DataLoader(
            self.val_set, batch_sampler=sampler,
            num_workers=self.num_workers, pin_memory=True, collate_fn=probe_collate_fn,
        )

    def test_dataloader(self) -> DataLoader:
        """Sequential test loader (no shuffle, no GLIM packing).

        Returns:
            Test DataLoader.
        """
        assert self.test_set is not None
        return DataLoader(
            self.test_set, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers, pin_memory=True, collate_fn=probe_collate_fn,
        )


In [ ]:
# =============================================================
# SemKey EEGEncoder (raw 1280 samples, no conv downsample)
# Adapted from SemKey-main/model/modules.py (GLIM backbone).
# =============================================================

def get_1d_sincos_pos_embed_from_grid(embed_dim: int, pos: np.ndarray) -> np.ndarray:
    """Sine-cosine 1-D positional encoding for a regularly spaced grid.

    Args:
        embed_dim: Even embedding width.
        pos: 1-D positions (typically ``0 .. in_len-1``).

    Returns:
        Array of shape ``(len(pos), embed_dim)``, float32.
    """
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=np.float64)
    omega /= embed_dim / 2.0
    omega = 1.0 / 10000 ** omega
    pos = pos.reshape(-1)
    out = np.einsum('m,d->md', pos, omega)
    emb = np.concatenate([np.sin(out), np.cos(out)], axis=1)
    return emb.astype(np.float32)


class Mlp(nn.Module):
    """Two-layer GELU MLP used inside encoder / decoder blocks."""

    def __init__(self, in_features: int, hidden_features: int, drop: float = 0.0) -> None:
        """Build ``Linear → GELU → Dropout → Linear → Dropout``.

        Args:
            in_features: Input and residual width.
            hidden_features: Expansion width.
            drop: Dropout probability after each linear.
        """
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU(approximate='tanh')
        self.drop = nn.Dropout(drop)
        self.fc2 = nn.Linear(hidden_features, in_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the MLP to the last dimension.

        Args:
            x: ``(..., in_features)``.

        Returns:
            Same shape as ``x``.
        """
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class SelfAttention(nn.MultiheadAttention):
    """Batch-first self-attention with optional causal or padding mask."""

    def __init__(self, hidden_dim: int, num_heads: int, dropout: float = 0.0,
                 is_causal: bool = False) -> None:
        """Configure multi-head self-attention.

        Args:
            hidden_dim: Model width (must be divisible by ``num_heads``).
            num_heads: Attention heads.
            dropout: Attention dropout.
            is_causal: If True and no padding mask is given, apply a causal mask.
        """
        super().__init__(hidden_dim, num_heads, dropout, batch_first=True)
        self.num_heads = num_heads
        self.is_causal = is_causal

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None,
                need_weights: bool = False) -> torch.Tensor:
        """Self-attend over ``x``.

        Args:
            x: ``(B, L, D)``.
            mask: Optional ``(B, L)`` keep-mask (True = valid).
            need_weights: Unused by callers; forwarded to MHA.

        Returns:
            Attended ``(B, L, D)``.
        """
        if self.is_causal and mask is None:
            B, L, _ = x.shape
            attn_mask = torch.triu(
                torch.full((L, L), float('-inf'), dtype=x.dtype, device=x.device), diagonal=1
            ).unsqueeze(0).expand(B * self.num_heads, -1, -1)
            return super().forward(x, x, x, attn_mask=attn_mask, need_weights=need_weights)[0]
        kpm = (~mask.bool()) if mask is not None else None
        return super().forward(x, x, x, key_padding_mask=kpm, need_weights=need_weights)[0]


class CrossAttention(nn.MultiheadAttention):
    """Batch-first cross-attention from queries onto a key/value memory."""

    def __init__(self, hidden_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        """Configure multi-head cross-attention.

        Args:
            hidden_dim: Shared query / memory width.
            num_heads: Attention heads.
            dropout: Attention dropout.
        """
        super().__init__(hidden_dim, num_heads, dropout, batch_first=True)

    def forward(self, q: torch.Tensor, x: torch.Tensor, mask: Optional[torch.Tensor] = None,
                need_weights: bool = False) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """Attend queries ``q`` to memory ``x``.

        Args:
            q: ``(B, Q, D)`` queries.
            x: ``(B, L, D)`` memory.
            mask: Optional ``(B, L)`` keep-mask over memory.
            need_weights: If True, also return attention weights.

        Returns:
            ``(attended_queries, weights_or_None)``.
        """
        kpm = (~mask.bool()) if mask is not None else None
        return super().forward(q, x, x, key_padding_mask=kpm, need_weights=need_weights)


class EncoderBlock(nn.Module):
    """Prompt-conditioned transformer block with optional temporal adaLN."""

    def __init__(self, hidden_dim: int, hidden_len: int, inject_prompt: bool = True,
                 temporal_modulate: bool = True, is_causal: bool = False,
                 num_heads: int = 8, mlp_ratio: int = 4, dropout: float = 0.0) -> None:
        """Build one in-stream SemKey encoder block.

        Args:
            hidden_dim: Token width.
            hidden_len: Sequence length (for temporal adaLN).
            inject_prompt: If True, modulate norms from the prompt embedding.
            temporal_modulate: If True (and prompt injection is on), also
                apply per-timestep scale/shift from the prompt.
            is_causal: Causal self-attention (unused for the in-stream).
            num_heads: Attention heads.
            mlp_ratio: MLP expansion.
            dropout: Residual dropout.
        """
        super().__init__()
        self.inject_prompt = inject_prompt
        self.temporal_modulate = temporal_modulate
        self.norm1 = nn.LayerNorm(hidden_dim, eps=1e-6, elementwise_affine=not inject_prompt)
        self.attn = SelfAttention(hidden_dim, num_heads, dropout, is_causal=is_causal)
        self.norm2 = nn.LayerNorm(hidden_dim, eps=1e-6, elementwise_affine=not inject_prompt)
        self.mlp = Mlp(hidden_dim, hidden_dim * mlp_ratio, drop=dropout)
        if inject_prompt:
            self.adaLN = nn.Sequential(nn.SiLU(), nn.Linear(hidden_dim, 6 * hidden_dim, bias=True))
            nn.init.zeros_(self.adaLN[1].weight)
            nn.init.zeros_(self.adaLN[1].bias)
            if temporal_modulate:
                self.t_adaLN = nn.Sequential(nn.SiLU(), nn.Linear(hidden_dim, 2 * hidden_len, bias=True))
                nn.init.zeros_(self.t_adaLN[1].weight)
                nn.init.zeros_(self.t_adaLN[1].bias)

    def modulate(self, x: torch.Tensor, shift: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
        """Channel-wise affine: ``x * (1 + scale) + shift``.

        Args:
            x: ``(B, L, D)``.
            shift: ``(B, D)``.
            scale: ``(B, D)``.

        Returns:
            Modulated ``x``.
        """
        return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

    def t_modulate(self, x: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
        """Time-wise affine from the prompt embedding.

        Args:
            x: ``(B, L, D)``.
            p: ``(B, D)`` prompt embedding.

        Returns:
            Temporally modulated ``x``.
        """
        scale, shift = self.t_adaLN(p).chunk(2, dim=1)
        return x * (1 + scale.unsqueeze(2)) + shift.unsqueeze(2)

    def forward(self, x: torch.Tensor, mask: torch.Tensor, p: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Run attention + MLP with optional prompt modulation.

        Args:
            x: ``(B, L, D)`` tokens.
            mask: ``(B, L)`` keep-mask.
            p: ``(B, D)`` prompt embedding when ``inject_prompt`` is True.

        Returns:
            Updated tokens, same shape as ``x``.
        """
        if not self.inject_prompt:
            x = x + self.attn(self.norm1(x), mask)
            return x + self.mlp(self.norm2(x))
        if self.temporal_modulate:
            x = self.t_modulate(x, p)
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.adaLN(p).chunk(6, dim=1)
        x = x + gate_msa.unsqueeze(1) * self.attn(self.modulate(self.norm1(x), shift_msa, scale_msa), mask)
        x = x + gate_mlp.unsqueeze(1) * self.mlp(self.modulate(self.norm2(x), shift_mlp, scale_mlp))
        return x


class DecoderBlock(nn.Module):
    """Causal self-attention + cross-attention Q-Merger block."""

    def __init__(self, dim: int, num_heads: int = 8, mlp_ratio: int = 4,
                 dropout: float = 0.0, is_causal: bool = True) -> None:
        """Build one out-stream (query) block.

        Args:
            dim: Query / memory width.
            num_heads: Attention heads.
            mlp_ratio: MLP expansion.
            dropout: Residual dropout.
            is_causal: Causal mask on query self-attention.
        """
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.self_attn = SelfAttention(dim, num_heads, dropout, is_causal=is_causal)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.cross_attn = CrossAttention(dim, num_heads, dropout)
        self.norm3 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = Mlp(dim, dim * mlp_ratio, drop=dropout)

    def forward(self, q: torch.Tensor, x: torch.Tensor, x_mask: torch.Tensor,
                need_weights: bool = False) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """Update queries by attending to the EEG memory.

        Args:
            q: ``(B, Q, D)`` learned queries.
            x: ``(B, L, D)`` encoder memory.
            x_mask: ``(B, L)`` keep-mask over memory.
            need_weights: If True, return cross-attention weights.

        Returns:
            ``(updated_queries, cross_attn_weights_or_None)``.
        """
        q = q + self.self_attn(self.norm1(q), need_weights=need_weights)
        attn_out, attn_w = self.cross_attn(self.norm2(q), x, x_mask, need_weights=need_weights)
        q = q + attn_out
        q = q + self.mlp(self.norm3(q))
        return q, attn_w


class PromptEmbedder(nn.Module):
    """Sum of task / dataset / subject embeddings, weighted by learned σ."""

    def __init__(self, dim: int = 128, prompt_keys: Optional[Dict[str, List[str]]] = None,
                 drop_probs: Tuple[float, float, float] = (0.0, 0.0, 0.0)) -> None:
        """Create one embedding table per prompt field.

        Args:
            dim: Embedding width (must match encoder ``in_dim``).
            prompt_keys: Token inventories; defaults to ``PROMPT_KEYS``.
            drop_probs: Per-field dropout to the ``<UNK>`` id during training.
        """
        super().__init__()
        self.prompt_keys = prompt_keys or PROMPT_KEYS
        prompt_nums = tuple(len(v) for v in self.prompt_keys.values())
        self.dim = dim
        self.drop_probs = drop_probs
        self.embedders = nn.ModuleList([nn.Embedding(n, dim) for n in prompt_nums])
        for emb in self.embedders:
            nn.init.normal_(emb.weight, std=0.02)
        self.sigma = nn.Parameter(torch.tensor([n / sum(prompt_nums) for n in prompt_nums], dtype=torch.float32))

    @torch.no_grad()
    def p_drop(self, src_pids: torch.Tensor, drop_prob: float) -> torch.Tensor:
        """Randomly replace prompt ids with 0 (``<UNK>``).

        Args:
            src_pids: Integer ids ``(B,)``.
            drop_prob: Independent drop probability.

        Returns:
            Possibly dropped ids, same shape.
        """
        if drop_prob <= 0.0:
            return src_pids
        drop_mask = torch.rand(src_pids.shape, device=src_pids.device) < drop_prob
        return torch.where(drop_mask, torch.zeros_like(src_pids), src_pids)

    def forward(self, prompt_ids: torch.Tensor,
                eval_pembed: Literal['zero', 'sum', 'mean', 'src'] = 'src') -> torch.Tensor:
        """Embed discrete prompt ids and sum σ-weighted fields.

        Args:
            prompt_ids: ``(B, 3)`` integer ids for task, dataset, subject.
            eval_pembed: Unused (kept for SemKey API compatibility).

        Returns:
            Prompt vector ``(B, dim)``.
        """
        p_embed = torch.zeros(prompt_ids.shape[0], self.dim, device=prompt_ids.device)
        for k, (embedder, prob) in enumerate(zip(self.embedders, self.drop_probs)):
            pids = prompt_ids[:, k]
            if self.training and prob > 0.0:
                pids = self.p_drop(pids, prob)
            p = embedder(pids)
            p_embed = p_embed + p * self.sigma[k]
        return p_embed

    def encode(self, prompts: List[List[str]], device: Optional[torch.device] = None) -> torch.Tensor:
        """Map string prompt columns to integer ids.

        Args:
            prompts: Three lists ``[tasks, datasets, subjects]``, each length B.
            device: Device for the returned LongTensor.

        Returns:
            ``(B, 3)`` ids; unknown strings map to 0.
        """
        bsz = len(prompts[0])
        ids = []
        for k, keys in enumerate(self.prompt_keys.values()):
            row = [keys.index(prompts[k][i]) if prompts[k][i] in keys else 0 for i in range(bsz)]
            ids.append(torch.tensor(row, dtype=torch.long, device=device))
        return torch.stack(ids, dim=-1)


class EEGEncoder(nn.Module):
    """Prompt-modulated encoder + learned-query Q-Merger on raw 1280 samples."""

    def __init__(
        self,
        in_len: int = 1280,
        out_len: int = 96,
        in_dim: int = 128,
        out_dim: int = 128,
        n_in_blocks: int = 6,
        n_out_blocks: int = 6,
        num_heads: int = 8,
        mlp_ratio: int = 4,
        dropout: float = 0.0,
        use_channel_weights: bool = True,
    ) -> None:
        """Build the SemKey encoder used by ``ProbeSystem``.

        Args:
            in_len: EEG time samples (1280).
            out_len: Q-Merger tokens (96).
            in_dim: Channel / prompt width (128).
            out_dim: Merger output width (``hidden_dim``).
            n_in_blocks: In-stream encoder depth.
            n_out_blocks: Query-merger depth.
            num_heads: Attention heads.
            mlp_ratio: MLP expansion.
            dropout: Block dropout.
            use_channel_weights: If True, learn a per-channel scale.
        """
        super().__init__()
        self.in_len = in_len
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.out_len = out_len
        self.use_channel_weights = use_channel_weights
        block_kw = dict(num_heads=num_heads, mlp_ratio=mlp_ratio, dropout=dropout)
        self.in_blocks = nn.ModuleList([
            EncoderBlock(in_dim, in_len, inject_prompt=True, temporal_modulate=True,
                         is_causal=False, **block_kw)
            for _ in range(n_in_blocks)
        ])
        self.x_proj = nn.Linear(in_dim, out_dim)
        self.norm1 = nn.LayerNorm(out_dim, eps=1e-6)
        self.shared_queries = nn.Parameter(torch.randn(1, out_len, out_dim) * 0.02)
        self.out_blocks = nn.ModuleList([
            DecoderBlock(out_dim, is_causal=True, **block_kw) for _ in range(n_out_blocks)
        ])
        self.norm2 = nn.LayerNorm(out_dim, eps=1e-6)
        pos = get_1d_sincos_pos_embed_from_grid(in_dim, np.arange(in_len))
        self.register_buffer('pos_embed', torch.from_numpy(pos).unsqueeze(0), persistent=False)
        if use_channel_weights:
            self.channel_weights = nn.Parameter(torch.ones(1, 1, in_dim))

    def forward(self, eeg: torch.Tensor, mask: torch.Tensor, p: torch.Tensor,
                need_weights: bool = False) -> Tuple[torch.Tensor, torch.Tensor, dict]:
        """Encode EEG to 96 merger tokens.

        Args:
            eeg: ``(B, 1280, 128)``.
            mask: ``(B, 1280)`` keep-mask.
            p: ``(B, 128)`` prompt embedding.
            need_weights: If True, collect per-block cross-attention weights.

        Returns:
            ``(Zi, memory, attn_weights)`` where ``Zi`` is ``(B, 96, out_dim)``.
        """
        assert eeg.ndim == 3 and eeg.shape[1:] == (self.in_len, self.in_dim)
        assert mask.shape == eeg.shape[:2]
        assert p.shape == (eeg.shape[0], self.in_dim)
        x = eeg * self.channel_weights if self.use_channel_weights else eeg
        x = x + self.pos_embed
        for block in self.in_blocks:
            x = block(x, mask, p)
        memory = self.norm1(self.x_proj(x))
        q = self.shared_queries.expand(eeg.shape[0], -1, -1)
        attn_weights: dict = {}
        for j, block in enumerate(self.out_blocks):
            q, aw = block(q, memory, mask, need_weights=need_weights)
            attn_weights[j] = aw
        Zi = self.norm2(q)
        assert Zi.shape == (eeg.shape[0], self.out_len, self.out_dim)
        assert not torch.isnan(Zi).any(), 'NaN in Zi'
        return Zi, memory, attn_weights


def build_eeg_encoder(config: Dict[str, Any]) -> EEGEncoder:
    """Construct the SemKey encoder from the notebook CONFIG.

    Args:
        config: Must include ``in_len``, ``out_len``, ``in_dim``, ``hidden_dim``,
            block counts, ``num_heads``, ``mlp_ratio``, ``dropout``, and
            ``use_channel_weights``.

    Returns:
        Uninitialized ``EEGEncoder`` (weights are random until training).
    """
    return EEGEncoder(
        in_len=int(config['in_len']),
        out_len=int(config['out_len']),
        in_dim=int(config['in_dim']),
        out_dim=int(config['hidden_dim']),
        n_in_blocks=int(config['n_in_blocks']),
        n_out_blocks=int(config['n_out_blocks']),
        num_heads=int(config['num_heads']),
        mlp_ratio=int(config['mlp_ratio']),
        dropout=float(config['dropout']),
        use_channel_weights=bool(config['use_channel_weights']),
    )


In [ ]:
# =============================================================
# ProbeSystem: CLIP + AR + EncDec KV (Language Prior)
# =============================================================

def set_seed(seed: int = 2026) -> None:
    """Seed Python/NumPy/PyTorch and request safe deterministic kernels.

    Args:
        seed: Integer seed written to CUDA, cuDNN, and the CPU RNGs.
    """
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def move_to_device(batch: Dict[str, Any], device: torch.device) -> Dict[str, Any]:
    """Copy tensor values in a batch dict onto ``device``; leave other fields as-is.

    Args:
        batch: Collated sample dict.
        device: Destination torch device.

    Returns:
        Shallow-copied batch with tensors moved via ``non_blocking=True``.
    """
    out: Dict[str, Any] = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device, non_blocking=True)
        else:
            out[k] = v
    return out


def pool_seq(x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
    """Mean-pool a sequence, optionally ignoring padded steps.

    Args:
        x: Token states ``(B, L, D)``.
        mask: Optional keep-mask ``(B, L)`` (True / 1 = valid).

    Returns:
        Pooled vectors ``(B, D)``.
    """
    assert x.ndim == 3
    if mask is None:
        return x.mean(dim=1)
    m = mask.bool().unsqueeze(-1).float()
    return (x * m).sum(dim=1) / m.sum(dim=1).clamp_min(1.0)


def clip_info_nce(
    ei: torch.Tensor,
    yi: torch.Tensor,
    sample_weights: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """SemKey symmetric CLIP InfoNCE on pooled embeddings.

    Args:
        ei: Source (EEG / KV) pooled vectors ``(B, D)``.
        yi: Frozen text pooled vectors ``(B, D)``.
        sample_weights: Optional per-row weights ``(B,)``.

    Returns:
        Scalar symmetric cross-entropy loss.
    """
    assert ei.ndim == 2 and yi.ndim == 2 and ei.shape == yi.shape
    bsz = int(ei.shape[0])
    assert bsz >= 1
    x_normed = ei / ei.norm(dim=1, keepdim=True).clamp_min(1e-8)
    y_normed = yi / yi.norm(dim=1, keepdim=True).clamp_min(1e-8)
    x_logits = x_normed @ y_normed.T
    y_logits = x_logits.T
    target = torch.arange(bsz, device=ei.device)
    x_logits = F.normalize(x_logits, p=2, dim=-1)
    y_logits = F.normalize(y_logits, p=2, dim=-1)
    per = (
        F.cross_entropy(x_logits, target, reduction='none')
        + F.cross_entropy(y_logits, target, reduction='none')
    ) / 2.0
    if sample_weights is None:
        loss = per.mean()
    else:
        w = sample_weights.to(device=ei.device, dtype=per.dtype).view(-1)
        assert w.shape[0] == bsz
        denom = w.sum().clamp_min(1e-8)
        loss = (per * w).sum() / denom
    assert not torch.isnan(loss).any()
    return loss


def token_commitment_mse(
    z_src: torch.Tensor,
    h_tgt: torch.Tensor,
    text_mask: torch.Tensor,
) -> torch.Tensor:
    """SemKey L2 commitment between normalized tokens, masked by text pads.

    Args:
        z_src: EEG / mixed KV tokens ``(B, L, D)``.
        h_tgt: Frozen text encoder states ``(B, L, D)``.
        text_mask: ``(B, L)`` with 1 on real text tokens.

    Returns:
        Scalar MSE averaged over valid elements (tokens × dim).
    """
    assert z_src.ndim == 3 and h_tgt.ndim == 3
    assert z_src.shape == h_tgt.shape
    assert text_mask.shape == z_src.shape[:2]
    x = F.normalize(z_src, p=2, dim=-1)
    y = F.normalize(h_tgt, p=2, dim=-1)
    valid = text_mask.to(device=x.device, dtype=x.dtype).unsqueeze(-1)
    valid_tokens = valid.sum()
    assert float(valid_tokens.detach().cpu()) > 0.0, 'commitment mask has no valid tokens'
    squared = (x - y).pow(2) * valid
    denom = valid_tokens * int(x.shape[-1])
    loss = squared.sum() / denom
    assert loss.ndim == 0 and torch.isfinite(loss)
    return loss


def full_target_rollout_mask(labels: torch.Tensor) -> torch.Tensor:
    """Gold-token scoring mask; intentionally independent of generated EOS.

    Args:
        labels: Teacher-forced / rollout label ids with ``-100`` pads.

    Returns:
        Boolean mask ``labels != -100``.
    """
    assert labels.ndim == 2
    return labels.ne(-100)


def mix_embedding_noise(
    emb: torch.Tensor,
    alpha: float,
    generator: Optional[torch.Generator] = None,
) -> torch.Tensor:
    """Mix embeddings with isotropic Gaussian noise: ``(1-α)*emb + α*ε``.

    Used for probe ``noise50`` (α=0.5) and ``noise100`` (α=1.0). α=0 is a no-op.

    Args:
        emb: Any tensor with ndim >= 2.
        alpha: Mix weight in ``[0, 1]``.
        generator: Optional CPU generator for reproducible noise.

    Returns:
        Mixed tensor, same shape and device as ``emb``.
    """
    assert emb.ndim >= 2
    a = float(alpha)
    assert 0.0 <= a <= 1.0
    if a <= 0.0:
        return emb
    if generator is None:
        noise = torch.randn_like(emb)
    else:
        noise = torch.randn(
            emb.shape, dtype=torch.float32, generator=generator,
        ).to(device=emb.device, dtype=emb.dtype)
    if a >= 1.0:
        return noise
    out = (1.0 - a) * emb + a * noise
    assert out.shape == emb.shape
    return out


def setup_scheduler(
    optimizer: torch.optim.Optimizer, config: Dict[str, Any],
) -> CosineAnnealingWarmRestarts:
    """Epoch-level cosine annealing with warm restarts.

    Args:
        optimizer: Optimizer whose LR is scheduled.
        config: Uses ``lr_restart_epochs`` (T_0) and ``min_lr``.

    Returns:
        ``CosineAnnealingWarmRestarts`` with ``T_mult=1``.
    """
    return CosineAnnealingWarmRestarts(
        optimizer,
        T_0=int(config.get('lr_restart_epochs', 15)),
        T_mult=1,
        eta_min=float(config.get('min_lr', 1e-9)),
    )


def _print_disk_usage(path: Optional[str]) -> None:
    """Print free/total disk space (GB) for path or its nearest existing parent."""
    if not path:
        print('  [disk] (no cache path)')
        return
    p = Path(path).resolve()
    probe = p if p.exists() else p.parent
    while not probe.exists() and probe != probe.parent:
        probe = probe.parent
    try:
        usage = shutil.disk_usage(str(probe))
        free_gb = usage.free / (1024 ** 3)
        total_gb = usage.total / (1024 ** 3)
        print(f'  [disk] {probe}: free={free_gb:.1f} GB / total={total_gb:.1f} GB')
    except Exception as exc:
        print(f'  [disk] unavailable for {probe}: {exc}')


def _hub_snapshot_dir(repo_id: str, cache_dir: Optional[str]) -> Optional[Path]:
    """Return a complete local Hub snapshot, or None if missing/incomplete."""
    name = 'models--' + repo_id.replace('/', '--')
    tok_markers = (
        'tokenizer.json', 'vocab.json', 'spiece.model',
        'tokenizer.model', 'merges.txt',
    )
    roots: List[Path] = []
    if cache_dir:
        roots.append(Path(cache_dir))
    for env_key in ('HUGGINGFACE_HUB_CACHE', 'HF_HUB_CACHE'):
        v = os.environ.get(env_key)
        if v:
            roots.append(Path(v))
    roots.append(Path.home() / '.cache' / 'huggingface' / 'hub')
    roots.append(Path.home() / '.cache' / 'huggingface')

    seen: set[str] = set()
    for root in roots:
        key = str(root.resolve()) if root.exists() else str(root)
        if key in seen:
            continue
        seen.add(key)
        for base in (root / name, root / 'hub' / name):
            snaps = base / 'snapshots'
            if not snaps.is_dir():
                continue
            for snap in snaps.iterdir():
                if not snap.is_dir():
                    continue
                if not (snap / 'config.json').is_file():
                    continue
                if list(snap.glob('*.incomplete')):
                    continue
                if not any((snap / m).is_file() for m in tok_markers):
                    continue
                has_weights = (
                    any(snap.glob('*.safetensors'))
                    or any(snap.glob('pytorch_model*.bin'))
                    or any(snap.glob('model.safetensors.index.json'))
                )
                if not has_weights:
                    continue
                return snap
    return None


def download_bart_backbone_safetensors(
    model_id: str = 'AI-ModelScope/bart-large',
    cache_dir: Optional[str] = None,
    max_retries: int = 5,
    backoff_s: float = 30.0,
) -> str:
    """Download BART via ModelScope and ensure model.safetensors exists (one-time convert)."""
    from modelscope import snapshot_download  # lazy import keeps encoder smoke offline

    n_retry = max(1, int(max_retries))
    backoff = float(backoff_s)
    last_exc: Optional[BaseException] = None
    model_dir: Optional[str] = None
    for attempt in range(1, n_retry + 1):
        try:
            print(f'  [ModelScope] snapshot_download({model_id}) (attempt {attempt}/{n_retry})')
            model_dir = snapshot_download(
                model_id,
                cache_dir=cache_dir,
                allow_file_pattern=['*.json', '*.txt', 'pytorch_model.bin', '*.safetensors'],
            )
            break
        except Exception as exc:
            last_exc = exc
            print(f'  [retry {attempt}/{n_retry}] ModelScope failed for {model_id}: {exc}')
            if attempt < n_retry:
                sleep_s = backoff * float(attempt)
                print(f'  [retry] sleeping {sleep_s:.1f}s before next attempt ...')
                time.sleep(sleep_s)
    if model_dir is None:
        raise RuntimeError(
            f'Failed to download {model_id} from ModelScope after {n_retry} attempts'
        ) from last_exc

    safetensors_path = Path(model_dir) / 'model.safetensors'
    if not safetensors_path.exists():
        import safetensors.torch  # lazy import

        bin_path = Path(model_dir) / 'pytorch_model.bin'
        assert bin_path.is_file(), f'Missing checkpoint: {bin_path}'
        try:
            state_dict = torch.load(bin_path, map_location='cpu', weights_only=True)
        except TypeError:
            state_dict = torch.load(bin_path, map_location='cpu')
        tensors = {key: value for key, value in state_dict.items() if isinstance(value, torch.Tensor)}
        safetensors.torch.save_file(tensors, str(safetensors_path))
        print(f'  [ModelScope] wrote {safetensors_path}')
    return str(model_dir)


def load_frozen_llm(
    repo_id: str,
    family: str,
    cache_dir: Optional[str],
    device: torch.device,
    max_retries: int = 5,
    backoff_s: float = 30.0,
    source: str = 'hf_mirror',
    modelscope_id: Optional[str] = None,
) -> Tuple[Any, Any, int, str]:
    """
    Load tokenizer + frozen EncDec LLM.

    Flan-T5: Hub/HF-mirror retries. BART: ModelScope snapshot + local safetensors.

    Returns:
        tokenizer, model, hidden_dim, family
    """
    _print_disk_usage(cache_dir)
    use_bf16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
    dtype = torch.bfloat16 if use_bf16 else torch.float32
    n_retry = max(1, int(max_retries))
    backoff = float(backoff_s)

    def _load_t5(load_id: Union[str, Path], local_only: bool) -> Tuple[Any, Any, int]:
        """Load Flan-T5 tokenizer + model from a Hub id or local snapshot.

        Args:
            load_id: Hugging Face repo id or local directory.
            local_only: If True, do not hit the network.

        Returns:
            ``(tokenizer, model, hidden_dim)``.
        """
        kwargs: Dict[str, Any] = {'use_safetensors': True}
        if cache_dir:
            kwargs['cache_dir'] = cache_dir
        if local_only:
            kwargs['local_files_only'] = True
        tok = AutoTokenizer.from_pretrained(load_id, **kwargs)
        model = T5ForConditionalGeneration.from_pretrained(load_id, dtype=dtype, **kwargs)
        hidden = int(model.config.d_model)
        return tok, model, hidden

    def _load_hub_t5_with_retry() -> Tuple[Any, Any, int]:
        """Download Flan-T5 from the Hub/mirror with exponential backoff.

        Returns:
            ``(tokenizer, model, hidden_dim)``.

        Raises:
            RuntimeError: After ``n_retry`` failed Hub attempts.
        """
        last_exc: Optional[BaseException] = None
        for attempt in range(1, n_retry + 1):
            try:
                print(f'  Loading LLM from Hub/mirror: {repo_id} (attempt {attempt}/{n_retry})')
                return _load_t5(repo_id, local_only=False)
            except Exception as exc:
                last_exc = exc
                print(f'  [retry {attempt}/{n_retry}] Hub load failed for {repo_id}: {exc}')
                if attempt < n_retry:
                    sleep_s = backoff * float(attempt)
                    print(f'  [retry] sleeping {sleep_s:.1f}s before next attempt ...')
                    time.sleep(sleep_s)
        raise RuntimeError(
            f'Failed to load {repo_id} after {n_retry} Hub attempts'
        ) from last_exc

    if family == 'encdec_bart' or source == 'modelscope':
        ms_id = modelscope_id or 'AI-ModelScope/bart-large'
        model_dir = download_bart_backbone_safetensors(
            ms_id, cache_dir=cache_dir, max_retries=n_retry, backoff_s=backoff,
        )
        print(f'  Loading BART from ModelScope dir: {model_dir}')
        tok = BartTokenizer.from_pretrained(model_dir)
        model = BartForConditionalGeneration.from_pretrained(
            model_dir, dtype=dtype, use_safetensors=True,
        )
        hidden = int(model.config.d_model)
        family = 'encdec_bart'
    elif family == 'encdec_t5':
        local = _hub_snapshot_dir(repo_id, cache_dir)
        if local is not None:
            print(f'  Loading LLM from local snapshot: {local}')
            try:
                tok, model, hidden = _load_t5(local, local_only=True)
            except Exception as exc:
                print(f'  [WARN] local snapshot load failed ({exc}); falling back to Hub id')
                tok, model, hidden = _load_hub_t5_with_retry()
        else:
            tok, model, hidden = _load_hub_t5_with_retry()
    else:
        raise ValueError(f'Unknown family/source: family={family} source={source}')

    model.to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    print(f'  Frozen LLM {repo_id} family={family} hidden={hidden} dtype={dtype}')
    return tok, model, hidden, family


def assemble_encdec_memory(
    z_src: torch.Tensor,
    ei: Optional[torch.Tensor],
    token_mask: Optional[torch.Tensor],
    prepend_ei: bool,
    use_ei: bool,
    model_dtype: torch.dtype,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Pack KV tokens + mask for EncDec encoder_outputs (decoder cross-attn K/V).

    gt_text: prepend_ei=False, token_mask=encoder pad mask → length 96.
    EEG/noise: prepend_ei follows use_ei; token_mask typically all ones.
    """
    assert z_src.ndim == 3
    B, L = int(z_src.shape[0]), int(z_src.shape[1])
    if token_mask is None:
        tok_m = torch.ones((B, L), device=z_src.device, dtype=torch.long)
    else:
        tok_m = token_mask.to(device=z_src.device, dtype=torch.long)
        assert tok_m.shape == (B, L), (tuple(tok_m.shape), (B, L))
    do_ei = bool(prepend_ei) and bool(use_ei) and ei is not None
    if do_ei:
        raw = torch.cat([ei.unsqueeze(1), z_src], dim=1)
        ones = torch.ones((B, 1), device=z_src.device, dtype=torch.long)
        enc_mask = torch.cat([ones, tok_m], dim=1)
    else:
        raw = z_src
        enc_mask = tok_m
    assert raw.shape[:2] == enc_mask.shape
    return raw.to(dtype=model_dtype), enc_mask


class ProbeSystem(nn.Module):
    """
    Trainable EEG encoder/projector + frozen EncDec LLM for CLIP, commitment, AR, and KV injection.
    """

    def __init__(
        self,
        config: Dict[str, Any],
        llm_key: str,
        repo_id: str,
        family: str,
        device: torch.device,
        source: str = 'hf_mirror',
        modelscope_id: Optional[str] = None,
    ) -> None:
        """Load a frozen EncDec LLM and randomly init the trainable EEG path.

        Args:
            config: Notebook CONFIG (encoder sizes, loss weights, cache).
            llm_key: Short id such as ``bart_large``.
            repo_id: Hugging Face repo id.
            family: ``encdec_bart`` or ``encdec_t5``.
            device: Compute device.
            source: ``hf_mirror`` or ``modelscope``.
            modelscope_id: ModelScope snapshot id when ``source='modelscope'``.
        """
        set_seed(int(config.get('seed', 2026)))
        super().__init__()
        self.config = config
        self.llm_key = llm_key
        self.repo_id = repo_id
        self.family = family
        self.source = str(source)
        self.modelscope_id = modelscope_id
        self.device = device
        self.data_key = 'eeg'
        cache_dir = config.get('model_cache_dir')
        self.tokenizer, self.llm, embed_dim, self.family = load_frozen_llm(
            repo_id,
            family,
            cache_dir,
            device,
            max_retries=int(config.get('llm_load_max_retries', 5)),
            backoff_s=float(config.get('llm_load_retry_backoff_s', 30.0)),
            source=source,
            modelscope_id=modelscope_id,
        )
        self.embed_dim = int(embed_dim)
        self.model_dtype = next(self.llm.parameters()).dtype

        self.max_target_tokens = int(config.get('max_target_tokens', 64))
        self.out_len = int(config.get('out_len', 96))
        self.input_text_len = int(config.get('input_text_len', self.out_len))
        assert self.input_text_len == self.out_len
        self.commitment_weight = float(config.get('commitment_weight', 0.7))
        self.w_clip = float(config.get('w_clip', 0.5))
        self.w_ar = float(config.get('w_ar', 0.5))
        self.use_ei = bool(config.get('use_ei', True))
        self.decoder_prompt = str(config.get(
            'decoder_prompt',
            'Based on the following signals, translate the sentence',
        ))
        self._build_trainable()

    def _build_trainable(self) -> None:
        """(Re)initialize prompt embedder, SemKey encoder, in_proj, identity projector."""
        cfg = self.config
        set_seed(int(cfg.get('seed', 2026)))
        self.prompt_embedder = PromptEmbedder(
            dim=int(cfg['prompt_dim']),
            prompt_keys=PROMPT_KEYS,
            drop_probs=tuple(cfg.get('prompt_drop_probs', (0.0, 0.0, 0.0))),
        )
        self.eeg_encoder = build_eeg_encoder(cfg)
        self.in_proj = nn.Linear(int(cfg['hidden_dim']), self.embed_dim)
        self.projector = nn.Linear(self.embed_dim, self.embed_dim)
        with torch.no_grad():
            self.projector.weight.copy_(torch.eye(self.embed_dim))
            self.projector.bias.zero_()

    def reset_trainable(self) -> None:
        """Fresh encoder/projector weights for the eeg train run; LLM stays frozen."""
        set_seed(int(self.config.get('seed', 2026)))
        self._build_trainable()
        self.to(self.device)
        self.llm.eval()
        for p in self.llm.parameters():
            p.requires_grad = False

    def set_data_key(self, data_key: str) -> None:
        """Train encoder/projector only for clean eeg (α=0). Freeze on gt_text and noise*."""
        assert data_key in DATA_KEY_ALPHA, data_key
        self.data_key = str(data_key)
        alpha = DATA_KEY_ALPHA[data_key]
        train_enc = alpha is not None and float(alpha) == 0.0
        for mod in (self.prompt_embedder, self.eeg_encoder, self.in_proj, self.projector):
            for p in mod.parameters():
                p.requires_grad = train_enc

    def trainable_parameters(self) -> List[nn.Parameter]:
        """Collect encoder / projector parameters that currently require grad.

        Returns:
            List of ``nn.Parameter`` (empty when the path is frozen, e.g. gt_text).
        """
        params: List[nn.Parameter] = []
        for mod in (self.prompt_embedder, self.eeg_encoder, self.in_proj, self.projector):
            params.extend([p for p in mod.parameters() if p.requires_grad])
        return params

    def encode_prompts(self, batch: Dict[str, Any]) -> torch.Tensor:
        """Convert batch prompt tuples into integer ids for ``PromptEmbedder``.

        Args:
            batch: Must contain ``prompt`` as a list of ``(task, dataset, subject)``.

        Returns:
            LongTensor ``(B, 3)`` on ``self.device``.
        """
        prompts = batch['prompt']
        tasks, datasets, subjects = unpack_prompt_batch(prompts)
        packed = [tasks, datasets, subjects]
        return self.prompt_embedder.encode(packed, device=self.device)

    def encode_eeg(
        self, batch: Dict[str, Any],
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            z_src: (B, out_len, D)
            ei_eeg: (B, D)
        """
        eeg = batch['eeg']
        mask = batch['mask']
        p_ids = self.encode_prompts(batch)
        p = self.prompt_embedder(p_ids)
        Zi, _, _ = self.eeg_encoder(eeg, mask, p)
        assert Zi.shape[:2] == (eeg.shape[0], self.out_len)
        z_src = self.projector(self.in_proj(Zi))
        ei_eeg = z_src.mean(dim=1)
        assert z_src.shape[-1] == self.embed_dim
        assert ei_eeg.shape == (z_src.shape[0], self.embed_dim)
        return z_src, ei_eeg

    @torch.no_grad()
    def encode_text(
        self, texts: List[str],
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns:
            h_tgt: (B, input_text_len, D)
            text_mask: (B, input_text_len)
            ei_text: (B, D)
        """
        prev_side = getattr(self.tokenizer, 'padding_side', 'right')
        self.tokenizer.padding_side = 'right'
        try:
            tok = self.tokenizer(
                texts,
                padding='max_length',
                truncation=True,
                max_length=self.input_text_len,
                return_tensors='pt',
            )
        finally:
            self.tokenizer.padding_side = prev_side
        input_ids = tok['input_ids'].to(self.device)
        attn = tok['attention_mask'].to(self.device)
        assert input_ids.shape[1] == self.input_text_len
        enc = self.llm.get_encoder()
        out = enc(input_ids=input_ids, attention_mask=attn, return_dict=True)
        h = out.last_hidden_state.to(dtype=torch.float32)
        assert h.shape[:2] == input_ids.shape
        ei = pool_seq(h, attn)
        return h, attn, ei

    def encode_kv(
        self,
        batch: Dict[str, Any],
        data_key: str,
        mix_seed: Optional[int] = None,
    ) -> Tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor,
    ]:
        """
        Build KV tokens for one data_key. One mix sample is shared by CLIP/commit/AR.

        Returns:
            z_src: (B, 96, D) KV tokens
            ei_kv: (B, D) pooled KV
            h_tgt: (B, 96, D) frozen text states
            text_mask: (B, 96) 1=real frozen-text token
            ei_text: (B, D)
            kv_mask: (B, 96) 1=valid cross-attention token
        """
        texts = [str(t) for t in batch.get('target text', batch['input text'])]
        h_tgt, text_mask, ei_text = self.encode_text(texts)
        if data_key == 'gt_text':
            assert text_mask.shape[:2] == h_tgt.shape[:2]
            return h_tgt, ei_text, h_tgt, text_mask, ei_text, text_mask
        z_src, _ei = self.encode_eeg(batch)
        alpha = DATA_KEY_ALPHA[data_key]
        assert alpha is not None
        if float(alpha) > 0.0:
            gen: Optional[torch.Generator] = None
            if mix_seed is not None:
                gen = torch.Generator()
                gen.manual_seed(int(mix_seed))
            z_src = mix_embedding_noise(z_src, float(alpha), generator=gen)
        ei_kv = z_src.mean(dim=1)
        assert z_src.shape[1] == h_tgt.shape[1] == self.out_len
        kv_mask = torch.ones(
            (z_src.shape[0], z_src.shape[1]), device=z_src.device, dtype=torch.long,
        )
        return z_src, ei_kv, h_tgt, text_mask, ei_text, kv_mask

    def build_decoder_memory(
        self,
        z_src: torch.Tensor,
        ei: Optional[torch.Tensor] = None,
        token_mask: Optional[torch.Tensor] = None,
        prepend_ei: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """KV memory for EncDec encoder_outputs.

        gt_text: prepend_ei=False and token_mask is the encoder pad mask.
        EEG/noise: prepend_ei follows use_ei; all-ones token mask.
        """
        return assemble_encdec_memory(
            z_src,
            ei,
            token_mask,
            prepend_ei=prepend_ei,
            use_ei=self.use_ei,
            model_dtype=self.model_dtype,
        )

    def _instruction_ids(self) -> List[int]:
        """Token ids for CONFIG decoder_prompt (no BOS/EOS)."""
        text = str(self.decoder_prompt).strip()
        if not text:
            return []
        start = self._decoder_start_id()
        eos = self.tokenizer.eos_token_id
        ids = list(self.tokenizer(
            text, add_special_tokens=False, padding=False, truncation=True,
            max_length=self.max_target_tokens,
        )['input_ids'])
        if ids and eos is not None and ids[-1] == int(eos):
            ids = ids[:-1]
        if ids and start is not None and ids[0] == int(start):
            ids = ids[1:]
        return ids

    def _prepare_lexical_decoder_text(
        self,
        text: str,
        follows_instruction: bool,
    ) -> str:
        """Apply family-aware text-boundary semantics before lexical tokenization."""
        lexical = str(text)
        if self.family == 'encdec_bart' and bool(follows_instruction) and lexical:
            return ' ' + lexical.lstrip()
        return lexical

    def _native_gt_path(self, data_key: str) -> bool:
        """True when KV is frozen encoder(GT): no instruction, no ei prepend."""
        return str(data_key) == 'gt_text'

    def _teacher_forced_ar(
        self,
        batch: Dict[str, Any],
        z_src: torch.Tensor,
        ei_kv: torch.Tensor,
        data_key: str = 'eeg',
        kv_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Teacher-forced CE with family-canonical masked prompt then lexical target/EOS."""
        native = self._native_gt_path(data_key)
        enc_hidden, enc_mask = self.build_decoder_memory(
            z_src, ei_kv, token_mask=kv_mask, prepend_ei=not native,
        )
        B = int(z_src.shape[0])
        pad = int(self.tokenizer.pad_token_id)
        prompt_ids_list, prompt_lens, target_ids_list = self._tokenize_prompt_target_ids(
            batch, include_instruction=not native,
        )
        merged: List[List[int]] = [p + t for p, t in zip(prompt_ids_list, target_ids_list)]
        if all(plen == len(m) for plen, m in zip(prompt_lens, merged)):
            return torch.zeros((), device=self.device, dtype=torch.float32)
        max_len = max(len(ids) for ids in merged)
        labels = torch.full((B, max_len), pad, dtype=torch.long, device=self.device)
        full = torch.full((B, max_len), pad, dtype=torch.long, device=self.device)
        for i, ids in enumerate(merged):
            row = torch.tensor(ids, dtype=torch.long, device=self.device)
            labels[i, :len(ids)] = row
            full[i, :len(ids)] = row
        for i, plen in enumerate(prompt_lens):
            labels[i, :plen] = -100
        labels[labels == pad] = -100
        start_id = self._decoder_start_id()
        assert start_id is not None
        start = torch.full((B, 1), int(start_id), dtype=torch.long, device=self.device)
        decoder_input_ids = torch.cat([start, full[:, :-1]], dim=1)
        if self.family == 'encdec_t5':
            outputs = self.llm(
                decoder_input_ids=decoder_input_ids,
                encoder_outputs=(enc_hidden,),
                attention_mask=enc_mask,
                labels=labels,
                return_dict=True,
            )
        else:
            outputs = self.llm(
                decoder_input_ids=decoder_input_ids,
                encoder_outputs=BaseModelOutput(last_hidden_state=enc_hidden),
                attention_mask=enc_mask,
                labels=labels,
                return_dict=True,
            )
        assert outputs.loss is not None
        loss_ar = outputs.loss
        assert loss_ar.ndim == 0
        assert torch.isfinite(loss_ar), f'non-finite L_ar: {loss_ar}'
        return loss_ar

    @torch.no_grad()
    def _free_run_ar_ce(
        self,
        batch: Dict[str, Any],
        z_src: torch.Tensor,
        ei_kv: torch.Tensor,
        return_trace: bool = False,
        data_key: str = 'eeg',
        kv_mask: Optional[torch.Tensor] = None,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor, torch.Tensor]]:
        """
        Greedy free-run CE vs GT. Decoder inputs after the (optional) instruction are argmax
        tokens, never GT. Labels remain the GT target ids.
        """
        native = self._native_gt_path(data_key)
        enc_hidden, enc_mask = self.build_decoder_memory(
            z_src, ei_kv, token_mask=kv_mask, prepend_ei=not native,
        )
        B = int(z_src.shape[0])
        pad = int(self.tokenizer.pad_token_id)
        start_id = self._decoder_start_id()
        assert start_id is not None
        prompt_ids_list, _prompt_lens, target_ids_list = self._tokenize_prompt_target_ids(
            batch, include_instruction=not native,
        )
        max_tgt = max((len(t) for t in target_ids_list), default=0)
        if max_tgt <= 0:
            zero = torch.zeros((), device=self.device, dtype=torch.float32)
            if return_trace:
                z1 = torch.full((B,), pad, dtype=torch.long, device=self.device)
                return zero, z1, z1
            return zero
        labels = torch.full((B, max_tgt), -100, dtype=torch.long, device=self.device)
        for i, t_ids in enumerate(target_ids_list):
            labels[i, :len(t_ids)] = torch.tensor(t_ids, dtype=torch.long, device=self.device)
        prefixes: List[List[int]] = [[int(start_id)] + list(p) for p in prompt_ids_list]
        max_pref = max(len(p) for p in prefixes)
        dec = torch.full((B, max_pref), pad, dtype=torch.long, device=self.device)
        dec_attn = torch.zeros((B, max_pref), dtype=torch.long, device=self.device)
        for i, p in enumerate(prefixes):
            dec[i, max_pref - len(p):] = torch.tensor(p, dtype=torch.long, device=self.device)
            dec_attn[i, max_pref - len(p):] = 1
        enc_out = BaseModelOutput(last_hidden_state=enc_hidden)
        ce_sum = torch.zeros((), device=self.device, dtype=torch.float32)
        n_tok = torch.zeros((), device=self.device, dtype=torch.float32)
        first_argmax: Optional[torch.Tensor] = None
        first_gold = labels[:, 0]

        for t in range(max_tgt):
            if self.family == 'encdec_t5':
                outputs = self.llm(
                    decoder_input_ids=dec,
                    decoder_attention_mask=dec_attn,
                    encoder_outputs=(enc_hidden,),
                    attention_mask=enc_mask,
                    return_dict=True,
                    use_cache=False,
                )
            else:
                outputs = self.llm(
                    decoder_input_ids=dec,
                    decoder_attention_mask=dec_attn,
                    encoder_outputs=enc_out,
                    attention_mask=enc_mask,
                    return_dict=True,
                    use_cache=False,
                )
            logits = outputs.logits[:, -1, :]
            gold = labels[:, t]
            valid = full_target_rollout_mask(labels)[:, t]
            if bool(valid.any()):
                logp = F.log_softmax(logits.float(), dim=-1)
                ce_sum = ce_sum + F.nll_loss(logp[valid], gold[valid], reduction='sum')
                n_tok = n_tok + valid.to(dtype=torch.float32).sum()
            next_tok = logits.argmax(dim=-1)
            if first_argmax is None:
                first_argmax = next_tok.detach()
            # Generated EOS is retained in the rollout history but never removes
            # later gold tokens from this display-only diagnostic denominator.
            next_tok = torch.where(valid, next_tok, torch.full_like(next_tok, pad))
            dec = torch.cat([dec, next_tok.unsqueeze(1)], dim=1)
            dec_attn = torch.cat([dec_attn, valid.to(dtype=torch.long).unsqueeze(1)], dim=1)

        assert first_argmax is not None
        assert int(n_tok.item()) == int(full_target_rollout_mask(labels).sum().item())
        if float(n_tok.detach().cpu()) <= 0:
            loss_ar = torch.zeros((), device=self.device, dtype=torch.float32)
        else:
            loss_ar = ce_sum / n_tok.clamp_min(1.0)
        assert loss_ar.ndim == 0
        assert torch.isfinite(loss_ar), f'non-finite free-run L_ar: {loss_ar}'
        if return_trace:
            return loss_ar, first_argmax, first_gold
        return loss_ar

    def forward_e2e(
        self,
        batch: Dict[str, Any],
        data_key: str,
        mix_seed: Optional[int] = None,
        teacher_force: bool = True,
    ) -> Dict[str, torch.Tensor]:
        """Joint weighted objective; checkpoint-selection calls use teacher-forced AR."""
        z_src, ei_kv, h_tgt, text_mask, ei_text, kv_mask = self.encode_kv(
            batch, data_key, mix_seed=mix_seed,
        )
        loss_clip = clip_info_nce(ei_kv, ei_text.detach())
        loss_commit = token_commitment_mse(z_src, h_tgt.detach(), text_mask)
        if teacher_force:
            loss_ar = self._teacher_forced_ar(
                batch, z_src, ei_kv, data_key=data_key, kv_mask=kv_mask,
            )
        else:
            fr = self._free_run_ar_ce(
                batch, z_src, ei_kv, return_trace=False,
                data_key=data_key, kv_mask=kv_mask,
            )
            assert isinstance(fr, torch.Tensor)
            loss_ar = fr
        weighted_clip = self.w_clip * loss_clip
        weighted_ar = self.w_ar * loss_ar
        weighted_commit = self.commitment_weight * loss_commit
        total = weighted_clip + weighted_ar + weighted_commit
        _, _, target_ids = self._tokenize_prompt_target_ids(
            batch, include_instruction=not self._native_gt_path(data_key),
        )
        target_token_count = sum(len(ids) for ids in target_ids)
        commitment_element_count = int(text_mask.sum().item()) * int(z_src.shape[-1])
        with torch.no_grad():
            cos = F.cosine_similarity(
                F.normalize(ei_kv, dim=-1), F.normalize(ei_text, dim=-1), dim=-1,
            ).mean()
        return {
            'loss': total,
            'loss_clip': loss_clip.detach(),
            'loss_commit': loss_commit.detach(),
            'loss_ar': loss_ar.detach(),
            'weighted_clip': weighted_clip.detach(),
            'weighted_ar': weighted_ar.detach(),
            'weighted_commit': weighted_commit.detach(),
            'batch_size': torch.tensor(z_src.shape[0], device=z_src.device),
            'target_token_count': torch.tensor(target_token_count, device=z_src.device),
            'commitment_element_count': torch.tensor(commitment_element_count, device=z_src.device),
            'cos': cos,
            'z_src': z_src,
            'ei_kv': ei_kv,
            'ei_text': ei_text.detach(),
            'text_mask': text_mask,
            'kv_mask': kv_mask,
        }

    def _tokenize_prompt_target_ids(
        self, batch: Dict[str, Any], include_instruction: bool = True,
    ) -> Tuple[List[List[int]], List[int], List[List[int]]]:
        """Build masked prompt tokens and lexical target + exactly one EOS."""
        pad = self.tokenizer.pad_token_id
        eos = self.tokenizer.eos_token_id
        assert pad is not None and eos is not None
        assert self.max_target_tokens >= 1
        texts = [str(t) for t in batch.get('target text', batch['input text'])]
        inst = self._instruction_ids() if include_instruction else []
        bart_bos = self._bart_bos_id()
        prompt = ([int(bart_bos)] if bart_bos is not None else []) + list(inst)
        prompt_ids_list: List[List[int]] = []
        prompt_lens: List[int] = []
        target_ids_list: List[List[int]] = []
        lexical_limit = max(0, self.max_target_tokens - 1)
        for text in texts:
            lexical_text = self._prepare_lexical_decoder_text(
                text, follows_instruction=bool(inst),
            )
            lexical = list(self.tokenizer(
                lexical_text,
                add_special_tokens=False,
                padding=False,
                truncation=True,
                max_length=lexical_limit,
            )['input_ids']) if lexical_limit > 0 else []
            while lexical and int(lexical[-1]) == int(eos):
                lexical.pop()
            target = [int(token_id) for token_id in lexical] + [int(eos)]
            assert len(target) <= self.max_target_tokens and target[-1] == int(eos)
            prompt_ids_list.append(list(prompt))
            prompt_lens.append(len(prompt))
            target_ids_list.append(target)
        return prompt_ids_list, prompt_lens, target_ids_list

    @staticmethod
    def _first_n_words(text: str, n: int) -> str:
        """Whitespace-split GT; keep at most the first n words. n<=0 → empty."""
        if int(n) <= 0:
            return ''
        parts = str(text).strip().split()
        return ' '.join(parts[: int(n)])

    def _decoder_start_id(self) -> Optional[int]:
        """EncDec decoder_start / pad fallback."""
        pad = self.tokenizer.pad_token_id
        start = self.llm.config.decoder_start_token_id
        if start is None:
            return int(pad) if pad is not None else None
        return int(start)

    def _bart_bos_id(self) -> Optional[int]:
        """BART <s> after </s> when the decoder prefix continues; None for T5."""
        if self.family != 'encdec_bart':
            return None
        bos = getattr(self.tokenizer, 'bos_token_id', None)
        start = self._decoder_start_id()
        if bos is None or start is None:
            return None
        if int(bos) == int(start):
            return None
        return int(bos)

    def _tokenize_decoder_prefix(
        self,
        batch: Dict[str, Any],
        prefill_n: int,
        include_instruction: bool = True,
    ) -> Tuple[List[List[int]], List[str]]:
        """
        EncDec generate prefix: [decoder_start] + optional BART <s> + optional
        instruction + optional first-n GT word tokens.

        BART inserts bos_token_id after decoder_start only when instruction or
        prefill follows, so gt_text BOS stays length-1 for HF forced-BOS.
        """
        gts = [str(t) for t in batch['input text']]
        start = self._decoder_start_id()
        eos = self.tokenizer.eos_token_id
        bart_bos = self._bart_bos_id()
        inst = self._instruction_ids() if include_instruction else []
        n_words = int(prefill_n)
        prefix_ids: List[List[int]] = []
        for gt in gts:
            ids: List[int] = []
            if start is not None:
                ids.append(int(start))
            extra_text = self._first_n_words(gt, n_words)
            if bart_bos is not None and (len(inst) > 0 or bool(extra_text)):
                ids.append(int(bart_bos))
            ids.extend(inst)
            if extra_text:
                lexical_text = self._prepare_lexical_decoder_text(
                    extra_text, follows_instruction=bool(inst),
                )
                extra = list(self.tokenizer(
                    lexical_text, add_special_tokens=False, padding=False,
                    truncation=True, max_length=self.max_target_tokens,
                )['input_ids'])
                if extra and eos is not None and extra[-1] == eos:
                    extra = extra[:-1]
                if extra and start is not None and extra[0] == int(start):
                    extra = extra[1:]
                if extra and bart_bos is not None and extra[0] == int(bart_bos):
                    extra = extra[1:]
                ids.extend(extra)
            prefix_ids.append(ids)
        assert len(prefix_ids) == len(gts)
        return prefix_ids, gts

    def _pad_prefix_ids(
        self, prefix_ids: List[List[int]],
    ) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]:
        """Right-pad prefix token ids for batched generate. (B, L) ids and attn, or None."""
        pad = int(self.tokenizer.pad_token_id)
        B = len(prefix_ids)
        max_len = max((len(p) for p in prefix_ids), default=0)
        if max_len == 0:
            return None, None
        ids = torch.full((B, max_len), pad, dtype=torch.long, device=self.device)
        attn = torch.zeros((B, max_len), dtype=torch.long, device=self.device)
        for i, p in enumerate(prefix_ids):
            if not p:
                continue
            n = len(p)
            ids[i, :n] = torch.tensor(p, dtype=torch.long, device=self.device)
            attn[i, :n] = 1
        assert ids.shape == (B, max_len) and attn.shape == (B, max_len)
        return ids, attn

    def _strip_instruction(self, decoded: str) -> str:
        """Drop the decoder instruction if generate echoed it."""
        inst = str(self.decoder_prompt).strip()
        out = str(decoded).strip()
        if inst and out.lower().startswith(inst.lower()):
            out = out[len(inst):].lstrip(' :,-')
        return out.strip()

    def _compose_pred(self, decoded: str, include_instruction: bool) -> str:
        """Strip only an instruction that was actually supplied to the decoder."""
        out = str(decoded).strip()
        return self._strip_instruction(out) if bool(include_instruction) else out

    def _run_encdec_generate(
        self,
        enc_hidden: torch.Tensor,
        enc_mask: torch.Tensor,
        prefix_ids: List[List[int]],
        pad: int,
        eos_id: int,
        max_new: int,
    ) -> Tuple[List[str], List[str]]:
        """Decode full outputs and token-exact continuations for equal-length prefixes."""
        G = len(prefix_ids)
        assert G >= 1
        plen = len(prefix_ids[0])
        assert plen >= 1
        assert all(len(p) == plen for p in prefix_ids)
        dec_ids, dec_attn = self._pad_prefix_ids(prefix_ids)
        assert dec_ids is not None and dec_attn is not None
        assert dec_ids.shape == (G, plen) and int(dec_attn.min().item()) == 1
        enc_out = BaseModelOutput(last_hidden_state=enc_hidden)
        out_ids = self.llm.generate(
            encoder_outputs=enc_out,
            attention_mask=enc_mask,
            decoder_input_ids=dec_ids,
            decoder_attention_mask=dec_attn,
            max_new_tokens=max_new,
            do_sample=False,
            num_beams=1,
            pad_token_id=pad,
            eos_token_id=eos_id,
            use_cache=True,
        )
        assert out_ids.ndim == 2 and int(out_ids.shape[0]) == G
        assert int(out_ids.shape[1]) >= plen
        assert torch.equal(out_ids[:, :plen], dec_ids), (
            'generate output must begin with the supplied decoder_input_ids',
            tuple(out_ids.shape), tuple(dec_ids.shape),
        )
        decoded = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        continuation = self.tokenizer.batch_decode(
            out_ids[:, plen:], skip_special_tokens=True,
        )
        assert len(decoded) == len(continuation) == G
        return list(decoded), [str(text).strip() for text in continuation]

    @torch.no_grad()
    def generate_batch(
        self,
        batch: Dict[str, Any],
        data_key: str,
        prefill_n: int = 0,
        seed: int = 2026,
    ) -> Tuple[List[str], List[str], List[str]]:
        """
        Free-run greedy generate returning GT, full decode, and token-sliced continuation.

        KV: data_key source as EncDec encoder_outputs (decoder cross-attn K/V).
        Decoder prompt: BOS + optional BART <s> + optional instruction (not used for
        gt_text) + first `prefill_n` GT words. Rows are grouped by prefix length so
        trailing pads never enter decoder_input_ids.
        """
        self.eval()
        batch = move_to_device(batch, self.device)
        native = self._native_gt_path(data_key)
        include_instruction = not native
        z_src, ei_kv, _h_tgt, _text_mask, _ei_text, kv_mask = self.encode_kv(
            batch, data_key, mix_seed=int(seed),
        )
        enc_hidden, enc_mask = self.build_decoder_memory(
            z_src, ei_kv, token_mask=kv_mask, prepend_ei=not native,
        )
        prefix_ids, gts = self._tokenize_decoder_prefix(
            batch, prefill_n=int(prefill_n), include_instruction=include_instruction,
        )
        B = int(z_src.shape[0])
        assert B == len(gts) == len(prefix_ids), (B, len(gts), len(prefix_ids))
        pad = int(self.tokenizer.pad_token_id)
        eos = self.tokenizer.eos_token_id
        eos_id = int(eos) if eos is not None else pad
        max_new = int(self.max_target_tokens)
        groups: Dict[int, List[int]] = {}
        for i, p in enumerate(prefix_ids):
            n = len(p)
            assert n >= 1, 'EncDec generate requires a BOS prefix'
            groups.setdefault(n, []).append(i)
        decoded: List[Optional[str]] = [None] * B
        continuations: List[Optional[str]] = [None] * B
        for idxs in groups.values():
            idx_t = torch.tensor(idxs, dtype=torch.long, device=self.device)
            texts, continuation_texts = self._run_encdec_generate(
                enc_hidden.index_select(0, idx_t),
                enc_mask.index_select(0, idx_t),
                [prefix_ids[i] for i in idxs],
                pad,
                eos_id,
                max_new,
            )
            assert len(texts) == len(continuation_texts) == len(idxs)
            for i, text, continuation in zip(idxs, texts, continuation_texts):
                decoded[i] = text
                continuations[i] = continuation
        assert all(text is not None for text in decoded)
        assert all(text is not None for text in continuations)
        raw_preds = [
            self._compose_pred(str(text), include_instruction=include_instruction)
            for text in decoded
        ]
        continuation_preds = [str(text).strip() for text in continuations]
        assert len(raw_preds) == len(continuation_preds) == len(gts)
        return gts, raw_preds, continuation_preds

    def unload_llm(self) -> None:
        """Drop the frozen LLM and tokenizer and free CUDA cache."""
        del self.llm
        del self.tokenizer
        self.llm = None  # type: ignore[assignment]
        self.tokenizer = None  # type: ignore[assignment]
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
# =============================================================
# One-stage E2E train (eeg) + eval-only noise sweep + t-SNE
# =============================================================

@torch.no_grad()
def collect_stage_activations(
    model: ProbeSystem,
    loader: DataLoader,
    data_key: str,
    max_batches: int = 16,
    seed: int = 2026,
) -> Dict[str, Any]:
    """Collect ei_kv / ei_text for joint t-SNE under the run's KV source."""
    model.eval()
    ei_kv_l: List[np.ndarray] = []
    ei_text_l: List[np.ndarray] = []
    for bi, batch in enumerate(loader):
        if bi >= max_batches:
            break
        batch = move_to_device(batch, model.device)
        _z, ei_kv, _h, _text_mask, ei_t, _kv_mask = model.encode_kv(
            batch, data_key, mix_seed=int(seed) + int(bi),
        )
        ei_kv_l.append(ei_kv.detach().float().cpu().numpy())
        ei_text_l.append(ei_t.detach().float().cpu().numpy())
    return {
        'ei_eeg': np.concatenate(ei_kv_l, axis=0) if ei_kv_l else np.zeros((0, 1)),
        'ei_text': np.concatenate(ei_text_l, axis=0) if ei_text_l else np.zeros((0, 1)),
        'ei_aligned': np.concatenate(ei_kv_l, axis=0) if ei_kv_l else np.zeros((0, 1)),
    }


def _l2_rows(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """L2-normalize each row of a 2-D array.

    Args:
        x: Array ``(N, D)``.
        eps: Floor added to row norms.

    Returns:
        Row-normalized copy of ``x``.
    """
    assert isinstance(x, np.ndarray) and x.ndim == 2
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, eps)


def _prepare_shared_tsne_arrays(
    snapshots: Dict[str, Dict[str, Any]],
    max_points: int,
    seed: int,
) -> Tuple[np.ndarray, List[str], int]:
    """Stack stage KV arrays and one shared text array using identical sampled rows."""
    order = [key for key in ('init', 'after') if key in snapshots]
    if not order:
        order = list(snapshots.keys())
    assert order and int(max_points) >= 1
    kv_arrays: List[np.ndarray] = []
    text_arrays: List[np.ndarray] = []
    for key in order:
        snap = snapshots[key]
        kv = np.asarray(snap.get('ei_aligned', snap.get('ei_eeg')), dtype=np.float64)
        text = np.asarray(snap['ei_text'], dtype=np.float64)
        assert kv.ndim == text.ndim == 2 and kv.shape[1] == text.shape[1]
        kv_arrays.append(kv)
        text_arrays.append(text)
    hidden_dims = {arr.shape[1] for arr in kv_arrays + text_arrays}
    assert len(hidden_dims) == 1
    n_common = min(arr.shape[0] for arr in kv_arrays + text_arrays)
    assert n_common >= 5, f'insufficient shared t-SNE rows: {n_common}'
    rng = np.random.RandomState(int(seed))
    sample_n = min(n_common, int(max_points))
    indices = (
        np.sort(rng.choice(n_common, size=sample_n, replace=False))
        if n_common > sample_n else np.arange(sample_n)
    )
    text_ref = text_arrays[0][indices]
    for text in text_arrays[1:]:
        assert np.allclose(text[indices], text_ref, atol=1e-5, rtol=1e-5), (
            'frozen text embeddings differ across t-SNE stages'
        )
    pieces = [_l2_rows(kv[indices]) for kv in kv_arrays] + [_l2_rows(text_ref)]
    stacked = np.concatenate(pieces, axis=0)
    assert stacked.shape == ((len(order) + 1) * sample_n, next(iter(hidden_dims)))
    return stacked, order, sample_n


def plot_tsne_stage_grid(
    snapshots: Dict[str, Dict[str, Any]],
    config: Dict[str, Any],
    seed: int = 2026,
    title: str = 't-SNE before / after',
    save_path: Optional[Path] = None,
) -> None:
    """Fit one t-SNE for all stages and shared text coordinates."""
    if not snapshots:
        print('No activation snapshots for t-SNE.')
        return
    stacked, order, n_pts = _prepare_shared_tsne_arrays(
        snapshots,
        max_points=int(config.get('tsne_max_points', 2000)),
        seed=int(seed),
    )
    requested = float(config.get('tsne_perplexity', 30))
    perplexity = min(requested, max(2.0, (stacked.shape[0] - 1) / 3.0), stacked.shape[0] - 1.0)
    embedding = TSNE(
        n_components=2,
        perplexity=perplexity,
        init='pca',
        learning_rate='auto',
        random_state=int(seed),
    ).fit_transform(stacked)
    text_2d = embedding[len(order) * n_pts:(len(order) + 1) * n_pts]
    fig, axes = plt.subplots(len(order), 1, figsize=(6.5, 3.6 * len(order)), squeeze=False)
    for row, key in enumerate(order):
        ax = axes[row][0]
        kv_2d = embedding[row * n_pts:(row + 1) * n_pts]
        ax.scatter(kv_2d[:, 0], kv_2d[:, 1], s=8, alpha=0.65, c='orange', label='KV')
        ax.scatter(text_2d[:, 0], text_2d[:, 1], s=8, alpha=0.65, c='blue', label='LLM text')
        ax.set_title(f'{key} — KV vs LLM (one shared fit)')
        ax.set_xticks([]); ax.set_yticks([])
        ax.legend(fontsize=7, loc='best')
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'  Saved t-SNE: {save_path}')
    plt.show()


def _is_noise_eval_key(data_key: str) -> bool:
    """True when KV is a post-hoc mix on a frozen trained eeg encoder."""
    alpha = DATA_KEY_ALPHA[str(data_key)]
    return alpha is not None and float(alpha) > 0.0


@torch.no_grad()
def report_noise_mix_rms(
    model: ProbeSystem,
    loader: DataLoader,
    data_key: str,
    seed: int,
) -> Tuple[float, float]:
    """Print raw signal/noise RMS for an alpha blend (alpha is not an SNR)."""
    alpha = DATA_KEY_ALPHA[str(data_key)]
    assert alpha is not None and float(alpha) > 0.0
    model.eval()
    batch = move_to_device(next(iter(loader)), model.device)
    signal, _, _, _, _, _ = model.encode_kv(batch, 'eeg', mix_seed=int(seed))
    mixed, _, _, _, _, _ = model.encode_kv(batch, data_key, mix_seed=int(seed))
    noise = (mixed - (1.0 - float(alpha)) * signal) / float(alpha)
    signal_rms = float(signal.float().pow(2).mean().sqrt().cpu())
    noise_rms = float(noise.float().pow(2).mean().sqrt().cpu())
    print(
        f'  {data_key}: alpha={float(alpha):.2f} is an embedding blend coefficient, not '
        f'{int(round(float(alpha) * 100))}% SNR; raw signal/noise RMS='
        f'{signal_rms:.6f}/{noise_rms:.6f}; mixed contributions='
        f'{(1.0 - float(alpha)) * signal_rms:.6f}/{float(alpha) * noise_rms:.6f}'
    )
    return signal_rms, noise_rms


def validate_non_llm_load_keys(
    missing_keys: Sequence[str], unexpected_keys: Sequence[str],
) -> None:
    """A non-LLM checkpoint may omit only frozen ``llm.*`` parameters."""
    invalid_missing = [key for key in missing_keys if not str(key).startswith('llm.')]
    assert not invalid_missing, f'non-LLM checkpoint missing trainable keys: {invalid_missing[:8]}'
    assert not list(unexpected_keys), f'unexpected checkpoint keys: {list(unexpected_keys)[:8]}'


def _stable_metadata_value(value: Any) -> Any:
    """Convert metadata to deterministic JSON-compatible values."""
    if value is None or isinstance(value, (bool, int, float, str)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {
            str(key): _stable_metadata_value(item)
            for key, item in sorted(value.items(), key=lambda pair: str(pair[0]))
        }
    if isinstance(value, (list, tuple)):
        return [_stable_metadata_value(item) for item in value]
    return str(value)


def _json_sha256(value: Any) -> str:
    """Fingerprint a normalized object with canonical UTF-8 JSON."""
    payload = json.dumps(
        _stable_metadata_value(value),
        ensure_ascii=False,
        sort_keys=True,
        separators=(',', ':'),
        allow_nan=False,
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def _resolved_model_path(model: ProbeSystem) -> str:
    """Resolve the active local model snapshot when one is available."""
    configured = str(getattr(model.llm.config, '_name_or_path', model.repo_id))
    configured_path = Path(configured).expanduser()
    if configured_path.exists():
        return str(configured_path.resolve())
    local = _hub_snapshot_dir(str(model.repo_id), model.config.get('model_cache_dir'))
    return str(local.resolve()) if local is not None else configured


def _immutable_hex_revision(value: Any) -> Optional[str]:
    """Normalize only exact 40/64-character hexadecimal revision digests."""
    if not isinstance(value, str):
        return None
    if len(value) not in (40, 64):
        return None
    lowered = value.lower()
    return lowered if all(char in '0123456789abcdef' for char in lowered) else None


def _immutable_model_commit(model: ProbeSystem, resolved_path: str) -> Optional[str]:
    """Return only a digest-shaped immutable model commit; reject branch labels."""
    candidates = [
        getattr(model.llm, '_commit_hash', None),
        getattr(model.llm.config, '_commit_hash', None),
    ]
    for candidate in candidates:
        commit_hash = _immutable_hex_revision(candidate)
        if commit_hash is not None:
            return commit_hash
    return _immutable_hex_revision(Path(resolved_path).name)


def _local_weight_files_sha256(resolved_path: str) -> str:
    """Fingerprint local model weight files in one streaming pass per file."""
    root = Path(resolved_path).resolve()
    assert root.is_dir(), f'cannot fingerprint weights without local model directory: {root}'
    files = sorted({
        path
        for pattern in ('*.safetensors', '*.bin')
        for path in root.rglob(pattern)
        if path.is_file()
    }, key=lambda path: str(path.relative_to(root)).replace('\\', '/'))
    assert files, f'no local safetensors/bin weights found under {root}'
    digest = hashlib.sha256()
    for path in files:
        relative = str(path.relative_to(root)).replace('\\', '/')
        digest.update(relative.encode('utf-8'))
        digest.update(b'\0')
        digest.update(str(path.stat().st_size).encode('ascii'))
        digest.update(b'\0')
        with open(path, 'rb') as handle:
            while True:
                chunk = handle.read(8 * 1024 * 1024)
                if not chunk:
                    break
                digest.update(chunk)
    return digest.hexdigest()


def _tokenizer_commit_and_revision(
    tokenizer: Any, resolved_model_path: str,
) -> Tuple[Optional[str], Optional[str]]:
    """Resolve tokenizer provenance independently from model provenance."""
    init_kwargs = getattr(tokenizer, 'init_kwargs', {})
    init_kwargs = init_kwargs if isinstance(init_kwargs, dict) else {}
    commit_candidates = [
        getattr(tokenizer, '_commit_hash', None),
        init_kwargs.get('_commit_hash'),
    ]
    commit_hash = next((str(value) for value in commit_candidates if value), None)
    tokenizer_path = str(getattr(tokenizer, 'name_or_path', resolved_model_path))
    path_name = Path(tokenizer_path).name.lower()
    if commit_hash is None and len(path_name) in (40, 64) and all(
        char in '0123456789abcdef' for char in path_name
    ):
        commit_hash = path_name
    revision_value = getattr(tokenizer, 'revision', None) or init_kwargs.get('revision')
    revision = str(revision_value) if revision_value is not None else None
    return commit_hash, revision


def _tokenizer_metadata_value(value: Any) -> Any:
    """Serialize token metadata without relying on object repr addresses."""
    if hasattr(value, 'content'):
        return {
            'type': type(value).__name__,
            'content': str(getattr(value, 'content')),
            'single_word': bool(getattr(value, 'single_word', False)),
            'lstrip': bool(getattr(value, 'lstrip', False)),
            'rstrip': bool(getattr(value, 'rstrip', False)),
            'normalized': bool(getattr(value, 'normalized', True)),
            'special': bool(getattr(value, 'special', False)),
        }
    if isinstance(value, dict):
        return {
            str(key): _tokenizer_metadata_value(item)
            for key, item in sorted(value.items(), key=lambda pair: str(pair[0]))
        }
    if isinstance(value, (list, tuple)):
        return [_tokenizer_metadata_value(item) for item in value]
    return _stable_metadata_value(value)


def _tokenizer_behavior_fingerprint(
    tokenizer: Any, family: str,
) -> Tuple[str, str]:
    """Fingerprint tokenizer rules plus added/special/config metadata."""
    representation_kind: Optional[str] = None
    representation_payload: Optional[bytes] = None

    backend = getattr(tokenizer, 'backend_tokenizer', None)
    backend_to_str = getattr(backend, 'to_str', None)
    if callable(backend_to_str):
        backend_text = backend_to_str()
        if isinstance(backend_text, str) and backend_text:
            representation_kind = 'backend_tokenizer_json'
            representation_payload = backend_text.encode('utf-8')

    if representation_payload is None:
        sp_model = getattr(tokenizer, 'sp_model', None)
        serialized_model_proto = getattr(sp_model, 'serialized_model_proto', None)
        if callable(serialized_model_proto):
            sentencepiece_payload = serialized_model_proto()
            if isinstance(sentencepiece_payload, (bytes, bytearray)) and sentencepiece_payload:
                representation_kind = 'sentencepiece_model_proto'
                representation_payload = bytes(sentencepiece_payload)

    if representation_payload is None:
        bpe_ranks = getattr(tokenizer, 'bpe_ranks', None)
        if isinstance(bpe_ranks, dict) and bpe_ranks:
            ordered_merges = sorted(
                (
                    int(rank),
                    _tokenizer_metadata_value(pair),
                )
                for pair, rank in bpe_ranks.items()
            )
            representation_kind = 'bart_bpe_ordered_merges'
            representation_payload = json.dumps(
                ordered_merges,
                ensure_ascii=False,
                sort_keys=True,
                separators=(',', ':'),
                allow_nan=False,
            ).encode('utf-8')

    if str(family) in {'encdec_bart', 'encdec_t5'}:
        assert representation_kind is not None and representation_payload is not None, (
            f'no tokenizer backend/model/merge representation for supported family={family}'
        )
    if representation_kind is None or representation_payload is None:
        representation_kind = 'metadata_only'
        representation_payload = b''

    added_decoder = getattr(tokenizer, 'added_tokens_decoder', {})
    added_decoder = added_decoder if isinstance(added_decoder, dict) else {}
    behavior_metadata = {
        'added_vocab': _tokenizer_metadata_value(
            tokenizer.get_added_vocab() if hasattr(tokenizer, 'get_added_vocab') else {},
        ),
        'added_tokens_decoder': {
            str(key): _tokenizer_metadata_value(value)
            for key, value in sorted(added_decoder.items(), key=lambda pair: str(pair[0]))
        },
        'special_tokens_map': _tokenizer_metadata_value(
            getattr(tokenizer, 'special_tokens_map_extended', {}),
        ),
        'tokenizer_config': _tokenizer_metadata_value({
            'init_kwargs': getattr(tokenizer, 'init_kwargs', {}),
            'model_input_names': getattr(tokenizer, 'model_input_names', None),
            'model_max_length': getattr(tokenizer, 'model_max_length', None),
            'padding_side': getattr(tokenizer, 'padding_side', None),
            'truncation_side': getattr(tokenizer, 'truncation_side', None),
            'clean_up_tokenization_spaces': getattr(
                tokenizer, 'clean_up_tokenization_spaces', None,
            ),
            'split_special_tokens': getattr(tokenizer, 'split_special_tokens', None),
        }),
    }
    metadata_payload = json.dumps(
        behavior_metadata,
        ensure_ascii=False,
        sort_keys=True,
        separators=(',', ':'),
        allow_nan=False,
    ).encode('utf-8')
    digest = hashlib.sha256()
    digest.update(representation_kind.encode('ascii'))
    digest.update(len(representation_payload).to_bytes(8, byteorder='big', signed=False))
    digest.update(representation_payload)
    digest.update(len(metadata_payload).to_bytes(8, byteorder='big', signed=False))
    digest.update(metadata_payload)
    return representation_kind, digest.hexdigest()


def build_checkpoint_metadata(
    model: ProbeSystem,
    config: Dict[str, Any],
    data_provenance: Dict[str, Any],
) -> Dict[str, Any]:
    """Build the complete expected checkpoint identity with cached fingerprints."""
    cache = getattr(model, '_probe_checkpoint_fingerprint_cache', None)
    if cache is None:
        resolved_path = _resolved_model_path(model)
        commit_hash = _immutable_model_commit(model, resolved_path)
        llm_config = model.llm.config.to_dict()
        tokenizer_vocab = model.tokenizer.get_vocab()
        tokenizer_commit, tokenizer_revision = _tokenizer_commit_and_revision(
            model.tokenizer, resolved_path,
        )
        behavior_kind, behavior_sha256 = _tokenizer_behavior_fingerprint(
            model.tokenizer, str(model.family),
        )
        weight_sha256 = None
        if commit_hash is None:
            weight_sha256 = _local_weight_files_sha256(resolved_path)
        cache = {
            'llm': {
                'key': str(model.llm_key),
                'repo_id': str(model.repo_id),
                'family': str(model.family),
                'source': str(model.source),
                'modelscope_id': model.modelscope_id,
                'model_class': type(model.llm).__name__,
                'resolved_path': resolved_path,
                '_commit_hash': commit_hash,
                'config_sha256': _json_sha256(llm_config),
                'weights_sha256': weight_sha256,
                'embed_dim': int(model.embed_dim),
            },
            'tokenizer': {
                'class': type(model.tokenizer).__name__,
                'resolved_name_or_path': str(
                    getattr(model.tokenizer, 'name_or_path', resolved_path)
                ),
                '_commit_hash': tokenizer_commit,
                'revision': tokenizer_revision,
                'vocab_size': int(len(tokenizer_vocab)),
                'vocab_sha256': _json_sha256(tokenizer_vocab),
                'behavior_representation': behavior_kind,
                'behavior_sha256': behavior_sha256,
                'special_ids': {
                    name: getattr(model.tokenizer, name, None)
                    for name in ('pad_token_id', 'bos_token_id', 'eos_token_id', 'unk_token_id')
                },
            },
        }
        cache['tokenizer']['special_ids']['decoder_start_token_id'] = getattr(
            model.llm.config, 'decoder_start_token_id', None,
        )
        cache['tokenizer']['special_ids']['forced_bos_token_id'] = getattr(
            model.llm.config, 'forced_bos_token_id', None,
        )
        cache = _stable_metadata_value(cache)
        setattr(model, '_probe_checkpoint_fingerprint_cache', cache)
    return {
        'seed': int(config.get('seed', 2026)),
        **cache,
        'effective_config': _stable_metadata_value(config),
        'data_provenance': _stable_metadata_value(data_provenance),
    }


def validate_eeg_checkpoint_payload(
    model: ProbeSystem,
    checkpoint: Dict[str, Any],
    expected_seed: int,
    expected_data_provenance: Dict[str, Any],
) -> Dict[str, torch.Tensor]:
    """Validate complete metadata and every non-LLM tensor before state loading."""
    assert isinstance(checkpoint, dict), 'checkpoint payload must be a dict'
    assert checkpoint.get('selection_ar') == 'teacher_forced', (
        f"checkpoint selection_ar must be 'teacher_forced', got {checkpoint.get('selection_ar')!r}"
    )
    assert checkpoint.get('data_key') == 'eeg', (
        f"checkpoint data_key must be 'eeg', got {checkpoint.get('data_key')!r}"
    )
    metadata = checkpoint.get('metadata')
    assert isinstance(metadata, dict), 'checkpoint metadata must exist and be a dict'
    seed = int(expected_seed)
    assert int(model.config.get('seed', seed)) == seed, (
        f'active config seed mismatch: expected {seed}, got {model.config.get("seed")!r}'
    )
    expected_metadata = build_checkpoint_metadata(
        model, model.config, expected_data_provenance,
    )
    assert metadata == expected_metadata, (
        'checkpoint metadata mismatch: '
        f'expected_sha256={_json_sha256(expected_metadata)} '
        f'stored_sha256={_json_sha256(metadata)}'
    )

    state = checkpoint.get('model')
    assert isinstance(state, dict), 'checkpoint model state must be a dict'
    expected_state = {
        key: value for key, value in model.state_dict().items()
        if not str(key).startswith('llm.')
    }
    assert set(state) == set(expected_state), (
        f'checkpoint non-LLM keys mismatch: missing={sorted(set(expected_state) - set(state))[:8]} '
        f'unexpected={sorted(set(state) - set(expected_state))[:8]}'
    )
    validated: Dict[str, torch.Tensor] = {}
    for key, expected_tensor in expected_state.items():
        value = state[key]
        assert isinstance(value, torch.Tensor), f'checkpoint {key} is not a tensor'
        assert value.shape == expected_tensor.shape, (
            f'checkpoint {key} shape mismatch: expected {tuple(expected_tensor.shape)}, '
            f'got {tuple(value.shape)}'
        )
        assert bool(torch.isfinite(value).all()), f'checkpoint {key} contains non-finite values'
        validated[key] = value
    return validated


def load_eeg_checkpoint(
    model: ProbeSystem,
    path: Union[str, Path],
    expected_seed: int,
    expected_data_provenance: Dict[str, Any],
) -> None:
    """Load a complete-metadata-checked, tensor-validated non-LLM EEG checkpoint."""
    ckpt_path = Path(path)
    assert ckpt_path.is_file(), f'eeg checkpoint missing: {ckpt_path}'
    try:
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    except TypeError:
        ckpt = torch.load(ckpt_path, map_location='cpu')
    assert isinstance(ckpt, dict), f'invalid checkpoint payload: {ckpt_path}'
    state = validate_eeg_checkpoint_payload(
        model,
        ckpt,
        expected_seed=int(expected_seed),
        expected_data_provenance=expected_data_provenance,
    )
    missing, unexpected = model.load_state_dict(state, strict=False)
    validate_non_llm_load_keys(missing, unexpected)
    model.to(model.device)
    model.eval()
    model.llm.eval()
    for p in model.llm.parameters():
        p.requires_grad = False
    print(
        f'  Loaded eeg checkpoint {ckpt_path} for seed={int(expected_seed)} '
        f'(allowed llm-missing={len(missing)}, unexpected=0)'
    )


def _weighted_average(values: Sequence[float], weights: Sequence[float]) -> float:
    """Numerically simple weighted mean for epoch reporting."""
    assert len(values) == len(weights)
    if not values:
        return float('nan')
    denominator = float(np.sum(np.asarray(weights, dtype=np.float64)))
    assert denominator > 0.0
    return float(np.dot(np.asarray(values, dtype=np.float64), np.asarray(weights, dtype=np.float64)) / denominator)


def train_e2e(
    model: ProbeSystem,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: Dict[str, Any],
    output_dir: Path,
    data_key: str,
    data_provenance: Optional[Dict[str, Any]] = None,
) -> Dict[str, List[float]]:
    """Train/select with teacher-forced AR; report full-target rollout CE diagnostically."""
    assert str(data_key) == 'eeg', f'train_e2e is eeg-only, got {data_key!r}'
    device = model.device
    epochs = int(config.get('epochs', 100))
    assert epochs >= 1, f'epochs must be >= 1, got {epochs}'
    patience = int(config.get('patience', 10))
    lr = float(config.get('lr', 1e-4))
    grad_clip = float(config.get('grad_clip', 1.0))
    seed = int(config.get('seed', 2026))
    model.set_data_key(data_key)
    params = model.trainable_parameters()
    assert len(params) >= 1, f'no trainable params for data_key={data_key}'
    opt = AdamW(params, lr=lr, weight_decay=float(config.get('weight_decay', 0.05)))
    scheduler = setup_scheduler(opt, config)
    history: Dict[str, List[float]] = {
        'train_loss': [], 'val_loss': [],
        'train_loss_clip': [], 'val_loss_clip': [],
        'train_loss_commit': [], 'val_loss_commit': [],
        'train_loss_ar': [], 'val_loss_ar': [], 'val_rollout_ar': [],
        'train_weighted_clip': [], 'val_weighted_clip': [],
        'train_weighted_ar': [], 'val_weighted_ar': [],
        'train_weighted_commit': [], 'val_weighted_commit': [],
        'train_cos': [], 'val_cos': [],
    }
    checkpoint_metadata = build_checkpoint_metadata(
        model, config, data_provenance or {},
    )
    best_val = float('inf')
    best_state: Optional[Dict[str, torch.Tensor]] = None
    bad = 0
    model.to(device)

    for epoch in range(1, epochs + 1):
        if hasattr(train_loader, 'batch_sampler') and hasattr(train_loader.batch_sampler, 'set_epoch'):
            train_loader.batch_sampler.set_epoch(epoch)
        model.train()
        model.llm.eval()
        tr_clip: List[float] = []
        tr_commit: List[float] = []
        tr_ar: List[float] = []
        tr_cos: List[float] = []
        tr_batch_weights: List[float] = []
        tr_token_weights: List[float] = []
        tr_commit_weights: List[float] = []
        for bi, batch in enumerate(train_loader):
            batch = move_to_device(batch, device)
            opt.zero_grad(set_to_none=True)
            out = model.forward_e2e(batch, data_key, mix_seed=None, teacher_force=True)
            out['loss'].backward()
            torch.nn.utils.clip_grad_norm_(params, grad_clip)
            opt.step()
            tr_clip.append(float(out['loss_clip'].detach().cpu()))
            tr_commit.append(float(out['loss_commit'].detach().cpu()))
            tr_ar.append(float(out['loss_ar'].detach().cpu()))
            tr_cos.append(float(out['cos'].detach().cpu()))
            tr_batch_weights.append(float(out['batch_size'].item()))
            tr_token_weights.append(float(out['target_token_count'].item()))
            tr_commit_weights.append(float(out['commitment_element_count'].item()))

        model.eval()
        va_clip: List[float] = []
        va_commit: List[float] = []
        va_ar: List[float] = []
        va_rollout_ar: List[float] = []
        va_cos: List[float] = []
        va_batch_weights: List[float] = []
        va_token_weights: List[float] = []
        va_commit_weights: List[float] = []
        with torch.no_grad():
            for bi, batch in enumerate(val_loader):
                batch = move_to_device(batch, device)
                out = model.forward_e2e(
                    batch, data_key, mix_seed=seed + int(bi), teacher_force=True,
                )
                rollout_ar = model._free_run_ar_ce(
                    batch,
                    out['z_src'],
                    out['ei_kv'],
                    return_trace=False,
                    data_key=data_key,
                    kv_mask=out['kv_mask'],
                )
                assert isinstance(rollout_ar, torch.Tensor)
                va_clip.append(float(out['loss_clip'].detach().cpu()))
                va_commit.append(float(out['loss_commit'].detach().cpu()))
                va_ar.append(float(out['loss_ar'].detach().cpu()))
                va_rollout_ar.append(float(rollout_ar.detach().cpu()))
                va_cos.append(float(out['cos'].detach().cpu()))
                va_batch_weights.append(float(out['batch_size'].item()))
                va_token_weights.append(float(out['target_token_count'].item()))
                va_commit_weights.append(float(out['commitment_element_count'].item()))

        scheduler.step()
        tr_cl = _weighted_average(tr_clip, tr_batch_weights)
        va_cl = _weighted_average(va_clip, va_batch_weights)
        tr_cm = _weighted_average(tr_commit, tr_commit_weights)
        va_cm = _weighted_average(va_commit, va_commit_weights)
        tr_a = _weighted_average(tr_ar, tr_token_weights)
        va_a = _weighted_average(va_ar, va_token_weights)
        va_roll = _weighted_average(va_rollout_ar, va_token_weights)
        tr_c = _weighted_average(tr_cos, tr_batch_weights)
        va_c = _weighted_average(va_cos, va_batch_weights)
        tr_w_cl = model.w_clip * tr_cl
        va_w_cl = model.w_clip * va_cl
        tr_w_a = model.w_ar * tr_a
        va_w_a = model.w_ar * va_a
        tr_w_cm = model.commitment_weight * tr_cm
        va_w_cm = model.commitment_weight * va_cm
        tr_m = tr_w_cl + tr_w_a + tr_w_cm
        va_m = va_w_cl + va_w_a + va_w_cm
        history['train_loss'].append(tr_m)
        history['val_loss'].append(va_m)
        history['train_loss_clip'].append(tr_cl)
        history['val_loss_clip'].append(va_cl)
        history['train_loss_commit'].append(tr_cm)
        history['val_loss_commit'].append(va_cm)
        history['train_loss_ar'].append(tr_a)
        history['val_loss_ar'].append(va_a)
        history['val_rollout_ar'].append(va_roll)
        history['train_weighted_clip'].append(tr_w_cl)
        history['val_weighted_clip'].append(va_w_cl)
        history['train_weighted_ar'].append(tr_w_a)
        history['val_weighted_ar'].append(va_w_a)
        history['train_weighted_commit'].append(tr_w_cm)
        history['val_weighted_commit'].append(va_w_cm)
        history['train_cos'].append(tr_c)
        history['val_cos'].append(va_c)
        lr_now = float(opt.param_groups[0]['lr'])
        print(
            f'  [E2E {data_key}] epoch {epoch:03d}/{epochs} '
            f'train={tr_m:.4f} val={va_m:.4f} '
            f'clip={tr_cl:.4f}/{va_cl:.4f} ar_tf={tr_a:.4f}/{va_a:.4f} '
            f'val_rollout={va_roll:.4f} commit={tr_cm:.4f}/{va_cm:.4f} '
            f'weighted(train/val)=clip {tr_w_cl:.4f}/{va_w_cl:.4f}, '
            f'ar {tr_w_a:.4f}/{va_w_a:.4f}, commit {tr_w_cm:.4f}/{va_w_cm:.4f} '
            f'cos={tr_c:.4f}/{va_c:.4f} lr={lr_now:.2e}'
        )

        if va_m < best_val - 1e-6:
            best_val = va_m
            bad = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
                if not k.startswith('llm.')
            }
            torch.save(
                {
                    'model': best_state,
                    'epoch': epoch,
                    'val_loss': best_val,
                    'data_key': data_key,
                    'selection_ar': 'teacher_forced',
                    'metadata': checkpoint_metadata,
                },
                output_dir / 'best.pt',
            )
        else:
            bad += 1
            if bad >= patience:
                print(f'  Early stop at epoch {epoch} (patience={patience})')
                break

    if best_state is not None:
        missing, unexpected = model.load_state_dict(best_state, strict=False)
        validate_non_llm_load_keys(missing, unexpected)
        print(f'  Restored best E2E checkpoint (allowed llm-missing={len(missing)}, unexpected=0)')

    fig, ax = plt.subplots(1, 4, figsize=(18, 3.5))
    ax[0].plot(history['train_loss'], label='train')
    ax[0].plot(history['val_loss'], label='val')
    ax[0].set_title('Total (0.5 CLIP + 0.5 AR + 0.7 commit)')
    ax[0].legend(); ax[0].set_xlabel('epoch')
    ax[1].plot(history['train_loss_clip'], label='train CLIP')
    ax[1].plot(history['val_loss_clip'], label='val CLIP')
    ax[1].plot(history['train_loss_ar'], label='train AR (TF)')
    ax[1].plot(history['val_loss_ar'], label='val AR (TF)')
    ax[1].plot(history['val_rollout_ar'], '--', label='val rollout AR (diagnostic)')
    ax[1].set_title('CLIP / AR (TF selects checkpoint)'); ax[1].legend(); ax[1].set_xlabel('epoch')
    ax[2].plot(history['train_loss_commit'], label='train')
    ax[2].plot(history['val_loss_commit'], label='val')
    ax[2].set_title('Commitment MSE'); ax[2].legend(); ax[2].set_xlabel('epoch')
    ax[3].plot(history['train_cos'], label='train')
    ax[3].plot(history['val_cos'], label='val')
    ax[3].set_title('Cosine(ei_kv, ei_text)'); ax[3].legend(); ax[3].set_xlabel('epoch')
    fig.suptitle(f'E2E curves — {data_key}', fontsize=12)
    fig.tight_layout()
    fig.savefig(output_dir / 'e2e_curves.png', dpi=120, bbox_inches='tight')
    plt.show()
    return history


@torch.no_grad()
def show_qualitative_nn(
    model: ProbeSystem,
    loader: DataLoader,
    data_key: str,
    n: int = 5,
    seed: int = 2026,
) -> None:
    """Nearest-neighbor retrieval: KV → closest in-batch texts via cosine."""
    model.eval()
    shown = 0
    for bi, batch in enumerate(loader):
        batch = move_to_device(batch, model.device)
        _z, ei_kv, _h, _text_mask, ei_t, _kv_mask = model.encode_kv(
            batch, data_key, mix_seed=seed + int(bi),
        )
        texts = [str(t) for t in batch['input text']]
        ei_n = F.normalize(ei_kv.float(), dim=-1)
        et_n = F.normalize(ei_t.float(), dim=-1)
        sim = ei_n @ et_n.T
        for i in range(ei_kv.shape[0]):
            if shown >= n:
                return
            j = int(sim[i].argmax().item())
            print(f'--- example {shown + 1} ---')
            print(f'  GT     : {texts[i]}')
            print(f'  NN text: {texts[j]}  (cos={float(sim[i, j]):.3f})')
            shown += 1


SBERT_MODEL_NAME: str = 'all-mpnet-base-v2'
SBERT_BATCH_SIZE: int = 64


def bleu_per_row(
    preds: Sequence[str],
    gts: Sequence[str],
    n_gram: int = 1,
) -> np.ndarray:
    """Per-row BLEU for qualitative display; empty predictions score zero."""
    assert len(preds) == len(gts)
    assert n_gram in (1, 2, 3, 4)
    scores: List[float] = []
    for pred, gt in zip(preds, gts):
        pred_text = pred if isinstance(pred, str) else ''
        gt_text = gt if isinstance(gt, str) else ''
        score = 0.0 if not pred_text.strip() else float(
            bleu_score([pred_text], [[gt_text]], n_gram=n_gram).item()
        )
        scores.append(score)
    out = np.asarray(scores, dtype=np.float32)
    assert out.shape == (len(preds),)
    return out


def mean_bleu(preds: Sequence[str], gts: Sequence[str]) -> Dict[str, float]:
    """True single-reference corpus BLEU-1..4 (one torchmetrics call per order)."""
    assert len(preds) == len(gts)
    if not preds:
        return {f'bleu{n}': 0.0 for n in range(1, 5)}
    pred_texts = [pred if isinstance(pred, str) else '' for pred in preds]
    references = [[gt if isinstance(gt, str) else ''] for gt in gts]
    metrics: Dict[str, float] = {}
    for n in range(1, 5):
        metrics[f'bleu{n}'] = float(
            bleu_score(pred_texts, references, n_gram=n).item()
        )
    return metrics


def compute_sbert_cosine_per_row(
    preds: Sequence[str],
    gts: Sequence[str],
    model: Any,
    device: str,
    batch_size: int = SBERT_BATCH_SIZE,
    seed: int = 2026,
    gt_emb: Optional[torch.Tensor] = None,
) -> np.ndarray:
    """Per-row SentenceTransformer cosine; empty generated strings score zero."""
    assert len(preds) == len(gts)
    set_seed(int(seed))
    pred_texts = [p if isinstance(p, str) else '' for p in preds]
    gt_texts = [g if isinstance(g, str) else '' for g in gts]
    emb_p = model.encode(
        pred_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        show_progress_bar=False,
        device=device,
    )
    if gt_emb is None:
        emb_t = model.encode(
            gt_texts,
            batch_size=batch_size,
            convert_to_tensor=True,
            show_progress_bar=False,
            device=device,
        )
    else:
        emb_t = gt_emb
    assert emb_p.shape[0] == len(pred_texts)
    assert emb_t.shape[0] == len(gt_texts)
    scores = F.cosine_similarity(emb_p, emb_t, dim=1)
    empty_pred = torch.tensor(
        [not text.strip() for text in pred_texts], dtype=torch.bool, device=scores.device,
    )
    scores = scores.masked_fill(empty_pred, 0.0)
    assert not torch.isnan(scores).any(), 'NaN in SBERT cosine scores'
    out = scores.detach().cpu().numpy().astype(np.float32)
    assert out.shape == (len(preds),)
    return out


def mean_sbert_cosine(
    preds: Sequence[str],
    gts: Sequence[str],
    model: Any,
    device: str,
    gt_emb: Optional[torch.Tensor] = None,
) -> float:
    """Corpus-mean per-row SBERT cosine. Empty pred is encoded as ''."""
    scores = compute_sbert_cosine_per_row(
        preds, gts, model=model, device=device, gt_emb=gt_emb,
    )
    assert scores.shape == (len(preds),)
    return float(np.mean(scores)) if len(preds) else 0.0


def plot_setting_metrics(metrics_df: pd.DataFrame, title: str) -> None:
    """Plot primary continuation metrics across dynamic prefill settings."""
    primary = (
        metrics_df.loc[metrics_df['scope'].astype(str) == 'continuation'].reset_index(drop=True)
        if 'scope' in metrics_df.columns else metrics_df.reset_index(drop=True)
    )
    metric_cols = ['bleu1', 'bleu2', 'bleu3', 'bleu4', 'sbert_cosine']
    settings = [str(s) for s in primary['setting'].tolist()]
    n_set = len(settings)
    x = np.arange(len(metric_cols))
    width = 0.8 / max(n_set, 1)
    fig, ax = plt.subplots(figsize=(10, 4))
    for i, setting in enumerate(settings):
        offset = (i - (n_set - 1) / 2.0) * width
        vals = [float(primary.iloc[i][c]) for c in metric_cols]
        ax.bar(x + offset, vals, width, label=setting)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_cols)
    ax.set_ylabel('score')
    ax.set_ylim(0.0, 1.0)
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    plt.show()


def plot_language_prior_heatmaps(
    metrics_rows: List[Dict[str, Any]],
    llm_key: str,
) -> None:
    """Heatmaps of BLEU-1 and SBERT cosine over data_key × prefill (display only)."""
    if not metrics_rows:
        return
    df = pd.DataFrame(metrics_rows)
    data_order = [k for k in CONFIG.get('data_keys', []) if k in set(df['data_key'])]
    pref_order = [
        label for _pred, _raw, label, _n in get_prefill_settings(CONFIG)
        if label in set(df['setting'])
    ]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, col, title in (
        (axes[0], 'bleu1', 'BLEU-1'),
        (axes[1], 'sbert_cosine', 'SBERT cosine'),
    ):
        pivot = df.pivot_table(index='data_key', columns='setting', values=col, aggfunc='mean')
        pivot = pivot.reindex(index=data_order, columns=pref_order)
        im = ax.imshow(pivot.to_numpy(dtype=np.float64), vmin=0.0, vmax=1.0, aspect='auto')
        ax.set_xticks(range(len(pref_order)))
        ax.set_xticklabels(list(pref_order), rotation=20)
        ax.set_yticks(range(len(data_order)))
        ax.set_yticklabels(list(data_order))
        ax.set_title(f'{title} — {llm_key}')
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.iloc[i, j]
                if pd.notna(val):
                    ax.text(j, i, f'{float(val):.3f}', ha='center', va='center', fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    plt.show()


SEED_SCORE_COLS: Tuple[str, ...] = ('bleu1', 'bleu2', 'bleu3', 'bleu4', 'sbert_cosine')


def _seed_metric_rows(
    metrics_df: pd.DataFrame,
    llm_key: str,
    data_key: str,
    seed: int,
) -> List[Dict[str, Any]]:
    """One primary continuation-score row per prefill setting, tagged with llm/seed."""
    rows: List[Dict[str, Any]] = []
    primary = (
        metrics_df.loc[metrics_df['scope'].astype(str) == 'continuation']
        if 'scope' in metrics_df.columns else metrics_df
    )
    for _, row in primary.iterrows():
        rec: Dict[str, Any] = {
            'llm_key': str(llm_key),
            'data_key': str(data_key),
            'setting': str(row['setting']),
            'seed': int(seed),
        }
        for col in SEED_SCORE_COLS:
            rec[col] = float(row[col])
        rows.append(rec)
    return rows


def plot_seed_mean_std_heatmaps(
    summary: pd.DataFrame,
    title: str,
) -> None:
    """Heatmaps of mean BLEU-1 / SBERT; cell text is mean±std (display only)."""
    if summary.empty:
        return
    llm_keys = list(dict.fromkeys(summary['llm_key'].tolist()))
    data_order = [
        key for key in CONFIG.get('data_keys', []) if key in set(summary['data_key'])
    ]
    pref_order = [
        label for _pred, _raw, label, _n in get_prefill_settings(CONFIG)
        if label in set(summary['setting'])
    ]
    n_llm = max(len(llm_keys), 1)
    fig, axes = plt.subplots(n_llm, 2, figsize=(12, 4.2 * n_llm), squeeze=False)
    for r, llm in enumerate(llm_keys):
        sub = summary[summary['llm_key'] == llm]
        for c, (col, mtitle) in enumerate((('bleu1', 'BLEU-1'), ('sbert_cosine', 'SBERT cosine'))):
            ax = axes[r][c]
            mean_p = sub.pivot_table(
                index='data_key', columns='setting', values=f'{col}_mean', aggfunc='mean',
            )
            std_p = sub.pivot_table(
                index='data_key', columns='setting', values=f'{col}_std', aggfunc='mean',
            )
            mean_p = mean_p.reindex(index=data_order, columns=pref_order)
            std_p = std_p.reindex(index=data_order, columns=pref_order)
            im = ax.imshow(mean_p.to_numpy(dtype=np.float64), vmin=0.0, vmax=1.0, aspect='auto')
            ax.set_xticks(range(len(pref_order)))
            ax.set_xticklabels(list(pref_order), rotation=20)
            ax.set_yticks(range(len(data_order)))
            ax.set_yticklabels(list(data_order))
            ax.set_title(f'{mtitle} mean±std — {llm}')
            for i in range(mean_p.shape[0]):
                for j in range(mean_p.shape[1]):
                    mv = mean_p.iloc[i, j]
                    sv = std_p.iloc[i, j]
                    if pd.notna(mv):
                        s = 0.0 if pd.isna(sv) else float(sv)
                        ax.text(
                            j, i, f'{float(mv):.3f}±{s:.3f}',
                            ha='center', va='center', fontsize=7,
                        )
            fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


def summarize_seed_metrics(
    rows: List[Dict[str, Any]],
    title: str = 'Mean ± std over seeds',
    show_plot: bool = True,
) -> pd.DataFrame:
    """Mean and sample std (ddof=1) of corpus scores over seeds. Display only."""
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    for col in ('llm_key', 'data_key', 'setting', 'seed'):
        assert col in df.columns, col
    for col in SEED_SCORE_COLS:
        assert col in df.columns, col
    keys = ['llm_key', 'data_key', 'setting']
    out_rows: List[Dict[str, Any]] = []
    grouped = df.groupby(keys, sort=False)
    for gkey, g in grouped:
        unique_seed_count = int(g['seed'].nunique())
        assert unique_seed_count == len(g), (
            f'duplicate seed rows for {gkey}: {g["seed"].tolist()}'
        )
        rec: Dict[str, Any] = {
            'llm_key': gkey[0],
            'data_key': gkey[1],
            'setting': gkey[2],
            'n_seeds': unique_seed_count,
        }
        for col in SEED_SCORE_COLS:
            vals = g[col].to_numpy(dtype=np.float64)
            assert vals.shape[0] >= 1
            mean = float(np.mean(vals))
            std = 0.0 if vals.shape[0] <= 1 else float(np.std(vals, ddof=1))
            rec[f'{col}_mean'] = mean
            rec[f'{col}_std'] = std
            rec[col] = f'{mean:.4f} ± {std:.4f}'
        out_rows.append(rec)
    summary = pd.DataFrame(out_rows)
    print(f'\n===== {title} (per-group n shown; sample std ddof=1, n=1 -> 0) =====')
    show_cols = keys + ['n_seeds'] + list(SEED_SCORE_COLS)
    print(summary[show_cols].to_string(index=False))
    if show_plot:
        plot_seed_mean_std_heatmaps(summary, title=title)
    return summary


def target_continuation(text: str, prefill_n: int) -> str:
    """Ground-truth words after an eval-only whitespace prefill."""
    words = str(text).strip().split()
    return ' '.join(words[max(0, int(prefill_n)):]).strip()


def print_qualitative_setting_examples(
    df: pd.DataFrame,
    row_scores: Dict[str, Dict[str, np.ndarray]],
    settings: Sequence[Tuple[str, str, str, int]],
    n: int = 3,
) -> None:
    """Print raw and continuation predictions with primary per-row scores."""
    n_show = min(int(n), len(df))
    print('\n  Qualitative free-run examples (primary=continuation):')
    for i in range(n_show):
        print(f'  --- example {i + 1} ---')
        print(f'    GT            : {df.loc[i, "gt"]}')
        for pred_col, raw_col, label, _n in settings:
            pred = df.loc[i, pred_col]
            raw = df.loc[i, raw_col]
            b1 = float(row_scores[pred_col]['bleu1'][i])
            cos = float(row_scores[pred_col]['sbert_cosine'][i])
            print(
                f'    {label:<10} raw={raw!s} | continuation={pred!s} '
                f'(BLEU-1={b1:.3f}, cos={cos:.3f})'
            )


def evaluate_settings(
    df: pd.DataFrame,
    device: Union[str, torch.device],
    settings: Optional[Sequence[Tuple[str, str, str, int]]] = None,
    seed: int = 2026,
    sbert_model_name: str = SBERT_MODEL_NAME,
    batch_size: int = SBERT_BATCH_SIZE,
    sbert_model: Optional[Any] = None,
    show_plot: bool = True,
    title: str = 'Probe generate settings',
) -> Tuple[pd.DataFrame, Dict[str, Dict[str, np.ndarray]]]:
    """Score continuation-only primaries and separately labelled whole outputs."""
    resolved = list(settings) if settings is not None else get_prefill_settings(CONFIG)
    required = ['gt'] + [name for pred, raw, _label, _n in resolved for name in (pred, raw)]
    for col in required:
        assert col in df.columns, f'missing column {col}'
    gts = [g if isinstance(g, str) else '' for g in df['gt'].tolist()]
    n_rows = len(gts)
    device_str = device if isinstance(device, str) else str(device)
    set_seed(int(seed))
    model = sbert_model if sbert_model is not None else SentenceTransformer(
        sbert_model_name, device=device_str,
    )
    full_gt_emb = model.encode(
        gts,
        batch_size=batch_size,
        convert_to_tensor=True,
        show_progress_bar=False,
        device=device_str,
    )
    assert full_gt_emb.shape[0] == n_rows

    rows: List[Dict[str, Any]] = []
    row_scores: Dict[str, Dict[str, np.ndarray]] = {}
    for pred_col, raw_col, label, n_words in resolved:
        continuation_preds = [str(value).strip() for value in df[pred_col].tolist()]
        continuation_gts = [target_continuation(gt, n_words) for gt in gts]
        continuation_gt_emb = model.encode(
            continuation_gts,
            batch_size=batch_size,
            convert_to_tensor=True,
            show_progress_bar=False,
            device=device_str,
        )
        primary_bleu = mean_bleu(continuation_preds, continuation_gts)
        primary_cos = compute_sbert_cosine_per_row(
            continuation_preds,
            continuation_gts,
            model=model,
            device=device_str,
            batch_size=batch_size,
            seed=seed,
            gt_emb=continuation_gt_emb,
        )
        primary_bleu1_rows = bleu_per_row(continuation_preds, continuation_gts, n_gram=1)
        row_scores[pred_col] = {
            'bleu1': primary_bleu1_rows,
            'sbert_cosine': primary_cos,
        }
        rows.append({
            'scope': 'continuation',
            'setting': label,
            **primary_bleu,
            'sbert_cosine': float(primary_cos.mean()) if n_rows else 0.0,
        })

        raw_preds = [str(value).strip() for value in df[raw_col].tolist()]
        whole_bleu = mean_bleu(raw_preds, gts)
        whole_cos = compute_sbert_cosine_per_row(
            raw_preds,
            gts,
            model=model,
            device=device_str,
            batch_size=batch_size,
            seed=seed,
            gt_emb=full_gt_emb,
        )
        rows.append({
            'scope': 'whole_output_diagnostic',
            'setting': label,
            **whole_bleu,
            'sbert_cosine': float(whole_cos.mean()) if n_rows else 0.0,
        })

    metrics_df = pd.DataFrame(rows)
    print('\n  Primary continuation-only corpus metrics:')
    print(
        metrics_df.loc[metrics_df['scope'] == 'continuation'].to_string(
            index=False, float_format=lambda value: f'{value:.4f}',
        )
    )
    print('\n  Whole-output diagnostics (includes any decoded prefill):')
    print(
        metrics_df.loc[metrics_df['scope'] == 'whole_output_diagnostic'].to_string(
            index=False, float_format=lambda value: f'{value:.4f}',
        )
    )
    if show_plot:
        plot_setting_metrics(metrics_df, title=title)
    return metrics_df, row_scores


@torch.no_grad()
def run_prefill_generate(
    model: ProbeSystem,
    test_loader: DataLoader,
    config: Dict[str, Any],
    output_dir: Path,
    data_key: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generate dynamic prefills and save raw plus continuation-only predictions."""
    model.eval()
    seed = int(config.get('seed', 2026))
    max_batches = config.get('gen_max_batches', None)
    settings = get_prefill_settings(config)
    print(f'  Free-run generate: data_key={data_key} prefills={[s[3] for s in settings]} ...')

    gts_all: List[str] = []
    metadata: Dict[str, List[Any]] = {
        'row_index': [], 'text uid': [], 'subject': [], 'task': [], 'dataset': [],
    }
    raw_by_col: Dict[str, List[str]] = {raw_col: [] for _pred, raw_col, _label, _n in settings}
    continuation_by_col: Dict[str, List[str]] = {
        pred_col: [] for pred_col, _raw, _label, _n in settings
    }

    for bi, batch in enumerate(test_loader):
        if max_batches is not None and int(max_batches) > 0 and bi >= int(max_batches):
            break
        batch_size = len(batch['input text'])
        row_values = batch['row_index'].tolist() if torch.is_tensor(batch['row_index']) else list(batch['row_index'])
        uid_values = batch['text uid'].tolist() if torch.is_tensor(batch['text uid']) else list(batch['text uid'])
        assert len(row_values) == len(uid_values) == batch_size
        metadata['row_index'].extend(int(value) for value in row_values)
        metadata['text uid'].extend(int(value) for value in uid_values)
        metadata['subject'].extend(str(value) for value in batch['subject_raw'])
        metadata['task'].extend(str(value) for value in batch['task_raw'])
        metadata['dataset'].extend(str(value) for value in batch['dataset_raw'])

        gts_b: Optional[List[str]] = None
        for pred_col, raw_col, _label, n_words in settings:
            gts_i, raw_i, continuation_i = model.generate_batch(
                batch,
                data_key=data_key,
                prefill_n=int(n_words),
                seed=seed + int(bi),
            )
            if gts_b is None:
                gts_b = gts_i
            else:
                assert gts_i == gts_b
            assert len(raw_i) == len(continuation_i) == len(gts_i)
            raw_by_col[raw_col].extend(str(value).strip() for value in raw_i)
            continuation_by_col[pred_col].extend(
                str(value).strip() for value in continuation_i
            )
        assert gts_b is not None
        gts_all.extend(gts_b)

    n_rows = len(gts_all)
    assert all(len(values) == n_rows for values in metadata.values())
    data: Dict[str, List[Any]] = {**metadata, 'gt': gts_all}
    for pred_col, raw_col, _label, _n_words in settings:
        raw_values = raw_by_col[raw_col]
        continuation_values = continuation_by_col[pred_col]
        assert len(raw_values) == len(continuation_values) == n_rows
        data[raw_col] = raw_values
        data[pred_col] = continuation_values
    df = pd.DataFrame(data)
    csv_path = output_dir / 'predictions.csv'
    df.to_csv(csv_path, index=False)
    print(f'  Wrote {csv_path}  ({n_rows} rows)')

    metrics_df, row_scores = evaluate_settings(
        df,
        device=model.device,
        settings=settings,
        seed=seed,
        title=f'{model.llm_key} / {data_key} continuation metrics',
    )
    print_qualitative_setting_examples(df, row_scores, settings=settings, n=3)
    return df, metrics_df


def _make_dummy_probe_batch(device: torch.device, batch_size: int = 2) -> Dict[str, Any]:
    """Synthetic B>=2 batch for per-LLM smoke tests."""
    B = int(batch_size)
    assert B >= 2
    T, C = 1280, 128
    return {
        'eeg': torch.randn(B, T, C, device=device),
        'mask': torch.ones(B, T, dtype=torch.int32, device=device),
        'input text': ['hello world extra words here', 'goodbye moon and stars tonight'][:B]
        + ['hello world extra words here'] * max(0, B - 2),
        'target text': ['hello world extra words here', 'goodbye moon and stars tonight'][:B]
        + ['hello world extra words here'] * max(0, B - 2),
        'text uid': torch.arange(1, B + 1, dtype=torch.long),
        'prompt': (
            [('<NR>', 'ZuCo1', 'ZAB'), ('<NR>', 'ZuCo1', 'ZDM')]
            + [('<TSR>', 'ZuCo2', 'ZGW')] * max(0, B - 2)
        )[:B],
        'dataset_raw': ['ZuCo1'] * B,
        'task_raw': ['task1'] * B,
        'subject_raw': (['ZAB', 'ZDM'] + ['ZGW'] * max(0, B - 2))[:B],
        'row_index': torch.arange(B, dtype=torch.long),
    }


def run_llm_smoke(
    llm_cfg: Dict[str, Any],
    config: Dict[str, Any],
    device: torch.device,
) -> Dict[str, bool]:
    """Load one frozen LLM and smoke E2E + generate on a tiny dummy batch.

    gt_text uses the native EncDec path (encoder pad mask, no ei, decoder_start only).
    BART gt_text greedy BOS and prefill-1 must near-copy the dummy sentences (BLEU-1 >= 0.8).
    noise50/noise100 are eval-only: encoder frozen; mix differs from clean eeg.
    """
    key = str(llm_cfg['key'])
    print(f'\n[SMOKE LLM] {key} ({llm_cfg["repo_id"]}) family={llm_cfg["family"]}')
    results: Dict[str, bool] = {}
    set_seed(int(config.get('seed', 2026)))
    model: Optional[ProbeSystem] = None
    try:
        model = ProbeSystem(
            config=config,
            llm_key=key,
            repo_id=str(llm_cfg['repo_id']),
            family=str(llm_cfg['family']),
            device=device,
            source=str(llm_cfg.get('source', 'hf_mirror')),
            modelscope_id=llm_cfg.get('modelscope_id'),
        )
        model.to(device)
        model.eval()
        D = int(model.embed_dim)
        print(f'  OK loaded D={D}')

        batch = _make_dummy_probe_batch(device, batch_size=2)
        z_src, ei_eeg = model.encode_eeg(batch)
        assert z_src.shape == (2, int(config['out_len']), D), tuple(z_src.shape)
        assert ei_eeg.shape == (2, D), tuple(ei_eeg.shape)
        assert not torch.isnan(z_src).any()
        results['encode_eeg'] = True
        print(f'  OK encode_eeg {tuple(z_src.shape)}')

        seed = int(config.get('seed', 2026))
        z_gt, ei_gt, h_tgt, text_mask, ei_text, kv_mask = model.encode_kv(
            batch, 'gt_text', mix_seed=seed,
        )
        assert torch.equal(z_gt, h_tgt) and torch.equal(ei_gt, ei_text)
        assert torch.equal(text_mask, kv_mask) and kv_mask.shape == z_gt.shape[:2]
        assert int(kv_mask.sum().item()) < int(kv_mask.numel()), 'gt_text pad mask should hide pads'
        enc_h, enc_m = model.build_decoder_memory(
            z_gt, ei_gt, token_mask=kv_mask, prepend_ei=False,
        )
        assert enc_h.shape[:2] == z_gt.shape[:2] and torch.equal(enc_m.cpu(), kv_mask.cpu())
        results['gt_text_kv_is_encode_text'] = True
        print('  OK gt_text KV = frozen encode_text (no ei, pad mask)')

        inst = model._instruction_ids()
        assert len(inst) >= 1
        start = model._decoder_start_id()
        eos = model.tokenizer.eos_token_id
        bart_bos = model._bart_bos_id()
        assert start is not None and eos is not None
        expected_prompt = ([int(bart_bos)] if bart_bos is not None else []) + inst
        p_ids, p_lens, target_ids = model._tokenize_prompt_target_ids(batch)
        assert p_ids[0] == expected_prompt and int(p_lens[0]) == len(expected_prompt)
        assert target_ids[0][-1] == int(eos) and target_ids[0].count(int(eos)) == 1
        expected_lexical_text = model._prepare_lexical_decoder_text(
            batch['target text'][0], follows_instruction=bool(inst),
        )
        expected_lexical = list(model.tokenizer(
            expected_lexical_text,
            add_special_tokens=False,
            padding=False,
            truncation=True,
            max_length=model.max_target_tokens - 1,
        )['input_ids'])
        while expected_lexical and int(expected_lexical[-1]) == int(eos):
            expected_lexical.pop()
        assert target_ids[0] == [int(token) for token in expected_lexical] + [int(eos)]
        decoded_prompt = model.tokenizer.decode(
            inst, skip_special_tokens=True, clean_up_tokenization_spaces=False,
        ).rstrip()
        decoded_prompt_and_lexical = model.tokenizer.decode(
            inst + [int(token) for token in expected_lexical],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        assert decoded_prompt_and_lexical.startswith(decoded_prompt)
        decoded_boundary = decoded_prompt_and_lexical[len(decoded_prompt):len(decoded_prompt) + 1]
        assert decoded_boundary.isspace(), (
            model.family, decoded_prompt[-20:], decoded_prompt_and_lexical[-40:]
        )
        teacher_history = [int(start)] + p_ids[0] + target_ids[0]
        manual_rollout_prefix = [int(start)] + p_ids[0]
        assert teacher_history[:len(manual_rollout_prefix)] == manual_rollout_prefix

        pref0, _ = model._tokenize_decoder_prefix(batch, 0)
        assert pref0[0] == manual_rollout_prefix
        first_word = model._first_n_words(batch['input text'][0], 1)
        instructed_first_word = model._prepare_lexical_decoder_text(
            first_word, follows_instruction=bool(inst),
        )
        instructed_first_word_ids = list(model.tokenizer(
            instructed_first_word,
            add_special_tokens=False,
            padding=False,
            truncation=True,
            max_length=model.max_target_tokens,
        )['input_ids'])
        native_first_word = model._prepare_lexical_decoder_text(
            first_word, follows_instruction=False,
        )
        native_first_word_ids = list(model.tokenizer(
            native_first_word,
            add_special_tokens=False,
            padding=False,
            truncation=True,
            max_length=model.max_target_tokens,
        )['input_ids'])
        pref1, _ = model._tokenize_decoder_prefix(batch, 1)
        assert pref1[0] == (
            [int(start)] + expected_prompt + [int(token) for token in instructed_first_word_ids]
        )

        p_gt, p_gt_lens, target_gt = model._tokenize_prompt_target_ids(
            batch, include_instruction=False,
        )
        expected_gt_prompt = [int(bart_bos)] if bart_bos is not None else []
        assert p_gt[0] == expected_gt_prompt and int(p_gt_lens[0]) == len(expected_gt_prompt)
        assert [int(start)] + p_gt[0] + target_gt[0] == (
            [int(start)] + expected_gt_prompt + target_gt[0]
        )
        pref_gt, _ = model._tokenize_decoder_prefix(batch, 0, include_instruction=False)
        expected_gt_generate = [int(start)]
        assert pref_gt[0] == expected_gt_generate
        pref_gt1, _ = model._tokenize_decoder_prefix(batch, 1, include_instruction=False)
        assert pref_gt1[0] == (
            [int(start)] + expected_gt_prompt + [int(token) for token in native_first_word_ids]
        )
        if str(model.family) == 'encdec_bart':
            assert bart_bos is not None
            assert p_ids[0][0] == int(bart_bos), 'BART BOS must be inside masked prompt'
            assert pref_gt[0] == [int(start)], 'native gt_text keeps HF forced-BOS-compatible start'
            assert expected_lexical_text.startswith(' ') and not expected_lexical_text.startswith('  ')
            assert instructed_first_word.startswith(' ') and not instructed_first_word.startswith('  ')
            assert native_first_word == first_word
        else:
            assert bart_bos is None and p_ids[0] == inst
            assert expected_lexical_text == batch['target text'][0]
            assert instructed_first_word == native_first_word == first_word
        padded, pad_attn = model._pad_prefix_ids([[7, 8], [7, 8, 9, 10]])
        assert padded is not None and pad_attn is not None
        assert int(padded[0, 0].item()) == 7 and int(padded[0, 1].item()) == 8
        assert int(pad_attn[0, 0].item()) == 1 and int(pad_attn[0, 2].item()) == 0
        results['canonical_decoder_order'] = True
        print(
            f'  OK canonical {model.family} decoder order; instruction={len(inst)} tokens; '
            f'generate groups by prefix length'
        )

        for dk in ('gt_text', 'eeg', 'noise50', 'noise100'):
            model.set_data_key(dk)
            out = model.forward_e2e(batch, dk, mix_seed=seed, teacher_force=True)
            loss = out['loss']
            assert loss.ndim == 0 and torch.isfinite(loss), dk
            assert torch.isfinite(out['loss_clip']) and torch.isfinite(out['loss_ar'])
            assert torch.isfinite(out['loss_commit'])
            results[f'forward_e2e_{dk}'] = True
            print(
                f'  OK forward_e2e[{dk}] (TF) loss={float(loss.detach().cpu()):.4f} '
                f'clip={float(out["loss_clip"].detach().cpu()):.4f} '
                f'ar={float(out["loss_ar"].detach().cpu()):.4f}'
            )

        model.set_data_key('noise50')
        assert all(not p.requires_grad for p in model.eeg_encoder.parameters())
        assert all(not p.requires_grad for p in model.in_proj.parameters())
        assert all(not p.requires_grad for p in model.projector.parameters())
        z_clean, _, _, _, _, _ = model.encode_kv(batch, 'eeg', mix_seed=seed)
        z_n50, _, _, _, _, _ = model.encode_kv(batch, 'noise50', mix_seed=seed)
        assert z_clean.shape == z_n50.shape
        assert not torch.allclose(z_clean, z_n50)
        results['noise_eval_frozen'] = True
        print('  OK noise eval: encoder frozen; noise50 mix != eeg')

        smoke_ckpt = Path(config['output_dir']) / '_smoke_eeg_best.pt'
        smoke_ckpt.parent.mkdir(parents=True, exist_ok=True)
        smoke_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
            if not k.startswith('llm.')
        }
        try:
            torch.save(
                {
                    'model': smoke_state,
                    'epoch': 0,
                    'val_loss': 0.0,
                    'data_key': 'eeg',
                    'selection_ar': 'teacher_forced',
                    'metadata': build_checkpoint_metadata(model, config, {'kind': 'smoke'}),
                },
                smoke_ckpt,
            )
            load_eeg_checkpoint(
                model,
                smoke_ckpt,
                expected_seed=seed,
                expected_data_provenance={'kind': 'smoke'},
            )
        finally:
            smoke_ckpt.unlink(missing_ok=True)
        validate_non_llm_load_keys(['llm.shared.weight'], [])
        rejected_invalid = False
        try:
            validate_non_llm_load_keys(['projector.weight'], [])
        except AssertionError:
            rejected_invalid = True
        assert rejected_invalid
        results['strict_checkpoint_keys'] = True
        print('  OK strict non-LLM checkpoint keys')

        model.set_data_key('eeg')
        out_fr = model.forward_e2e(batch, 'eeg', mix_seed=seed, teacher_force=False)
        assert torch.isfinite(out_fr['loss']) and torch.isfinite(out_fr['loss_ar'])
        z_e, ei_e, _, _, _, kv_e = model.encode_kv(batch, 'eeg', mix_seed=seed)
        fr = model._free_run_ar_ce(
            batch, z_e, ei_e, return_trace=True, data_key='eeg', kv_mask=kv_e,
        )
        assert isinstance(fr, tuple)
        loss_fr, first_am, first_gold = fr
        assert torch.isfinite(loss_fr)
        assert first_am.shape == first_gold.shape == (2,)
        results['free_run_ar_ce'] = True
        print(
            f'  OK free-run L_ar={float(loss_fr.detach().cpu()):.4f} '
            f'first_argmax={first_am.tolist()} first_gold={first_gold.tolist()}'
        )

        inst_l = str(model.decoder_prompt).strip().lower()
        gts0, preds0, continuations0 = model.generate_batch(
            batch, data_key='eeg', prefill_n=0, seed=seed,
        )
        assert len(gts0) == len(preds0) == len(continuations0) == 2
        assert all(isinstance(p, str) for p in preds0 + continuations0)
        assert all(not p.lower().startswith(inst_l) for p in preds0), preds0
        legitimate_native = f'{model.decoder_prompt} is legitimate target text'
        assert model._compose_pred(legitimate_native, include_instruction=False) == legitimate_native
        assert model._compose_pred(legitimate_native, include_instruction=True) != legitimate_native
        results['gen_bos'] = True
        print(f'  OK gen_bos {[p[:40] for p in preds0]}')

        gts1, preds1, continuations1 = model.generate_batch(
            batch, data_key='eeg', prefill_n=1, seed=seed,
        )
        assert gts1 == gts0 and len(preds1) == len(continuations1) == 2
        assert all(not p.lower().startswith(inst_l) for p in preds1), preds1
        one0 = ProbeSystem._first_n_words(gts1[0], 1)
        if one0:
            assert preds1[0].lower().startswith(one0.lower()), preds1[0]
        results['gen_prefill1'] = True
        results['token_exact_continuation'] = True
        print(f'  OK gen_prefill1 {[p[:40] for p in preds1]}')

        gts_gt, preds_gt, continuations_gt = model.generate_batch(
            batch, data_key='gt_text', prefill_n=0, seed=seed,
        )
        assert gts_gt == gts0 and len(preds_gt) == len(continuations_gt) == 2
        assert preds_gt == continuations_gt
        bleu_gt = mean_bleu(preds_gt, gts_gt)
        print(f'  OK gt_text gen_bos BLEU-1={bleu_gt["bleu1"]:.3f} {[p[:40] for p in preds_gt]}')
        if str(model.family) == 'encdec_bart':
            assert float(bleu_gt['bleu1']) >= 0.8, (preds_gt, bleu_gt)
        results['gt_text_gen_bos'] = True

        gts_p1, preds_p1, continuations_p1 = model.generate_batch(
            batch, data_key='gt_text', prefill_n=1, seed=seed,
        )
        assert gts_p1 == gts0 and len(preds_p1) == len(continuations_p1) == 2
        bleu_p1 = mean_bleu(preds_p1, gts_p1)
        print(
            f'  OK gt_text gen_prefill1 BLEU-1={bleu_p1["bleu1"]:.3f} '
            f'{[p[:40] for p in preds_p1]}'
        )
        if str(model.family) == 'encdec_bart':
            assert float(bleu_p1['bleu1']) >= 0.8, (preds_p1, bleu_p1)
        results['gt_text_gen_prefill1'] = True

        model.reset_trainable()
        z2, _ = model.encode_eeg(batch)
        assert z2.shape == z_src.shape
        results['reset_trainable'] = True
        print('  OK reset_trainable')

        assert all(results.values()), f'{key} smoke incomplete: {results}'
        print(f'  [SMOKE LLM] {key} PASSED')
        return results
    finally:
        if model is not None:
            model.unload_llm()
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        _print_disk_usage(config.get('model_cache_dir'))


def validate_probe_run_config(config: Dict[str, Any]) -> None:
    """Validate run ordering and repetition invariants before model/data work."""
    epochs = int(config.get('epochs', 0))
    assert epochs >= 1, f'epochs must be >= 1, got {epochs}'
    seeds = [int(seed) for seed in config.get('seeds', [config.get('seed', 2026)])]
    assert seeds, 'at least one seed is required'
    assert len(seeds) == len(set(seeds)), f'seeds must be unique: {seeds}'
    data_keys = [str(key) for key in config.get('data_keys', ['eeg'])]
    noise_keys = [key for key in data_keys if _is_noise_eval_key(key)]
    if noise_keys:
        assert 'eeg' in data_keys, 'noise* evaluation requires eeg in the same data_keys run'
        eeg_index = data_keys.index('eeg')
        assert all(eeg_index < data_keys.index(key) for key in noise_keys), (
            'eeg must appear before every noise* key in data_keys'
        )


def select_requested_llm_configs(
    llm_configs: Sequence[Dict[str, Any]],
    requested_keys: Optional[Sequence[str]],
) -> List[Dict[str, Any]]:
    """Validate requested LLM keys before filtering and preserve request order."""
    configs = list(llm_configs)
    available_keys = [str(config['key']) for config in configs]
    assert len(available_keys) == len(set(available_keys)), (
        f'duplicate configured LLM keys: {available_keys}'
    )
    if requested_keys is None:
        assert configs, 'No LLM configs available'
        return configs
    requested = [str(key) for key in requested_keys]
    assert requested, 'llm_keys_to_run must not be empty'
    assert len(requested) == len(set(requested)), (
        f'duplicate llm_keys_to_run entries: {requested}'
    )
    unknown = [key for key in requested if key not in set(available_keys)]
    assert not unknown, f'unknown llm_keys_to_run entries: {unknown}; available={available_keys}'
    by_key = {str(config['key']): config for config in configs}
    return [by_key[key] for key in requested]


def run_all_llm_smokes(
    llm_configs: List[Dict[str, Any]],
    config: Dict[str, Any],
    device: torch.device,
) -> Dict[str, Dict[str, bool]]:
    """Smoke every requested LLM after strict key validation. Fail-fast."""
    configs = select_requested_llm_configs(
        llm_configs, config.get('llm_keys_to_run'),
    )
    print(f'\n========== Per-LLM smoke tests ({len(configs)} models) ==========')
    all_results: Dict[str, Dict[str, bool]] = {}
    for llm_cfg in configs:
        key = str(llm_cfg['key'])
        try:
            all_results[key] = run_llm_smoke(llm_cfg, config, device)
        except Exception as exc:
            print(f'[SMOKE FAIL] {key}: {exc}')
            raise RuntimeError(f'LLM smoke failed for {key}: {exc}') from exc
    print('\nAll per-LLM smokes passed:', list(all_results.keys()))
    return all_results


def run_one_llm(
    llm_cfg: Dict[str, Any],
    dm: 'PhaseDataModule',
    config: Dict[str, Any],
    device: torch.device,
) -> Tuple[pd.DataFrame, List[Dict[str, Any]]]:
    """Full pipeline: one frozen LLM; train eeg per seed; noise* is eval-only mix; gt_text is generate-only."""
    validate_probe_run_config(config)
    key = llm_cfg['key']
    print(f'\n========== {key} ({llm_cfg["repo_id"]}) ==========')

    bs = int(llm_cfg.get('batch_size', config['batch_size']))
    dm.batch_size = bs
    test_loader = dm.test_dataloader()

    orig_seed = int(config.get('seed', 2026))
    seeds = [int(s) for s in config.get('seeds', [orig_seed])]
    assert len(seeds) >= 1

    set_seed(int(seeds[0]))
    model = ProbeSystem(
        config=config,
        llm_key=key,
        repo_id=llm_cfg['repo_id'],
        family=llm_cfg['family'],
        device=device,
        source=str(llm_cfg.get('source', 'hf_mirror')),
        modelscope_id=llm_cfg.get('modelscope_id'),
    )
    model.to(device)

    data_keys = [str(k) for k in config.get('data_keys', ['eeg'])]
    noise_keys = [k for k in data_keys if _is_noise_eval_key(k)]
    if noise_keys and 'eeg' in data_keys:
        eeg_i = data_keys.index('eeg')
        assert all(eeg_i < data_keys.index(k) for k in noise_keys), (
            'eeg must appear before noise* in data_keys'
        )

    all_rows: List[Dict[str, Any]] = []
    last_df = pd.DataFrame()
    base_out = Path(config['output_dir'])
    gt_text_completed = False

    for seed in seeds:
        config['seed'] = int(seed)
        model.config['seed'] = int(seed)
        set_seed(int(seed))
        seed_out = base_out / f'seed_{int(seed)}'
        train_loader = dm.train_dataloader(seed=int(seed))
        val_loader = dm.val_dataloader(seed=int(seed))
        tsne_loader = dm.val_dataloader(seed=int(seed), sequential=True)
        seed_rows: List[Dict[str, Any]] = []
        print(f'\n===== {key} / seed={seed} =====')

        for data_key in data_keys:
            if str(data_key) == 'gt_text' and gt_text_completed:
                print(f'\n----- {key} / gt_text: deterministic run already completed (n=1) -----')
                continue
            print(f'\n----- {key} / {data_key} / seed={seed} -----')
            out_dir = seed_out / key / str(data_key)
            out_dir.mkdir(parents=True, exist_ok=True)

            if str(data_key) == 'gt_text':
                print('  gt_text: forward-only (frozen encode_text KV; no E2E train)')
                model.set_data_key('gt_text')
                print('  [t-SNE] snapshot: forward')
                snapshots = {
                    'init': collect_stage_activations(
                        model, tsne_loader, data_key='gt_text',
                        max_batches=int(config.get('tsne_max_batches', 16)),
                        seed=int(seed),
                    ),
                }
                plot_tsne_stage_grid(
                    snapshots, config, seed=int(seed),
                    title=f't-SNE forward — {key} / gt_text / seed={seed}',
                    save_path=out_dir / 'tsne_forward.png',
                )
                last_df, metrics_df = run_prefill_generate(
                    model, test_loader, config, out_dir, 'gt_text',
                )
                seed_rows.extend(_seed_metric_rows(metrics_df, str(key), 'gt_text', int(seed)))
                gt_text_completed = True
                continue

            if _is_noise_eval_key(str(data_key)):
                eeg_ckpt = seed_out / key / 'eeg' / 'best.pt'
                print(
                    f'  {data_key}: eval-only noise sweep '
                    f'(load {eeg_ckpt}, no E2E train)'
                )
                load_eeg_checkpoint(
                    model,
                    eeg_ckpt,
                    expected_seed=int(seed),
                    expected_data_provenance=dm.data_provenance,
                )
                model.set_data_key(str(data_key))
                report_noise_mix_rms(model, val_loader, str(data_key), int(seed))
                print('  [t-SNE] snapshot: forward')
                snapshots = {
                    'init': collect_stage_activations(
                        model, tsne_loader, data_key=str(data_key),
                        max_batches=int(config.get('tsne_max_batches', 16)),
                        seed=int(seed),
                    ),
                }
                plot_tsne_stage_grid(
                    snapshots, config, seed=int(seed),
                    title=f't-SNE forward — {key} / {data_key} / seed={seed}',
                    save_path=out_dir / 'tsne_forward.png',
                )
                show_qualitative_nn(
                    model, val_loader, data_key=str(data_key), n=5, seed=int(seed),
                )
                last_df, metrics_df = run_prefill_generate(
                    model, test_loader, config, out_dir, str(data_key),
                )
                seed_rows.extend(
                    _seed_metric_rows(metrics_df, str(key), str(data_key), int(seed)),
                )
                continue

            model.reset_trainable()
            model.set_data_key(str(data_key))

            snapshots = {}
            print('  [t-SNE] snapshot: init')
            snapshots['init'] = collect_stage_activations(
                model, tsne_loader, data_key=str(data_key),
                max_batches=int(config.get('tsne_max_batches', 16)),
                seed=int(seed),
            )

            _ = train_e2e(
                model,
                train_loader,
                val_loader,
                config,
                out_dir,
                str(data_key),
                data_provenance=dm.data_provenance,
            )
            show_qualitative_nn(
                model, val_loader, data_key=str(data_key), n=5, seed=int(seed),
            )

            print('  [t-SNE] snapshot: after')
            snapshots['after'] = collect_stage_activations(
                model, tsne_loader, data_key=str(data_key),
                max_batches=int(config.get('tsne_max_batches', 16)),
                seed=int(seed),
            )
            plot_tsne_stage_grid(
                snapshots, config, seed=int(seed),
                title=f't-SNE before/after — {key} / {data_key} / seed={seed}',
                save_path=out_dir / 'tsne_before_after.png',
            )

            last_df, metrics_df = run_prefill_generate(
                model, test_loader, config, out_dir, str(data_key),
            )
            seed_rows.extend(
                _seed_metric_rows(metrics_df, str(key), str(data_key), int(seed)),
            )

        plot_language_prior_heatmaps(seed_rows, llm_key=f'{key} seed={seed}')
        all_rows.extend(seed_rows)

    if 'gt_text' in data_keys:
        assert gt_text_completed, 'configured gt_text run did not complete'
    _ = summarize_seed_metrics(
        all_rows, title=f'{key} mean ± std over seeds {seeds}',
    )
    config['seed'] = orig_seed
    model.config['seed'] = orig_seed
    model.unload_llm()
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _print_disk_usage(config.get('model_cache_dir'))
    return last_df, all_rows


In [ ]:
# =============================================================
# Cell: Encoder smoke (no LLM download) — fast pre-check
# =============================================================

def _smoke_glim_sampler() -> None:
    """Exhaustive row coverage, UID uniqueness, determinism, and partial batches."""
    class _Dummy(Dataset):
        """Tiny dataset with repeated UIDs for GLIM packing tests."""

        def __init__(self) -> None:
            """Assign nine rows to five text UIDs."""
            self.text_uid = [0, 0, 0, 0, 1, 1, 2, 3, 4]

        def __len__(self) -> int:
            """Number of dummy rows."""
            return len(self.text_uid)

        def __getitem__(self, idx: int) -> int:
            """Return the index itself (sampler only needs length / UIDs).

            Args:
                idx: Row index.

            Returns:
                The same index as an int.
            """
            return int(idx)

    ds = _Dummy()
    ids = _dataset_text_uids(ds)
    for shuffle in (False, True):
        sampler = GLIMSampler(
            ds, ids, phase='train', batch_size=3, seed=2026, shuffle=shuffle,
        )
        batches = list(iter(sampler))
        assert len(batches) == len(sampler) == 4
        flat = [index for batch in batches for index in batch]
        assert sorted(flat) == list(range(len(ds))), flat
        assert len(flat) == len(set(flat))
        assert all(len({ids[index] for index in batch}) == len(batch) for batch in batches)
        assert all(1 <= len(batch) <= 3 for batch in batches)
        assert len(batches[-1]) < 3, batches


def run_smoke_tests() -> Dict[str, bool]:
    """Offline correctness smokes; no corpus, LLM, or SBERT download."""
    results: Dict[str, bool] = {}
    set_seed(2026)

    prompt_rows = [
        ('<NR>', 'ZuCo1', 'ZAB'),
        ('<TSR>', 'ZuCo2', 'ZDM'),
        ('<NR>', 'ZuCo1', 'ZGW'),
    ]
    tasks, datasets, subjects = unpack_prompt_batch(prompt_rows)
    assert tasks == ['<NR>', '<TSR>', '<NR>']
    assert datasets == ['ZuCo1', 'ZuCo2', 'ZuCo1']
    assert subjects == ['ZAB', 'ZDM', 'ZGW']
    assert _ORIG_NEW_INDEX is _pd_indexes_base._probe_orig_new_index
    assert _ORIG_UNPICKLE_BLOCK is _pd_libinternals._probe_orig_unpickle_block
    results['prompt_batch_b3'] = True
    print('  OK prompt_batch_b3 + idempotent pandas patch')

    assert canonicalize_text_uid('01') == canonicalize_text_uid(1) == 1
    assert canonicalize_text_uid('＋００１') == 1
    assert canonicalize_text_uid('-001') == -1
    assert canonicalize_text_uid(np.int64(7)) == 7
    assert canonicalize_text_uid(_INT64_MIN) == _INT64_MIN
    assert canonicalize_text_uid(_INT64_MAX) == _INT64_MAX
    assert canonicalize_text_uid(str(_INT64_MIN)) == _INT64_MIN
    assert canonicalize_text_uid(f'+{_INT64_MAX}') == _INT64_MAX
    assert canonicalize_text_uid(float(_IEEE754_SAFE_INTEGER_MAX)) == _IEEE754_SAFE_INTEGER_MAX
    rejected_uid_values = (
        1.5,
        True,
        None,
        np.nan,
        float(1 << 53),
        object(),
        _INT64_MAX + 1,
        _INT64_MIN - 1,
        str(_INT64_MAX + 1),
        str(_INT64_MIN - 1),
        np.uint64(1 << 63),
    )
    rejected_uids = 0
    for invalid_uid in rejected_uid_values:
        try:
            canonicalize_text_uid(invalid_uid)
        except AssertionError:
            rejected_uids += 1
    assert rejected_uids == len(rejected_uid_values)
    rejected_null_texts = 0
    for null_text in (None, np.nan, pd.NA):
        try:
            _normalize_corpus_text(null_text)
        except AssertionError:
            rejected_null_texts += 1
    assert rejected_null_texts == 3

    split_df = pd.DataFrame({
        'phase': ['train', 'val', 'test'],
        'text uid': ['01', 2.0, np.int64(3)],
        'input text': ['Alpha one', 'Beta two', 'Gamma three'],
    })
    validate_split_integrity(split_df)
    assert split_df['text uid'].tolist() == [1, 2, 3]
    assert all(type(value) is int for value in split_df['text uid'])
    leaked = split_df.copy()
    leaked.loc[2, 'input text'] = '  ALPHA   ONE '
    rejected_overlap = False
    try:
        validate_split_integrity(leaked)
    except AssertionError:
        rejected_overlap = True

    collapsed_conflict = pd.DataFrame({
        'phase': ['train', 'train', 'val', 'test'],
        'text uid': ['01', 1, 2, 3],
        'input text': ['Alpha one', 'different', 'Beta two', 'Gamma three'],
    })
    rejected_uid_mapping = False
    try:
        validate_split_integrity(collapsed_conflict)
    except AssertionError as exc:
        rejected_uid_mapping = 'multiple normalized texts' in str(exc)

    collapsed_leakage = pd.DataFrame({
        'phase': ['train', 'val', 'test'],
        'text uid': ['01', 1, 3],
        'input text': ['shared text', 'shared text', 'Gamma three'],
    })
    rejected_canonical_uid_leakage = False
    try:
        validate_split_integrity(collapsed_leakage)
    except AssertionError as exc:
        rejected_canonical_uid_leakage = 'UID leakage' in str(exc)
    assert rejected_overlap and rejected_uid_mapping and rejected_canonical_uid_leakage
    results['canonical_uid_validation'] = True
    print('  OK canonical UIDs collapse aliases and reject unsafe identities')
    sample_df = pd.DataFrame({
        'eeg': [np.zeros((1280, 128), dtype=np.float32)],
        'mask': [np.ones((1280,), dtype=np.int32)],
    })
    validate_corpus_samples(sample_df)
    dataset_df = sample_df.assign(**{
        'input text': ['dataset text'],
        'text uid': ['01'],
        'dataset': ['ZuCo1'],
        'task': ['task1'],
        'subject': ['ZAB'],
        'row_index': [0],
    })
    canonical_dataset = ProbeDataset(dataset_df)
    assert canonical_dataset.text_uid == [1]
    assert canonical_dataset[0]['text uid'] == 1
    results['split_overlap_detection'] = True
    print('  OK split overlap + sample/Dataset validation')

    B, T, C = 2, 1280, 128
    eeg = torch.randn(B, T, C, device=DEVICE)
    mask = torch.ones(B, T, dtype=torch.int32, device=DEVICE)
    batch = {
        'eeg': eeg,
        'mask': mask,
        'input text': ['hello world extra words here', 'goodbye moon and stars tonight'],
        'target text': ['hello world extra words here', 'goodbye moon and stars tonight'],
        'text uid': torch.tensor([1, 2], dtype=torch.long),
        'prompt': [('<NR>', 'ZuCo1', 'ZAB'), ('<NR>', 'ZuCo1', 'ZDM')],
        'dataset_raw': ['ZuCo1', 'ZuCo1'],
        'task_raw': ['task1', 'task1'],
        'subject_raw': ['ZAB', 'ZDM'],
        'row_index': torch.tensor([0, 1], dtype=torch.long),
    }
    set_seed(2026)
    pe = PromptEmbedder(dim=CONFIG['prompt_dim'], prompt_keys=PROMPT_KEYS)
    set_seed(2026)
    enc = build_eeg_encoder(CONFIG)
    pe.to(DEVICE); enc.to(DEVICE)
    p_ids = pe.encode([
        [batch['prompt'][i][0] for i in range(B)],
        [batch['prompt'][i][1] for i in range(B)],
        [batch['prompt'][i][2] for i in range(B)],
    ], device=DEVICE)
    p = pe(p_ids)
    Zi, memory, _ = enc(eeg, mask, p)
    assert Zi.shape == (B, CONFIG['out_len'], CONFIG['hidden_dim']), tuple(Zi.shape)
    assert memory.shape[1] == CONFIG['in_len'], tuple(memory.shape)
    assert hasattr(enc, 'channel_weights')
    results['eeg_encoder_shapes'] = True
    print('  OK eeg_encoder_shapes', tuple(Zi.shape))

    D = 64
    ei = torch.randn(B, D, device=DEVICE)
    yi = torch.randn(B, D, device=DEVICE)
    loss = clip_info_nce(ei, yi)
    assert loss.ndim == 0
    results['clip_info_nce'] = True
    print('  OK clip_info_nce', float(loss))

    z_eq = torch.randn(B, 96, D, device=DEVICE)
    text_mask = torch.zeros(B, 96, dtype=torch.long, device=DEVICE)
    text_mask[:, :7] = 1
    commit0 = token_commitment_mse(z_eq, z_eq, text_mask)
    assert commit0.ndim == 0 and float(commit0) < 1e-6
    target = torch.randn(B, 96, D, device=DEVICE)
    source = torch.randn(B, 96, D, device=DEVICE)
    commit_r = token_commitment_mse(source, target, text_mask)
    source_changed = source.clone()
    target_changed = target.clone()
    source_changed[:, 7:] = 1e6 * torch.randn_like(source_changed[:, 7:])
    target_changed[:, 7:] = 1e6 * torch.randn_like(target_changed[:, 7:])
    commit_changed = token_commitment_mse(source_changed, target_changed, text_mask)
    assert torch.isfinite(commit_r) and float(commit_r) > 0.0
    assert torch.allclose(commit_r, commit_changed, atol=1e-7, rtol=1e-6)
    results['masked_commitment_invariance'] = True
    print('  OK masked_commitment_invariance', float(commit0), float(commit_r))

    z = torch.randn(B, 96, D, device=DEVICE)
    z50 = mix_embedding_noise(z, 0.5)
    z100 = mix_embedding_noise(z, 1.0)
    assert z50.shape == z.shape and z100.shape == z.shape
    assert not torch.allclose(z50, z)
    results['mix_noise'] = True
    print('  OK mix_noise 50/100')

    assert not _is_noise_eval_key('eeg')
    assert not _is_noise_eval_key('gt_text')
    assert _is_noise_eval_key('noise50')
    assert _is_noise_eval_key('noise100')
    results['noise_eval_keys'] = True
    print('  OK noise_eval_keys')

    validate_probe_run_config(CONFIG)
    invalid_run_configs: List[Dict[str, Any]] = []
    for update in (
        {'epochs': 0},
        {'seeds': [2026, 2026]},
        {'data_keys': ['noise50']},
        {'data_keys': ['noise50', 'eeg']},
    ):
        candidate = dict(CONFIG)
        candidate.update(update)
        invalid_run_configs.append(candidate)
    rejected_run_configs = 0
    for candidate in invalid_run_configs:
        try:
            validate_probe_run_config(candidate)
        except AssertionError:
            rejected_run_configs += 1
    assert rejected_run_configs == len(invalid_run_configs)
    selected = select_requested_llm_configs(LLM_CONFIGS, ['flan_t5_large', 'bart_large'])
    assert [config['key'] for config in selected] == ['flan_t5_large', 'bart_large']
    rejected_llm_filters = 0
    for requested in (['bart_large', 'bart_large'], ['unknown_llm']):
        try:
            select_requested_llm_configs(LLM_CONFIGS, requested)
        except AssertionError:
            rejected_llm_filters += 1
    assert rejected_llm_filters == 2
    results['strict_run_configuration'] = True
    results['strict_llm_filtering'] = True
    print('  OK epochs/seeds/noise ordering + strict LLM filtering')

    w_clip = float(CONFIG['w_clip'])
    w_ar = float(CONFIG['w_ar'])
    w_c = float(CONFIG['commitment_weight'])
    assert abs(w_clip - 0.5) < 1e-12 and abs(w_ar - 0.5) < 1e-12
    assert abs(w_c - 0.7) < 1e-12
    dummy_clip = torch.tensor(1.0)
    dummy_ar = torch.tensor(2.0)
    dummy_commit = torch.tensor(3.0)
    total = w_clip * dummy_clip + w_ar * dummy_ar + w_c * dummy_commit
    assert abs(float(total) - (0.5 * 1.0 + 0.5 * 2.0 + 0.7 * 3.0)) < 1e-6
    results['e2e_loss_weights'] = True
    print('  OK e2e_loss_weights', float(total))

    rollout_labels = torch.tensor([[11, 12, 2, -100], [21, 2, -100, -100]])
    generated_eos_at_first_step = torch.tensor([True, True])
    rollout_mask = full_target_rollout_mask(rollout_labels)
    assert bool(generated_eos_at_first_step.all())
    truncated_after_generated_eos = rollout_mask.clone()
    truncated_after_generated_eos[:, 1:] = False
    assert int(rollout_mask.sum().item()) == 5
    assert int(truncated_after_generated_eos.sum().item()) == 2
    assert rollout_mask[0, 1] and rollout_mask[0, 2] and rollout_mask[1, 1]
    results['early_eos_denominator'] = True
    print('  OK early_eos_denominator keeps all 5 gold tokens')

    set_seed(2026)
    dummy_lin = nn.Linear(4, 4)
    opt = AdamW(dummy_lin.parameters(), lr=float(CONFIG['lr']), weight_decay=float(CONFIG['weight_decay']))
    sch = setup_scheduler(opt, CONFIG)
    assert isinstance(sch, CosineAnnealingWarmRestarts)
    assert int(sch.T_0) == 15
    results['cosine_scheduler'] = True
    print('  OK cosine_scheduler T_0=15')

    _smoke_glim_sampler()
    results['glim_sampler'] = True
    print('  OK exhaustive glim_sampler')

    validate_non_llm_load_keys(['llm.encoder.weight'], [])
    strict_rejected = 0
    for missing, unexpected in ((['projector.bias'], []), (['llm.x'], ['extra'])):
        try:
            validate_non_llm_load_keys(missing, unexpected)
        except AssertionError:
            strict_rejected += 1
    assert strict_rejected == 2

    assert _immutable_hex_revision('a' * 40) == 'a' * 40
    assert _immutable_hex_revision('ABCDEF12' * 8) == ('abcdef12' * 8)
    for mutable_or_malformed in ('master', 'main', '', 'g' * 40, 'a' * 39, 'a' * 65, None):
        assert _immutable_hex_revision(mutable_or_malformed) is None

    class _CommitConfigStub:
        """LLM config whose ``_commit_hash`` is a mutable branch name."""
        _commit_hash = 'master'

    class _CommitLlmStub:
        """LLM stub exposing a mutable commit hash on itself and its config."""
        _commit_hash = 'main'
        config = _CommitConfigStub()

    class _CommitModelStub:
        """Probe-shaped stub for ``_immutable_model_commit``."""
        llm = _CommitLlmStub()

    commit_model_stub = _CommitModelStub()
    assert _immutable_model_commit(commit_model_stub, 'cache/master') is None
    commit_model_stub.llm._commit_hash = 'B' * 40
    assert _immutable_model_commit(commit_model_stub, 'cache/master') == 'b' * 40
    results['immutable_commit_validation'] = True
    print('  OK immutable revisions require 40/64 hex characters')

    class _DummyLlmConfig:
        """Minimal EncDec config used to fingerprint checkpoint metadata."""
        decoder_start_token_id = 0
        forced_bos_token_id = None
        _name_or_path = 'dummy/repo'
        _commit_hash = 'a' * 40

        def to_dict(self) -> Dict[str, Any]:
            """Stable config dict for ``config_sha256``.

            Returns:
                Small JSON-serializable config payload.
            """
            return {'d_model': 3, 'decoder_start_token_id': 0}

    class _DummyCheckpointLlm(nn.Linear):
        """Tiny Linear stand-in for a frozen LLM module."""

        def __init__(self) -> None:
            """3×3 linear plus dummy config."""
            super().__init__(3, 3)
            self.config = _DummyLlmConfig()

    class _DummyBackendTokenizer:
        """tokenizers-backend stub whose ``to_str`` is hashed for behavior."""

        def to_str(self) -> str:
            """Return a fixed JSON serialization of the dummy backend.

            Returns:
                JSON string used as the behavior fingerprint payload.
            """
            return '{"model":{"type":"dummy","merges":["h e"]}}'

    class _DummyCheckpointTokenizer:
        """Tokenizer stub with vocab, added tokens, and backend JSON."""
        pad_token_id = 0
        bos_token_id = 1
        eos_token_id = 2
        unk_token_id = 3
        _commit_hash = 'dummy-tokenizer-commit'
        revision = 'dummy-tokenizer-revision'
        name_or_path = 'dummy/tokenizer'
        init_kwargs: Dict[str, Any] = {
            '_commit_hash': 'dummy-tokenizer-commit',
            'revision': 'dummy-tokenizer-revision',
            'do_lower_case': False,
        }
        backend_tokenizer = _DummyBackendTokenizer()
        added_tokens_decoder: Dict[int, Any] = {}
        special_tokens_map_extended: Dict[str, str] = {
            'pad_token': '<pad>', 'bos_token': '<bos>', 'eos_token': '</s>',
        }
        model_input_names = ['input_ids', 'attention_mask']
        model_max_length = 128
        padding_side = 'right'
        truncation_side = 'right'
        clean_up_tokenization_spaces = False
        split_special_tokens = False

        def get_vocab(self) -> Dict[str, int]:
            """Return the dummy base vocabulary.

            Returns:
                Token-to-id map used for ``vocab_sha256``.
            """
            return {'<pad>': 0, '<bos>': 1, '</s>': 2, '<unk>': 3, 'hello': 4}

        def get_added_vocab(self) -> Dict[str, int]:
            """Return extra tokens attached after the base vocab.

            Returns:
                Added-token map included in the tokenizer fingerprint.
            """
            return {'<extra>': 5}

    class _DummyCheckpointModel(nn.Module):
        """Minimal ProbeSystem stand-in for checkpoint metadata smokes."""

        def __init__(self) -> None:
            """Attach a dummy LLM, tokenizer, and one trainable Linear."""
            super().__init__()
            self.linear = nn.Linear(3, 2)
            self.llm = _DummyCheckpointLlm()
            self.tokenizer = _DummyCheckpointTokenizer()
            self.llm_key = 'dummy_llm'
            self.repo_id = 'dummy/repo'
            self.family = 'encdec_dummy'
            self.source = 'dummy'
            self.modelscope_id = None
            self.embed_dim = 3
            self.config: Dict[str, Any] = {'seed': 2026, 'epochs': 1}

    set_seed(2026)
    checkpoint_model = _DummyCheckpointModel()
    checkpoint_state = {
        key: value.detach().clone()
        for key, value in checkpoint_model.state_dict().items()
        if not key.startswith('llm.')
    }
    checkpoint_provenance: Dict[str, Any] = {
        'resolved_path': 'memory://dummy-corpus', 'sha256': 'corpus-sha256', 'rows': 3,
    }
    checkpoint_metadata = build_checkpoint_metadata(
        checkpoint_model, checkpoint_model.config, checkpoint_provenance,
    )
    valid_checkpoint: Dict[str, Any] = {
        'model': checkpoint_state,
        'data_key': 'eeg',
        'selection_ar': 'teacher_forced',
        'metadata': checkpoint_metadata,
    }
    validated_state = validate_eeg_checkpoint_payload(
        checkpoint_model,
        valid_checkpoint,
        expected_seed=2026,
        expected_data_provenance=checkpoint_provenance,
    )
    assert set(validated_state) == set(checkpoint_state)
    assert checkpoint_metadata['tokenizer']['behavior_representation'] == 'backend_tokenizer_json'
    assert len(checkpoint_metadata['tokenizer']['behavior_sha256']) == 64
    assert checkpoint_metadata['tokenizer']['_commit_hash'] == 'dummy-tokenizer-commit'
    assert checkpoint_metadata['tokenizer']['revision'] == 'dummy-tokenizer-revision'

    class _ChangedBackendTokenizer:
        """Backend whose JSON differs from ``_DummyBackendTokenizer``."""

        def to_str(self) -> str:
            """Return an alternate merge table so the behavior hash changes.

            Returns:
                JSON string that must not match the dummy backend hash.
            """
            return '{"model":{"type":"dummy","merges":["he l"]}}'

    same_vocab_changed_behavior = _DummyCheckpointTokenizer()
    same_vocab_changed_behavior.backend_tokenizer = _ChangedBackendTokenizer()
    changed_kind, changed_behavior_sha256 = _tokenizer_behavior_fingerprint(
        same_vocab_changed_behavior, 'encdec_t5',
    )
    assert changed_kind == 'backend_tokenizer_json'
    assert same_vocab_changed_behavior.get_vocab() == checkpoint_model.tokenizer.get_vocab()
    assert changed_behavior_sha256 != checkpoint_metadata['tokenizer']['behavior_sha256']

    class _MissingTokenizerRepresentation:
        """Tokenizer stub that cannot produce a behavior fingerprint."""

        def get_added_vocab(self) -> Dict[str, int]:
            """Empty added-vocab; no backend JSON or ``__len__``.

            Returns:
                Empty dict (fingerprint code must still reject this stub).
            """
            return {}

    rejected_missing_representation = False
    try:
        _tokenizer_behavior_fingerprint(_MissingTokenizerRepresentation(), 'encdec_bart')
    except AssertionError:
        rejected_missing_representation = True
    assert rejected_missing_representation

    metadata_mutations: List[Tuple[str, str, Any]] = [
        ('llm', 'key', 'other_llm'),
        ('llm', 'repo_id', 'other/repo'),
        ('llm', 'family', 'other_family'),
        ('llm', '_commit_hash', 'other-commit'),
        ('llm', 'resolved_path', 'other/path'),
        ('llm', 'config_sha256', 'changed-config-fingerprint'),
        ('tokenizer', 'vocab_sha256', 'changed-vocab-fingerprint'),
        ('tokenizer', 'behavior_sha256', 'changed-behavior-fingerprint'),
        ('tokenizer', 'behavior_representation', 'changed-backend-kind'),
        ('tokenizer', '_commit_hash', 'changed-tokenizer-commit'),
        ('tokenizer', 'revision', 'changed-tokenizer-revision'),
        ('effective_config', 'epochs', 2),
        ('data_provenance', 'sha256', 'changed-corpus-fingerprint'),
    ]
    mutated_metadata: List[Dict[str, Any]] = []
    for section, field, value in metadata_mutations:
        mutation = json.loads(json.dumps(checkpoint_metadata))
        mutation[section][field] = value
        mutated_metadata.append(mutation)
    wrong_seed_metadata = json.loads(json.dumps(checkpoint_metadata))
    wrong_seed_metadata['seed'] = 42
    mutated_metadata.append(wrong_seed_metadata)
    wrong_shape_state = {
        **checkpoint_state,
        'linear.weight': checkpoint_state['linear.weight'][:1],
    }
    nonfinite_state = {
        **checkpoint_state,
        'linear.weight': torch.full_like(checkpoint_state['linear.weight'], float('nan')),
    }
    invalid_checkpoints: List[Dict[str, Any]] = [
        *[{**valid_checkpoint, 'metadata': metadata} for metadata in mutated_metadata],
        {**valid_checkpoint, 'selection_ar': 'rollout'},
        {**valid_checkpoint, 'data_key': 'noise50'},
        {**valid_checkpoint, 'metadata': None},
        {**valid_checkpoint, 'model': wrong_shape_state},
        {**valid_checkpoint, 'model': nonfinite_state},
    ]
    metadata_rejected = 0
    for invalid_checkpoint in invalid_checkpoints:
        try:
            validate_eeg_checkpoint_payload(
                checkpoint_model,
                invalid_checkpoint,
                expected_seed=2026,
                expected_data_provenance=checkpoint_provenance,
            )
        except AssertionError:
            metadata_rejected += 1
    assert metadata_rejected == len(invalid_checkpoints)
    assert hasattr(checkpoint_model, '_probe_checkpoint_fingerprint_cache')
    assert build_checkpoint_metadata(
        checkpoint_model, checkpoint_model.config, checkpoint_provenance,
    ) == checkpoint_metadata
    results['strict_checkpoint_keys'] = True
    results['checkpoint_metadata_and_tensors'] = True
    print('  OK complete checkpoint metadata/fingerprints, shapes, and finiteness')

    dynamic_config = dict(CONFIG)
    dynamic_config['prefill_ns'] = [0, 2, 4]
    dynamic_settings = get_prefill_settings(dynamic_config)
    assert dynamic_settings == [
        ('pred_bos', 'raw_pred_bos', 'bos', 0),
        ('pred_prefill2', 'raw_pred_prefill2', 'prefill2', 2),
        ('pred_prefill4', 'raw_pred_prefill4', 'prefill4', 4),
    ]
    results['dynamic_prefills'] = True
    print('  OK dynamic_prefills')

    sent = 'alpha beta gamma delta epsilon zeta'
    assert ProbeSystem._first_n_words(sent, 0) == ''
    assert ProbeSystem._first_n_words(sent, 1) == 'alpha'
    assert ProbeSystem._first_n_words(sent, 3) == 'alpha beta gamma'
    assert ProbeSystem._first_n_words(sent, 5) == 'alpha beta gamma delta epsilon'
    assert ProbeSystem._first_n_words('', 3) == ''
    assert target_continuation(sent, 3) == 'delta epsilon zeta'

    class _GenerateTokenizerStub:
        """Decode token-id rows as space-joined integers (skip pad/EOS)."""
        pad_token_id = 0

        def batch_decode(
            self, rows: torch.Tensor, skip_special_tokens: bool = True,
        ) -> List[str]:
            """Turn each id row into a space-separated string.

            Args:
                rows: ``(B, L)`` token ids.
                skip_special_tokens: Unused; pad (0) and EOS (2) are always dropped.

            Returns:
                One decoded string per row.
            """
            del skip_special_tokens
            return [
                ' '.join(str(int(token)) for token in row if int(token) not in (0, 2))
                for row in rows
            ]

    class _GenerateLlmStub:
        """``generate`` that appends a fixed two-token suffix to the prefix."""

        def generate(self, **kwargs: Any) -> torch.Tensor:
            """Concatenate ``decoder_input_ids`` with ``[[30, 2], [31, 2]]``.

            Args:
                **kwargs: Must include ``decoder_input_ids``.

            Returns:
                Prefix plus suffix along the time axis.
            """
            prefix = kwargs['decoder_input_ids']
            suffix = torch.tensor([[30, 2], [31, 2]], device=prefix.device)
            return torch.cat([prefix, suffix], dim=1)

    class _GenerateProbeStub:
        """Probe-shaped object for ``_run_encdec_generate`` without a real LLM."""
        device = DEVICE
        tokenizer = _GenerateTokenizerStub()
        llm = _GenerateLlmStub()
        _pad_prefix_ids = ProbeSystem._pad_prefix_ids

    generated_full, generated_continuation = ProbeSystem._run_encdec_generate(
        _GenerateProbeStub(),
        torch.zeros(2, 3, 4, device=DEVICE),
        torch.ones(2, 3, dtype=torch.long, device=DEVICE),
        [[0, 10], [0, 11]],
        pad=0,
        eos_id=2,
        max_new=2,
    )
    assert generated_full == ['10 30', '11 31']
    assert generated_continuation == ['30', '31']
    results['token_exact_continuation'] = True

    class _DecoderFamilyStub:
        """Minimal object with a ``family`` attribute for lexical-boundary tests."""

        def __init__(self, family: str) -> None:
            """Store the EncDec family string.

            Args:
                family: ``encdec_bart`` or ``encdec_t5``.
            """
            self.family = family

    bart_stub = _DecoderFamilyStub('encdec_bart')
    t5_stub = _DecoderFamilyStub('encdec_t5')
    prepare_lexical = ProbeSystem._prepare_lexical_decoder_text
    assert prepare_lexical(bart_stub, 'hello', True) == ' hello'
    assert prepare_lexical(bart_stub, '   hello', True) == ' hello'
    assert prepare_lexical(bart_stub, 'hello', False) == 'hello'
    assert prepare_lexical(t5_stub, 'hello', True) == 'hello'
    results['target_continuation'] = True
    results['family_lexical_boundary'] = True
    print('  OK target continuation + family lexical boundary')

    assert CONFIG['decoder_prompt'] == 'Based on the following signals, translate the sentence'
    results['decoder_prompt'] = True
    print('  OK decoder_prompt')

    Bm, Lm, Dm = 2, 8, 4
    z_mem = torch.randn(Bm, Lm, Dm, device=DEVICE)
    ei_mem = torch.randn(Bm, Dm, device=DEVICE)
    pad_m = torch.tensor(
        [[1, 1, 1, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0]],
        device=DEVICE, dtype=torch.long,
    )
    gt_h, gt_m = assemble_encdec_memory(
        z_mem, ei_mem, pad_m, prepend_ei=False, use_ei=True, model_dtype=z_mem.dtype,
    )
    assert gt_h.shape == z_mem.shape and torch.equal(gt_m, pad_m)
    eeg_h, eeg_m = assemble_encdec_memory(
        z_mem, ei_mem, None, prepend_ei=True, use_ei=True, model_dtype=z_mem.dtype,
    )
    assert eeg_h.shape == (Bm, Lm + 1, Dm) and eeg_m.shape == (Bm, Lm + 1)
    assert bool(eeg_m.eq(1).all())
    results['gt_text_memory_no_ei'] = True
    print('  OK gt_text_memory_no_ei')

    tsne_text = np.arange(24, dtype=np.float64).reshape(6, 4) + 1.0
    tsne_snapshots = {
        'init': {'ei_aligned': tsne_text + 0.5, 'ei_text': tsne_text},
        'after': {'ei_aligned': tsne_text + 1.0, 'ei_text': tsne_text.copy()},
    }
    tsne_stacked, tsne_order, tsne_n = _prepare_shared_tsne_arrays(
        tsne_snapshots, max_points=5, seed=2026,
    )
    assert tsne_order == ['init', 'after'] and tsne_n == 5
    assert tsne_stacked.shape == (15, 4)
    results['shared_tsne_shapes'] = True
    print('  OK shared_tsne_shapes')

    id_preds = ['hello world foo bar', 'the cat sat down']
    id_gts = ['hello world foo bar', 'the cat sat down']
    bleu_id = mean_bleu(id_preds, id_gts)
    for n in range(1, 5):
        assert abs(bleu_id[f'bleu{n}'] - 1.0) < 1e-5, bleu_id
    bleu_empty = mean_bleu(['', ''], id_gts)
    for n in range(1, 5):
        assert bleu_empty[f'bleu{n}'] == 0.0, bleu_empty
    results['mean_bleu'] = True
    print('  OK mean_bleu identical=1 empty=0')

    class _DummySbert:
        """Deterministic bag-of-chars encoder for smoke (no Hub download)."""

        def encode(
            self,
            texts: Sequence[str],
            batch_size: int = 64,
            convert_to_tensor: bool = True,
            show_progress_bar: bool = False,
            device: Optional[str] = None,
        ) -> torch.Tensor:
            """Hash each string into a fixed 8-D bag-of-bytes vector.

            Args:
                texts: Input strings.
                batch_size: Unused (API compatibility).
                convert_to_tensor: Unused; always returns a tensor.
                show_progress_bar: Unused.
                device: Unused; vectors stay on CPU.

            Returns:
                ``(len(texts), 8)`` float32 tensor of L2-normalized vectors.
            """
            dim = 8
            rows: List[np.ndarray] = []
            for t in texts:
                s = t if isinstance(t, str) else ''
                vec = np.zeros(dim, dtype=np.float32)
                for i, ch in enumerate(s.encode('utf-8')):
                    vec[i % dim] += float(ch)
                nrm = float(np.linalg.norm(vec))
                if nrm > 0:
                    vec = vec / nrm
                else:
                    vec[0] = 1.0
                rows.append(vec)
            return torch.tensor(np.stack(rows, axis=0), dtype=torch.float32)

    dummy = _DummySbert()
    cos_id = compute_sbert_cosine_per_row(
        id_preds, id_gts, dummy, device=str(DEVICE), seed=2026,
    )
    assert cos_id.shape == (2,)
    assert not np.isnan(cos_id).any()
    assert np.allclose(cos_id, 1.0, atol=1e-5), cos_id
    empty_cos = compute_sbert_cosine_per_row(
        ['', id_preds[1]], id_gts, dummy, device=str(DEVICE), seed=2026,
    )
    assert float(empty_cos[0]) == 0.0
    assert abs(mean_sbert_cosine(id_preds, id_gts, dummy, device=str(DEVICE)) - 1.0) < 1e-5
    results['sbert_cosine'] = True
    print('  OK sbert_cosine including empty=0')

    eval_settings = get_prefill_settings(CONFIG)
    dummy_columns: Dict[str, List[str]] = {'gt': id_gts}
    for pred_col, raw_col, _label, n_words in eval_settings:
        dummy_columns[raw_col] = list(id_preds)
        dummy_columns[pred_col] = [
            target_continuation(prediction, n_words) for prediction in id_preds
        ]
    dummy_df = pd.DataFrame(dummy_columns)
    metrics_df, row_scores = evaluate_settings(
        dummy_df,
        device=DEVICE,
        settings=eval_settings,
        seed=2026,
        sbert_model=dummy,
        show_plot=False,
    )
    primary = metrics_df.loc[metrics_df['scope'] == 'continuation'].reset_index(drop=True)
    diagnostic = metrics_df.loc[
        metrics_df['scope'] == 'whole_output_diagnostic'
    ].reset_index(drop=True)
    assert list(primary['setting']) == [setting[2] for setting in eval_settings]
    assert len(primary) == len(diagnostic) == len(eval_settings)
    assert abs(float(primary.iloc[0]['bleu1']) - 1.0) < 1e-5
    assert row_scores['pred_bos']['bleu1'].shape == (2,)
    results['continuation_metrics'] = True
    print('  OK continuation primary + whole-output diagnostics')

    assert list(CONFIG['seeds']) == [2026, 42, 36]
    dummy_seed_rows: List[Dict[str, Any]] = []
    for s, v in ((2026, 0.1), (42, 0.2), (36, 0.3)):
        dummy_seed_rows.append({
            'llm_key': 'dummy',
            'data_key': 'eeg',
            'setting': 'bos',
            'seed': int(s),
            'bleu1': float(v),
            'bleu2': float(v),
            'bleu3': float(v),
            'bleu4': float(v),
            'sbert_cosine': float(v),
        })
    seed_summary = summarize_seed_metrics(
        dummy_seed_rows, title='smoke mean±std', show_plot=False,
    )
    assert len(seed_summary) == 1
    assert abs(float(seed_summary.iloc[0]['bleu1_mean']) - 0.2) < 1e-8
    expect_std = float(np.std(np.array([0.1, 0.2, 0.3], dtype=np.float64), ddof=1))
    assert abs(float(seed_summary.iloc[0]['bleu1_std']) - expect_std) < 1e-8
    assert abs(float(seed_summary.iloc[0]['sbert_cosine_mean']) - 0.2) < 1e-8
    one_row = summarize_seed_metrics(
        dummy_seed_rows[:1], title='smoke n=1 std=0', show_plot=False,
    )
    assert abs(float(one_row.iloc[0]['bleu1_std'])) < 1e-12
    results['summarize_seed_metrics'] = True
    print('  OK summarize_seed_metrics')

    print('Encoder smoke results:', results)
    assert all(results.values())
    return results


print('========== Encoder smoke ==========')
_ = run_smoke_tests()


In [ ]:
# =============================================================
# Main: encoder smoke (prev cell) → per-LLM smokes → corpus → train/generate
# =============================================================

set_seed(int(CONFIG['seed']))
validate_probe_run_config(CONFIG)

# Main filtering validates unknown/duplicate requests before any model work.
configs = select_requested_llm_configs(
    LLM_CONFIGS, CONFIG.get('llm_keys_to_run'),
)

# Smoke filtering independently applies the same strict validation.
LLM_SMOKE_RESULTS = run_all_llm_smokes(LLM_CONFIGS, CONFIG, DEVICE)
assert list(LLM_SMOKE_RESULTS) == [str(config['key']) for config in configs]

if CONFIG.get('run_smoke_only', False):
    print('run_smoke_only=True — skipping full training / generate loop.')
else:
    merged = load_merged_corpus(CONFIG['data_path'])
    dm = PhaseDataModule(
        merged, batch_size=int(CONFIG['batch_size']), num_workers=int(CONFIG['num_workers']),
    )
    dm.setup()

    all_metric_rows: List[Dict[str, Any]] = []
    for llm_cfg in configs:
        df, rows = run_one_llm(llm_cfg, dm, CONFIG, DEVICE)
        all_metric_rows.extend(rows)
        print(f'  {llm_cfg["key"]} last predictions: {len(df)} rows')

    assert all_metric_rows, 'full run produced no metrics'
    _ = summarize_seed_metrics(
        all_metric_rows,
        title=f'All LLMs mean ± std over seeds {CONFIG["seeds"]}',
    )
    print('Done: every configured LLM completed successfully.')
